# 06 — CGGA external evaluation (Sections 3.4.3, 4.8–4.9)

In [ ]:
import os
from pathlib import Path

# Folder that holds the downloaded public data; all outputs are written into it.
# Set the environment variable GBM_DATA_DIR, or edit the default below.
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
print("Data folder:", BASE)

### INSPECT CGGA FILES -- run this first, before anything else.

In [ ]:
"""
INSPECT CGGA FILES -- run this first, before anything else.
Just prints the shape and column names of whatever you downloaded, so we
can write the exact filtering/renaming code that matches your real files
instead of guessing at CGGA's column names.

EDIT the two paths below to point at wherever you saved the downloaded
CGGA clinical file and expression (read counts) file.
"""

from pathlib import Path
import pandas as pd

# EDIT if these don't match exactly what's in your Downloads folder
CGGA_CLINICAL_RAW = BASE / "CGGA.mRNAseq_693_clinical.20200506"
CGGA_EXPRESSION_RAW = BASE / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"


def resolve_path(path):
    """CGGA downloads sometimes unzip into a folder containing the actual
    .txt file rather than being a file directly. If `path` is a directory,
    look inside it for a .txt or .csv file and use that instead."""
    if path.is_file():
        return path
    if path.is_dir():
        candidates = list(path.glob("*.txt")) + list(path.glob("*.csv"))
        if candidates:
            print(f"  NOTE: {path} is a directory -- using {candidates[0].name} found inside it")
            return candidates[0]
    # try common extension variants if the exact name wasn't found
    for ext in [".txt", ".csv", ".tsv"]:
        alt = path.with_suffix(ext) if path.suffix == "" else path
        if alt.exists():
            return alt
    return path  # return as-is; existence check below will report it clearly


CGGA_CLINICAL_RAW = resolve_path(CGGA_CLINICAL_RAW)
CGGA_EXPRESSION_RAW = resolve_path(CGGA_EXPRESSION_RAW)

for label, path in [("CLINICAL", CGGA_CLINICAL_RAW), ("EXPRESSION (counts)", CGGA_EXPRESSION_RAW)]:
    print("=" * 80)
    print(f"{label}: {path}")
    print("=" * 80)
    if not path.exists():
        print(f"  FILE NOT FOUND. Check the exact filename in your Downloads folder and edit the path above.")
        continue

    # try common CGGA delimiters -- their files are sometimes tab-separated
    for sep, sep_name in [("\t", "tab"), (",", "comma")]:
        try:
            df = pd.read_csv(path, sep=sep, nrows=5)
            if df.shape[1] > 1:  # more than one column means we guessed the separator right
                print(f"  Detected separator: {sep_name}")
                print(f"  Shape (first 5 rows preview): {df.shape}")
                print(f"  Column names ({len(df.columns)} total):")
                print(f"    {list(df.columns[:20])}" + (" ..." if len(df.columns) > 20 else ""))
                print(f"\n  First few rows:")
                print(df.head())

                if label.startswith("EXPRESSION"):
                    first_col = df.columns[0]
                    print(f"\n  ORIENTATION CHECK: first column is '{first_col}'.")
                    print(f"  If this looks like a gene ID (e.g. starts with 'ENSG' or is a gene")
                    print(f"  symbol like 'TP53') and the remaining ~693 columns look like sample")
                    print(f"  IDs, this file is GENES-AS-ROWS (CGGA's typical format) and will need")
                    print(f"  to be TRANSPOSED before use -- your TCGA pipeline expects patients as")
                    print(f"  rows, genes as columns. If instead the first column looks like a")
                    print(f"  sample/patient ID, it's already in the right orientation.")
                break
        except Exception as e:
            continue
    print()

print("=" * 80)
print("NEXT STEP: paste the column names printed above back to me, and I'll write the")
print("exact filtering (primary, IDH-wildtype GBM) and column-renaming code to match")
print("your TCGA clinical format.")
print("=" * 80)

### PROCESS CGGA -> harmonized format matching your TCGA pipeline

In [ ]:
"""
PROCESS CGGA -> harmonized format matching your TCGA pipeline
==================================================================
Run this in two passes:
  PASS 1 (default): prints unique values of the filtering columns so you can
  CONFIRM the exact GBM/IDH-wildtype label strings before we filter on them.
  PASS 2: set CONFIRMED = True once you've checked the printed values match
  the FILTER_* constants below, then rerun to actually produce the files.

Outputs (once CONFIRMED = True):
  CGGA_SURVIVAL_COHORT.csv   -- patient_id, OS_months, OS_event
  CGGA_CLINICAL.csv          -- patient_id index, columns matching your TCGA
                                 clinical file (Diagnosis Age, Sex, MGMT Status)
  CGGA_EXPRESSION.csv        -- patient_id rows, ENSG... gene columns,
                                 log2(CPM+1) normalized, mapped from CGGA's
                                 gene SYMBOLS to Ensembl IDs

NOT covered here: rewiring scores. That still needs the original
TCGA rewiring computation method/healthy-reference applied to CGGA --
share that script separately once this part is working.
"""

from pathlib import Path
import numpy as np
import pandas as pd

try:
    import mygene
except ImportError:
    raise SystemExit("Run: pip install mygene --break-system-packages")

DOWNLOADS = BASE
STAGE_D_DIR = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL"
CGGA_OUT_DIR = STAGE_D_DIR / "CGGA_EXTERNAL_VALIDATION"
CGGA_OUT_DIR.mkdir(parents=True, exist_ok=True)

CGGA_CLINICAL_PATH = (DOWNLOADS / "CGGA.mRNAseq_693_clinical.20200506"
                       / "CGGA.mRNAseq_693_clinical.20200506.txt")
CGGA_EXPRESSION_PATH = (DOWNLOADS / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"
                         / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt")

# ---- confirmed from your PASS 1 output: 109 patients (Histology='GBM',
# PRS_type='Primary', IDH_mutation_status='Wildtype') ----
CONFIRMED = True
FILTER_HISTOLOGY = "GBM"          # exact string for primary GBM in Histology column
FILTER_PRS_TYPE = "Primary"       # exclude Recurrent/secondary
FILTER_IDH_STATUS = "Wildtype"    # match TCGA's IDH-wildtype inclusion rule


print("=" * 80)
print("PASS 1 -- CHECKING FILTER COLUMN VALUES")
print("=" * 80)

clinical = pd.read_csv(CGGA_CLINICAL_PATH, sep="\t")
print(f"\nTotal CGGA patients in file: {len(clinical)}")

for col in ["PRS_type", "Histology", "Grade", "IDH_mutation_status"]:
    print(f"\n{col} value counts:")
    print(clinical[col].value_counts(dropna=False))

candidate_mask = (
    (clinical["Histology"] == FILTER_HISTOLOGY) &
    (clinical["PRS_type"] == FILTER_PRS_TYPE) &
    (clinical["IDH_mutation_status"] == FILTER_IDH_STATUS)
)
print(f"\nWith current filter settings (Histology='{FILTER_HISTOLOGY}', "
      f"PRS_type='{FILTER_PRS_TYPE}', IDH='{FILTER_IDH_STATUS}'): "
      f"{candidate_mask.sum()} patients would be selected.")

if not CONFIRMED:
    print("\n" + "=" * 80)
    print("STOPPING HERE. Check the value_counts above --")
    print("if 'GBM' / 'Primary' / 'Wildtype' are exactly right (correct spelling/")
    print("capitalization) and the resulting patient count looks reasonable")
    print("(expect roughly 100-140 based on published CGGA primary GBM counts),")
    print("set CONFIRMED = True at the top of this script and rerun.")
    print("=" * 80)
    raise SystemExit(0)


# ============================================================================
# PASS 2 -- build the harmonized files
# ============================================================================
print("\n" + "=" * 80)
print("PASS 2 -- BUILDING HARMONIZED CGGA FILES")
print("=" * 80)

clinical_filtered = clinical[candidate_mask].copy()
print(f"Filtered cohort (before missing-survival-data check): {len(clinical_filtered)} patients")

# drop any patients missing OS time or event status -- can't build a survival
# label without both, and forcing NaN to int (as the original code did) errors
survival_cols = ["OS", "Censor (alive=0; dead=1)"]
missing_survival = clinical_filtered[survival_cols].isna().any(axis=1)
if missing_survival.any():
    dropped_ids = clinical_filtered.loc[missing_survival, "CGGA_ID"].tolist()
    print(f"  Dropping {missing_survival.sum()} patients with missing OS/Censor data: {dropped_ids}")
    clinical_filtered = clinical_filtered[~missing_survival].copy()
print(f"Filtered cohort (after missing-survival-data check): {len(clinical_filtered)} patients")

# ---- survival cohort file ----
survival_out = pd.DataFrame({
    "patient_id": clinical_filtered["CGGA_ID"],
    "OS_months": clinical_filtered["OS"].astype(float) / 30.44,  # days -> months
    "OS_event": clinical_filtered["Censor (alive=0; dead=1)"].astype(float).astype(int),  # 1=death, matches TCGA convention
}).set_index("patient_id")
survival_out.to_csv(CGGA_OUT_DIR / "CGGA_SURVIVAL_COHORT.csv")
print(f"Saved: CGGA_SURVIVAL_COHORT.csv ({len(survival_out)} patients)")
print(f"  Median OS: {survival_out['OS_months'].median():.2f} months, "
      f"events: {survival_out['OS_event'].sum()}/{len(survival_out)}")

# ---- clinical file, renamed to match TCGA column names ----
clinical_out = pd.DataFrame({
    "patient_id": clinical_filtered["CGGA_ID"],
    "Diagnosis Age": clinical_filtered["Age"].astype(float),
    "Sex": clinical_filtered["Gender"],
    "MGMT Status": clinical_filtered["MGMTp_methylation_status"],
    # No Karnofsky Performance Score available in CGGA -- column intentionally
    # omitted. Your build_clinical_preprocessor() already checks column
    # existence before use, so this will NOT crash -- M1 on CGGA will just
    # use Age + Sex + MGMT Status, one fewer numeric feature than on TCGA.
}).set_index("patient_id")
clinical_out.to_csv(CGGA_OUT_DIR / "CGGA_CLINICAL.csv")
print(f"\nSaved: CGGA_CLINICAL.csv")
print(f"  NOTE: Karnofsky Performance Score is NOT available in CGGA and was omitted.")
print(f"  MGMT Status values in CGGA: {clinical_out['MGMT Status'].unique().tolist()}")
print(f"  >> Check these match your TCGA M1 file's MGMT Status category strings")
print(f"     EXACTLY (capitalization matters for the OneHotEncoder) -- if they")
print(f"     differ (e.g. TCGA uses 'Methylated' vs CGGA's 'methylated'), rename")
print(f"     CGGA's values to match before scoring, or the encoder will treat")
print(f"     them as unseen categories via handle_unknown='ignore' (silently")
print(f"     zeroing that feature rather than erroring, which is worse).")

# ---- expression file: transpose, subset to filtered patients, log-CPM, map gene IDs ----
print(f"\nLoading expression matrix (this may take a minute -- it's a large file)...")
expr_raw = pd.read_csv(CGGA_EXPRESSION_PATH, sep="\t", index_col="gene_name")
print(f"Raw expression shape (genes x samples): {expr_raw.shape}")

filtered_ids = clinical_filtered["CGGA_ID"].tolist()
missing_ids = [pid for pid in filtered_ids if pid not in expr_raw.columns]
if missing_ids:
    print(f"  WARNING: {len(missing_ids)} filtered clinical patients have no expression "
          f"column -- excluding them: {missing_ids[:10]}{'...' if len(missing_ids) > 10 else ''}")
filtered_ids = [pid for pid in filtered_ids if pid in expr_raw.columns]

expr_subset = expr_raw[filtered_ids].T  # transpose -> patients as rows now
expr_subset.index.name = "patient_id"

# log2(CPM + 1) normalization, per patient
library_sizes = expr_subset.sum(axis=1)
cpm = expr_subset.div(library_sizes, axis=0) * 1e6
log_cpm = np.log2(cpm + 1)
print(f"Log-CPM normalized shape (patients x genes): {log_cpm.shape}")

# map gene SYMBOLS -> Ensembl IDs to match TCGA's ENSG... column format
print("Mapping gene symbols to Ensembl IDs via mygene (batched, this takes a few minutes)...")
mg = mygene.MyGeneInfo()
gene_symbols = log_cpm.columns.tolist()

symbol_to_ensembl = {}
batch_size = 1000
for i in range(0, len(gene_symbols), batch_size):
    batch = gene_symbols[i:i + batch_size]
    results = mg.querymany(batch, scopes="symbol", fields="ensembl.gene", species="human")
    for r in results:
        if "notfound" in r or "ensembl" not in r:
            continue
        ens = r["ensembl"]
        ens_id = ens[0]["gene"] if isinstance(ens, list) else ens.get("gene")
        if ens_id:
            symbol_to_ensembl[r["query"]] = ens_id
    print(f"  Mapped batch {i // batch_size + 1}/{(len(gene_symbols) - 1) // batch_size + 1}")

mapped_cols = {sym: symbol_to_ensembl[sym] for sym in log_cpm.columns if sym in symbol_to_ensembl}
unmapped_count = log_cpm.shape[1] - len(mapped_cols)
print(f"\nMapped {len(mapped_cols)} / {log_cpm.shape[1]} genes to Ensembl IDs "
      f"({unmapped_count} unmapped, likely deprecated symbols or non-coding IDs -- dropped)")

log_cpm_mapped = log_cpm[list(mapped_cols.keys())].rename(columns=mapped_cols)
# if multiple symbols mapped to the same Ensembl ID, keep the first occurrence
log_cpm_mapped = log_cpm_mapped.loc[:, ~log_cpm_mapped.columns.duplicated()]

log_cpm_mapped.to_csv(CGGA_OUT_DIR / "CGGA_EXPRESSION.csv")
print(f"\nSaved: CGGA_EXPRESSION.csv ({log_cpm_mapped.shape[0]} patients, "
      f"{log_cpm_mapped.shape[1]} genes, Ensembl-ID columns)")

print("\n" + "=" * 80)
print("DONE. Still missing before full CGGA validation can run:")
print("  - CGGA_REWIRING.csv -- needs your original rewiring computation method")
print("    applied to this same filtered/normalized CGGA expression matrix,")
print("    against the SAME healthy-brain reference used for TCGA.")
print("=" * 80)

### CGGA EXTERNAL VALIDATION — FROZEN V4 M2 EXPRESSION MODEL

In [ ]:
"""
===============================================================================
CGGA EXTERNAL VALIDATION — FROZEN V4 M2 EXPRESSION MODEL
===============================================================================

Goal
----
External validation of the already-frozen TCGA V4 M2 model.

Frozen model:
    Random Survival Forest
    n_estimators = 300
    min_samples_leaf = 5
    max_features = "sqrt"
    min_samples_split = 10
    random_state = 42

Frozen V4 genes:
    top_k = 25
    selected using TCGA development data ONLY.

CGGA is NEVER used for:
    - gene selection
    - hyperparameter tuning
    - model selection
    - survival-based preprocessing

CGGA cohort:
    - Primary
    - GBM
    - WHO IV
    - IDH Wildtype
    - Adult (>=18 years)
    - valid OS
    - valid event status

IMPORTANT
---------
This script first searches Downloads for the CGGA mRNAseq_693 RNA file.

It supports:
    1. samples as rows / genes as columns
    2. genes as rows / samples as columns

It DOES NOT replace missing genes by zero.

===============================================================================
"""

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored


warnings.filterwarnings("ignore")


# =============================================================================
# CONFIG
# =============================================================================

RANDOM_SEED = 42

DOWNLOADS = BASE

STAGE_D_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

# TCGA files already used in Stage D
SPLIT_FILE = (
    STAGE_D_DIR
    / "STAGE_D_LOCKED_SPLIT.csv"
)

TCGA_SURVIVAL_FILE = (
    STAGE_D_DIR
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

TCGA_EXPRESSION_FILE = (
    STAGE_D_DIR
    / "STAGE_D_M2_EXPRESSION_RAW.csv"
)


# -------------------------------------------------------------------------
# CGGA clinical file
# -------------------------------------------------------------------------

# First try common locations automatically.
CGGA_CLINICAL_CANDIDATES = [

    DOWNLOADS
    / "CGGA.mRNAseq_693_clinical.20200506.txt",

    STAGE_D_DIR
    / "CGGA_EXTERNAL_VALIDATION"
    / "CGGA.mRNAseq_693_clinical.20200506.txt",

]


# =============================================================================
# FROZEN V4 M2 GENES
# =============================================================================

V4_GENES = [

    "ENSG00000147606",
    "ENSG00000060688",
    "ENSG00000054356",
    "ENSG00000100364",
    "ENSG00000176454",
    "ENSG00000155093",
    "ENSG00000185133",
    "ENSG00000175264",
    "ENSG00000113946",
    "ENSG00000144119",
    "ENSG00000089225",
    "ENSG00000182870",
    "ENSG00000106123",
    "ENSG00000254415",
    "ENSG00000087152",
    "ENSG00000137843",
    "ENSG00000170786",
    "ENSG00000157214",
    "ENSG00000167778",
    "ENSG00000104722",
    "ENSG00000153790",
    "ENSG00000167074",
    "ENSG00000116783",
    "ENSG00000118402",
    "ENSG00000135324",

]


# =============================================================================
# FROZEN MODEL
# =============================================================================

RSF_PARAMS = {

    "n_estimators": 300,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "min_samples_split": 10,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,

}


# =============================================================================
# UTILITIES
# =============================================================================

def section(title):

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def make_survival_y(df):

    return np.array(

        [
            (
                bool(event),
                float(time),
            )

            for event, time in zip(
                df["OS_event"],
                df["OS_months"],
            )
        ],

        dtype=[
            ("event", "?"),
            ("time", "<f8"),
        ],

    )


def cindex(y, risk):

    risk = np.asarray(
        risk,
        dtype=float,
    )

    return float(
        concordance_index_censored(
            y["event"],
            y["time"],
            risk,
        )[0]
    )


def clean_gene_id(x):

    """
    Removes Ensembl version suffix.

    Example:
        ENSG00000147606.15
        ->
        ENSG00000147606
    """

    x = str(x).strip()

    if x.startswith("ENSG"):

        x = x.split(".")[0]

    return x


def clean_patient_id(x):

    return str(x).strip()


# =============================================================================
# FIND CGGA CLINICAL FILE
# =============================================================================

section(
    "STEP 1 — FIND CGGA CLINICAL FILE"
)


CGGA_CLINICAL_FILE = None

for candidate in CGGA_CLINICAL_CANDIDATES:

    if candidate.exists():

        CGGA_CLINICAL_FILE = candidate
        break


if CGGA_CLINICAL_FILE is None:

    matches = list(
        DOWNLOADS.rglob(
            "*mRNAseq_693*clinical*.txt"
        )
    )

    if len(matches) == 0:

        raise FileNotFoundError(
            "Could not find CGGA mRNAseq_693 clinical file in Downloads."
        )

    CGGA_CLINICAL_FILE = matches[0]


print(
    "Clinical file:"
)

print(
    CGGA_CLINICAL_FILE
)


# =============================================================================
# FIND CGGA RNA FILE
# =============================================================================

section(
    "STEP 2 — FIND CGGA RNA FILE"
)


all_candidates = []

patterns = [

    "*mRNAseq_693*.txt",
    "*mRNAseq_693*.tsv",
    "*mRNAseq_693*.csv",

]


for pattern in patterns:

    all_candidates.extend(
        DOWNLOADS.rglob(
            pattern
        )
    )


# remove clinical file
rna_candidates = [

    p for p in all_candidates

    if (
        "clinical" not in p.name.lower()
        and p != CGGA_CLINICAL_FILE
    )

]


# Prefer likely RSEM / gene-expression file
rna_candidates = sorted(

    rna_candidates,

    key=lambda p: (

        "rsem" not in p.name.lower(),
        "gene" not in p.name.lower(),
        -p.stat().st_size,

    ),

)


print(
    f"Candidate RNA files found: "
    f"{len(rna_candidates)}"
)


for i, p in enumerate(
    rna_candidates[:10],
    start=1,
):

    print(
        f"{i}. {p} "
        f"({p.stat().st_size / 1024**2:.1f} MB)"
    )


if len(rna_candidates) == 0:

    raise FileNotFoundError(

        "\nNo CGGA mRNAseq_693 RNA file was found.\n"
        "Place the RNA file somewhere inside Downloads and rerun."

    )


CGGA_RNA_FILE = rna_candidates[0]


print(
    "\nSelected RNA file:"
)

print(
    CGGA_RNA_FILE
)


# =============================================================================
# LOAD CGGA CLINICAL
# =============================================================================

section(
    "STEP 3 — LOAD AND FILTER CGGA CLINICAL COHORT"
)


clinical = pd.read_csv(

    CGGA_CLINICAL_FILE,
    sep="\t",
    low_memory=False,

)


print(
    "Raw clinical shape:",
    clinical.shape,
)


required_columns = [

    "CGGA_ID",
    "PRS_type",
    "Histology",
    "Grade",
    "Age",
    "OS",
    "Censor (alive=0; dead=1)",
    "IDH_mutation_status",

]


missing_columns = [

    c for c in required_columns

    if c not in clinical.columns

]


if missing_columns:

    raise RuntimeError(

        f"Missing clinical columns: "
        f"{missing_columns}"

    )


# numeric conversion
clinical["Age"] = pd.to_numeric(
    clinical["Age"],
    errors="coerce",
)

clinical["OS"] = pd.to_numeric(
    clinical["OS"],
    errors="coerce",
)

clinical["event"] = pd.to_numeric(

    clinical[
        "Censor (alive=0; dead=1)"
    ],

    errors="coerce",

)


# Exact external cohort definition
mask = (

    clinical[
        "PRS_type"
    ].astype(str).str.strip().str.lower().eq(
        "primary"
    )

    &

    clinical[
        "Histology"
    ].astype(str).str.strip().str.upper().eq(
        "GBM"
    )

    &

    clinical[
        "Grade"
    ].astype(str).str.strip().str.upper().eq(
        "WHO IV"
    )

    &

    clinical[
        "IDH_mutation_status"
    ].astype(str).str.strip().str.lower().eq(
        "wildtype"
    )

    &

    clinical[
        "Age"
    ].ge(
        18
    )

    &

    clinical[
        "OS"
    ].notna()

    &

    clinical[
        "event"
    ].isin(
        [0, 1]
    )

    &

    clinical[
        "OS"
    ].gt(
        0
    )

)


cgga = (
    clinical.loc[
        mask
    ]
    .copy()
)


cgga["patient_id"] = (
    cgga[
        "CGGA_ID"
    ]
    .map(
        clean_patient_id
    )
)


# CGGA OS is in DAYS.
# Convert to months.
cgga["OS_months"] = (
    cgga["OS"]
    / 30.4375
)


cgga["OS_event"] = (
    cgga["event"]
    .astype(int)
)


cgga = (
    cgga
    .drop_duplicates(
        subset="patient_id"
    )
    .set_index(
        "patient_id"
    )
)


print(
    "Eligible CGGA validation patients:",
    len(cgga),
)

print(
    "Events:",
    int(
        cgga[
            "OS_event"
        ].sum()
    ),
)

print(
    "Censored:",
    int(
        len(cgga)
        - cgga[
            "OS_event"
        ].sum()
    ),
)

print(
    "Median age:",
    float(
        cgga["Age"].median()
    ),
)

print(
    "Median OS months:",
    float(
        cgga[
            "OS_months"
        ].median()
    ),
)


# =============================================================================
# LOAD RNA FILE
# =============================================================================

section(
    "STEP 4 — LOAD CGGA RNA"
)


suffix = (
    CGGA_RNA_FILE
    .suffix
    .lower()
)


if suffix == ".csv":

    rna_raw = pd.read_csv(
        CGGA_RNA_FILE,
        low_memory=False,
    )

else:

    rna_raw = pd.read_csv(
        CGGA_RNA_FILE,
        sep="\t",
        low_memory=False,
    )


print(
    "Raw RNA shape:",
    rna_raw.shape,
)

print(
    "\nFirst 10 column names:"
)

print(
    list(
        rna_raw.columns[:10]
    )
)


# =============================================================================
# DETECT RNA ORIENTATION
# =============================================================================

section(
    "STEP 5 — DETECT RNA ORIENTATION"
)


cgga_ids = set(
    cgga.index
)


column_ids = set(
    map(
        clean_patient_id,
        rna_raw.columns,
    )
)


first_col_values = set(

    rna_raw.iloc[
        :,
        0
    ]
    .astype(str)
    .map(
        clean_patient_id
    )
    .head(
        5000
    )

)


n_patient_columns = len(
    cgga_ids
    & column_ids
)


n_patient_rows = len(
    cgga_ids
    & first_col_values
)


print(
    "Eligible patient IDs found in RNA columns:",
    n_patient_columns,
)

print(
    "Eligible patient IDs found in RNA first column:",
    n_patient_rows,
)


# -------------------------------------------------------------------------
# CASE A: genes are rows and patients are columns
# -------------------------------------------------------------------------

if n_patient_columns > n_patient_rows:

    print(
        "\nDetected format:"
        "\nGENES = ROWS"
        "\nPATIENTS = COLUMNS"
    )


    # Look for Ensembl IDs in first few columns
    gene_col = None

    for col in rna_raw.columns[:5]:

        values = (
            rna_raw[col]
            .astype(str)
            .head(
                5000
            )
        )

        frac_ensembl = (
            values
            .str.startswith(
                "ENSG"
            )
            .mean()
        )

        if frac_ensembl > 0.20:

            gene_col = col
            break


    if gene_col is None:

        print(
            "\nWARNING:"
        )

        print(
            "No obvious Ensembl gene-ID column detected."
        )

        print(
            "Available first columns:"
        )

        print(
            list(
                rna_raw.columns[:5]
            )
        )

        raise RuntimeError(

            "RNA appears to use gene symbols instead of Ensembl IDs. "
            "Stop here and check the printed first column names."

        )


    print(
        "Gene ID column:",
        gene_col,
    )


    rna_raw[
        gene_col
    ] = (
        rna_raw[
            gene_col
        ]
        .map(
            clean_gene_id
        )
    )


    rna_raw = (
        rna_raw
        .drop_duplicates(
            subset=gene_col
        )
        .set_index(
            gene_col
        )
    )


    matched_patient_columns = [

        p for p in cgga.index

        if p in rna_raw.columns

    ]


    print(
        "Clinical/RNA matched patients:",
        len(
            matched_patient_columns
        ),
    )


    # genes x patients -> patients x genes
    expression = (
        rna_raw.loc[
            :,
            matched_patient_columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .T
    )


# -------------------------------------------------------------------------
# CASE B: patients are rows and genes are columns
# -------------------------------------------------------------------------

else:

    print(
        "\nDetected format:"
        "\nPATIENTS = ROWS"
        "\nGENES = COLUMNS"
    )


    patient_col = rna_raw.columns[0]


    rna_raw[
        patient_col
    ] = (
        rna_raw[
            patient_col
        ]
        .map(
            clean_patient_id
        )
    )


    rna_raw = (
        rna_raw
        .drop_duplicates(
            subset=patient_col
        )
        .set_index(
            patient_col
        )
    )


    rna_raw.columns = [

        clean_gene_id(
            c
        )

        for c in rna_raw.columns

    ]


    matched_ids = [

        p for p in cgga.index

        if p in rna_raw.index

    ]


    expression = (
        rna_raw.loc[
            matched_ids
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )


print(
    "\nProcessed CGGA expression shape:",
    expression.shape,
)


# =============================================================================
# MATCH CLINICAL + RNA
# =============================================================================

section(
    "STEP 6 — MATCH CGGA CLINICAL AND RNA"
)


common_ids = [

    p for p in cgga.index

    if p in expression.index

]


cgga = (
    cgga.loc[
        common_ids
    ]
    .copy()
)


expression = (
    expression.loc[
        common_ids
    ]
    .copy()
)


print(
    "Final matched validation patients:",
    len(
        common_ids
    ),
)


if len(
    common_ids
) < 20:

    raise RuntimeError(

        "Very few CGGA patients matched RNA. "
        "Patient orientation or IDs may be wrong."

    )


# =============================================================================
# CHECK FROZEN GENES
# =============================================================================

section(
    "STEP 7 — CHECK FROZEN V4 GENES"
)


available_genes = [

    g for g in V4_GENES

    if g in expression.columns

]


missing_genes = [

    g for g in V4_GENES

    if g not in expression.columns

]


print(
    f"Available frozen genes: "
    f"{len(available_genes)}/{len(V4_GENES)}"
)


print(
    "Missing genes:"
)

print(
    missing_genes
)


if len(
    available_genes
) < 20:

    raise RuntimeError(

        "\nToo many V4 genes are missing.\n"
        "Do NOT continue validation.\n"
        "The RNA file may use gene symbols rather than Ensembl IDs."

    )


# IMPORTANT:
# We DO NOT create missing genes with zeros.
#
# If exactly one is missing, validation is performed using the common
# feature set in BOTH TCGA and CGGA.


validation_genes = (
    available_genes
)


print(
    "\nGenes used for external validation:",
    len(
        validation_genes
    ),
)


# =============================================================================
# LOAD TCGA DEVELOPMENT DATA
# =============================================================================

section(
    "STEP 8 — LOAD FROZEN TCGA DEVELOPMENT DATA"
)


split_df = pd.read_csv(
    SPLIT_FILE
)


development_ids = (

    split_df.loc[
        split_df[
            "split"
        ]
        == "development",
        "patient_id",
    ]
    .tolist()

)


tcga_survival = (
    pd.read_csv(
        TCGA_SURVIVAL_FILE
    )
    .set_index(
        "patient_id"
    )
)


tcga_expression = (
    pd.read_csv(
        TCGA_EXPRESSION_FILE,
        index_col=0,
    )
)


print(
    "TCGA development patients:",
    len(
        development_ids
    ),
)


missing_tcga_genes = [

    g for g in validation_genes

    if g not in tcga_expression.columns

]


if missing_tcga_genes:

    raise RuntimeError(

        f"Unexpected TCGA missing genes: "
        f"{missing_tcga_genes}"

    )


# =============================================================================
# EXPRESSION AUDIT
# =============================================================================

section(
    "STEP 9 — TCGA vs CGGA EXPRESSION SCALE AUDIT"
)


tcga_values = (
    tcga_expression.loc[
        development_ids,
        validation_genes,
    ]
    .to_numpy(
        dtype=float
    )
)


cgga_values = (
    expression.loc[
        common_ids,
        validation_genes,
    ]
    .to_numpy(
        dtype=float
    )
)


def expression_summary(
    name,
    values,
):

    finite = values[
        np.isfinite(
            values
        )
    ]

    print(
        f"\n{name}"
    )

    print(
        f"min     = {np.min(finite):.4f}"
    )

    print(
        f"Q25     = {np.quantile(finite, .25):.4f}"
    )

    print(
        f"median  = {np.median(finite):.4f}"
    )

    print(
        f"Q75     = {np.quantile(finite, .75):.4f}"
    )

    print(
        f"Q95     = {np.quantile(finite, .95):.4f}"
    )

    print(
        f"max     = {np.max(finite):.4f}"
    )


expression_summary(
    "TCGA development",
    tcga_values,
)


expression_summary(
    "CGGA external",
    cgga_values,
)


print(
    "\nPer-gene median comparison:"
)


audit_rows = []


for gene in validation_genes:

    x_tcga = pd.to_numeric(
        tcga_expression.loc[
            development_ids,
            gene,
        ],
        errors="coerce",
    )

    x_cgga = pd.to_numeric(
        expression.loc[
            common_ids,
            gene,
        ],
        errors="coerce",
    )

    audit_rows.append(

        {

            "gene": gene,

            "tcga_median":
                float(
                    np.nanmedian(
                        x_tcga
                    )
                ),

            "tcga_sd":
                float(
                    np.nanstd(
                        x_tcga
                    )
                ),

            "cgga_median":
                float(
                    np.nanmedian(
                        x_cgga
                    )
                ),

            "cgga_sd":
                float(
                    np.nanstd(
                        x_cgga
                    )
                ),

        }

    )


audit_df = pd.DataFrame(
    audit_rows
)


print(
    audit_df.to_string(
        index=False
    )
)


# =============================================================================
# REMOVE PATIENTS WITH MISSING VALUES
# =============================================================================

section(
    "STEP 10 — COMPLETE-CASE CHECK"
)


X_cgga_df = (
    expression.loc[
        common_ids,
        validation_genes,
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)


valid_expression_mask = (
    X_cgga_df
    .notna()
    .all(
        axis=1
    )
)


removed = int(
    (
        ~valid_expression_mask
    ).sum()
)


print(
    "Patients removed because of missing expression:",
    removed,
)


valid_ids = (
    X_cgga_df.index[
        valid_expression_mask
    ]
    .tolist()
)


X_cgga_df = (
    X_cgga_df.loc[
        valid_ids
    ]
)


cgga_final = (
    cgga.loc[
        valid_ids
    ]
)


print(
    "Final CGGA validation n:",
    len(
        cgga_final
    ),
)


print(
    "Final CGGA events:",
    int(
        cgga_final[
            "OS_event"
        ].sum()
    ),
)


# =============================================================================
# TRAIN FROZEN TCGA MODEL
# =============================================================================

section(
    "STEP 11 — TRAIN FROZEN V4 M2 MODEL ON TCGA DEVELOPMENT"
)


X_train = (
    tcga_expression.loc[
        development_ids,
        validation_genes,
    ]
    .to_numpy(
        dtype=np.float64
    )
)


y_train = make_survival_y(

    tcga_survival.loc[
        development_ids
    ]

)


model = RandomSurvivalForest(
    **RSF_PARAMS
)


model.fit(
    X_train,
    y_train,
)


print(
    "Frozen RSF fitted."
)


# =============================================================================
# CGGA EXTERNAL VALIDATION
# =============================================================================

section(
    "STEP 12 — CGGA EXTERNAL VALIDATION"
)


X_cgga = (
    X_cgga_df
    .to_numpy(
        dtype=np.float64
    )
)


y_cgga = make_survival_y(
    cgga_final
)


risk_cgga = (
    model.predict(
        X_cgga
    )
)


external_cindex = cindex(
    y_cgga,
    risk_cgga,
)


print(
    "\n"
    + "*" * 90
)


print(
    f"CGGA EXTERNAL C-INDEX = "
    f"{external_cindex:.4f}"
)


print(
    "*" * 90
)


# =============================================================================
# SAVE RESULTS
# =============================================================================

section(
    "STEP 13 — SAVE EXTERNAL VALIDATION RESULTS"
)


OUTPUT_DIR = (
    STAGE_D_DIR
    / "CGGA_EXTERNAL_VALIDATION_FINAL"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


cohort_out = (
    cgga_final.reset_index()
)


cohort_out.to_csv(

    OUTPUT_DIR
    / "CGGA_FINAL_IDHWT_GBM_COHORT.csv",

    index=False,

)


audit_df.to_csv(

    OUTPUT_DIR
    / "CGGA_TCGA_EXPRESSION_SCALE_AUDIT.csv",

    index=False,

)


prediction_df = pd.DataFrame(

    {

        "patient_id":
            cgga_final.index,

        "OS_months":
            cgga_final[
                "OS_months"
            ].values,

        "OS_event":
            cgga_final[
                "OS_event"
            ].values,

        "predicted_risk":
            risk_cgga,

    }

)


prediction_df.to_csv(

    OUTPUT_DIR
    / "CGGA_V4_M2_EXTERNAL_PREDICTIONS.csv",

    index=False,

)


summary_df = pd.DataFrame(

    [

        {

            "model":
                "V4_M2_RSF",

            "external_dataset":
                "CGGA_mRNAseq_693",

            "cohort":
                "Primary adult GBM WHO IV IDH-wildtype",

            "n":
                len(
                    cgga_final
                ),

            "events":
                int(
                    cgga_final[
                        "OS_event"
                    ].sum()
                ),

            "genes_requested":
                len(
                    V4_GENES
                ),

            "genes_available":
                len(
                    validation_genes
                ),

            "missing_genes":
                ";".join(
                    missing_genes
                ),

            "c_index":
                external_cindex,

        }

    ]

)


summary_df.to_csv(

    OUTPUT_DIR
    / "CGGA_V4_M2_EXTERNAL_SUMMARY.csv",

    index=False,

)


print(
    "Saved to:"
)

print(
    OUTPUT_DIR
)


print(
    "\nMain result file:"
)

print(
    OUTPUT_DIR
    / "CGGA_V4_M2_EXTERNAL_SUMMARY.csv"
)


print(
    "\nPredictions:"
)

print(
    OUTPUT_DIR
    / "CGGA_V4_M2_EXTERNAL_PREDICTIONS.csv"
)


print(
    "\nExpression audit:"
)

print(
    OUTPUT_DIR
    / "CGGA_TCGA_EXPRESSION_SCALE_AUDIT.csv"
)


print(
    "\nDONE."
)

### STEP 5 — CGGA FORMAT: GENE SYMBOLS IN ROWS, PATIENTS IN COLUMNS

In [ ]:
# =============================================================================
# STEP 5 — CGGA FORMAT: GENE SYMBOLS IN ROWS, PATIENTS IN COLUMNS
# =============================================================================

section("STEP 5 — PREPARE CGGA GENE-SYMBOL EXPRESSION MATRIX")

import requests
import time


# We already know from STEP 4 that:
#   first column = gene_name
#   remaining columns = CGGA patients
#
# Example:
#   A1BG, A1BG-AS1, A1CF, A2M...


if "gene_name" not in rna_raw.columns:
    raise RuntimeError(
        "Expected a 'gene_name' column in CGGA RNA file, but it was not found."
    )


rna_raw["gene_name"] = (
    rna_raw["gene_name"]
    .astype(str)
    .str.strip()
)


print("Number of gene-symbol rows:", len(rna_raw))
print("Unique gene symbols:", rna_raw["gene_name"].nunique())


# =============================================================================
# STEP 6 — MAP FROZEN V4 ENSEMBL IDs TO HGNC/GENE SYMBOLS
# =============================================================================

section("STEP 6 — MAP V4 ENSEMBL IDS TO CGGA GENE SYMBOLS")


def ensembl_to_symbol(ensembl_id):
    """
    Query Ensembl REST API for the display_name corresponding
    to an Ensembl gene stable ID.
    """

    url = (
        "https://rest.ensembl.org"
        f"/lookup/id/{ensembl_id}"
    )

    headers = {
        "Content-Type": "application/json"
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=20
        )

        if response.status_code != 200:
            return None

        data = response.json()

        return data.get("display_name", None)

    except Exception as e:
        print(
            f"Mapping error for {ensembl_id}: {e}"
        )
        return None


ensembl_symbol_map = {}

for i, gene in enumerate(V4_GENES, start=1):

    symbol = ensembl_to_symbol(gene)

    ensembl_symbol_map[gene] = symbol

    print(
        f"{i:02d}/{len(V4_GENES)}  "
        f"{gene} -> {symbol}"
    )

    # Be polite to API
    time.sleep(0.15)


mapping_df = pd.DataFrame(
    {
        "ensembl_id": list(ensembl_symbol_map.keys()),
        "gene_symbol": list(ensembl_symbol_map.values()),
    }
)


print("\nV4 Ensembl-to-symbol mapping:")
print(mapping_df.to_string(index=False))


# =============================================================================
# STEP 7 — CHECK SYMBOLS AGAINST CGGA
# =============================================================================

section("STEP 7 — CHECK V4 GENE AVAILABILITY IN CGGA")


cgga_gene_symbols = set(
    rna_raw["gene_name"]
)


mapping_df["found_in_CGGA"] = (
    mapping_df["gene_symbol"]
    .isin(cgga_gene_symbols)
)


print(
    mapping_df.to_string(index=False)
)


found_mapping = mapping_df.loc[
    mapping_df["found_in_CGGA"]
].copy()


missing_mapping = mapping_df.loc[
    ~mapping_df["found_in_CGGA"]
].copy()


print(
    f"\nMapped and found in CGGA: "
    f"{len(found_mapping)}/{len(V4_GENES)}"
)


if len(missing_mapping) > 0:

    print("\nMissing/unresolved genes:")

    print(
        missing_mapping[
            ["ensembl_id", "gene_symbol"]
        ].to_string(index=False)
    )


if len(found_mapping) < 20:
    raise RuntimeError(
        "\nFewer than 20/25 frozen V4 genes could be mapped "
        "and found in CGGA. Stop validation."
    )


# =============================================================================
# STEP 8 — BUILD CGGA PATIENT x GENE MATRIX
# =============================================================================

section("STEP 8 — BUILD CGGA EXPRESSION MATRIX")


# Handle duplicated gene symbols safely:
# if multiple rows map to the same gene symbol, sum raw counts.
#
# This is preferable for raw-count data where duplicated annotation rows
# represent the same gene symbol.

cgga_symbol_counts = (
    rna_raw
    .groupby(
        "gene_name",
        as_index=True
    )
    .sum(
        numeric_only=True
    )
)


print(
    "Collapsed symbol-level matrix:",
    cgga_symbol_counts.shape
)


# Clinical patient IDs that are also RNA columns
matched_patient_ids = [
    pid
    for pid in cgga.index
    if pid in cgga_symbol_counts.columns
]


print(
    "Eligible clinical patients with RNA:",
    len(matched_patient_ids)
)


if len(matched_patient_ids) < 20:
    raise RuntimeError(
        "Too few CGGA patients matched between clinical and RNA files."
    )


# Selected symbol list
VALIDATION_ENSEMBL = (
    found_mapping["ensembl_id"]
    .tolist()
)

VALIDATION_SYMBOLS = (
    found_mapping["gene_symbol"]
    .tolist()
)


# genes x patients -> patients x genes
X_cgga_raw_symbol = (
    cgga_symbol_counts.loc[
        VALIDATION_SYMBOLS,
        matched_patient_ids
    ]
    .T
    .copy()
)


# Rename CGGA columns back to Ensembl IDs.
# This ensures the model input uses the exact same feature names/order as TCGA.
symbol_to_ensembl = dict(
    zip(
        found_mapping["gene_symbol"],
        found_mapping["ensembl_id"]
    )
)


X_cgga_raw_symbol = (
    X_cgga_raw_symbol
    .rename(
        columns=symbol_to_ensembl
    )
)


# Enforce exact frozen ordering
X_cgga_raw_symbol = (
    X_cgga_raw_symbol[
        VALIDATION_ENSEMBL
    ]
)


print(
    "Selected CGGA matrix:",
    X_cgga_raw_symbol.shape
)


# =============================================================================
# STEP 9 — LOAD TCGA DEVELOPMENT DATA
# =============================================================================

section("STEP 9 — LOAD TCGA DEVELOPMENT DATA")


split_df = pd.read_csv(
    SPLIT_FILE
)


development_ids = (
    split_df.loc[
        split_df["split"] == "development",
        "patient_id"
    ]
    .astype(str)
    .tolist()
)


tcga_survival = (
    pd.read_csv(
        TCGA_SURVIVAL_FILE
    )
    .set_index("patient_id")
)


tcga_expression = pd.read_csv(
    TCGA_EXPRESSION_FILE,
    index_col=0
)


tcga_survival.index = (
    tcga_survival.index.astype(str)
)

tcga_expression.index = (
    tcga_expression.index.astype(str)
)


tcga_expression.columns = [
    clean_gene_id(x)
    for x in tcga_expression.columns
]


missing_tcga = [
    gene
    for gene in VALIDATION_ENSEMBL
    if gene not in tcga_expression.columns
]


if missing_tcga:
    raise RuntimeError(
        f"Frozen genes unexpectedly missing from TCGA:\n"
        f"{missing_tcga}"
    )


X_tcga_df = (
    tcga_expression.loc[
        development_ids,
        VALIDATION_ENSEMBL
    ]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


print(
    "TCGA development matrix:",
    X_tcga_df.shape
)


# =============================================================================
# STEP 10 — COMPUTE CGGA LOG2(CPM + 1)
# =============================================================================

section("STEP 10 — COMPUTE CGGA LOG2(CPM + 1)")


# IMPORTANT:
# CPM MUST be calculated using the entire RNA matrix,
# not only the selected V4 genes.

all_cgga_counts = (
    cgga_symbol_counts[
        matched_patient_ids
    ]
    .T
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)


# Counts must not be negative
if (
    all_cgga_counts
    .lt(0)
    .any()
    .any()
):
    raise RuntimeError(
        "Negative values detected in a raw Read_Counts file."
    )


library_sizes = (
    all_cgga_counts
    .sum(axis=1)
)


print(
    "Median library size:",
    float(library_sizes.median())
)

print(
    "Minimum library size:",
    float(library_sizes.min())
)

print(
    "Maximum library size:",
    float(library_sizes.max())
)


if (
    library_sizes <= 0
).any():
    raise RuntimeError(
        "At least one CGGA patient has a zero library size."
    )


cgga_cpm_all = (
    all_cgga_counts
    .div(
        library_sizes,
        axis=0
    )
    * 1_000_000.0
)


cgga_logcpm_all = np.log2(
    cgga_cpm_all + 1.0
)


# Select the V4 symbols
X_cgga_logcpm = (
    cgga_logcpm_all[
        VALIDATION_SYMBOLS
    ]
    .copy()
)


# Rename symbols -> frozen Ensembl IDs
X_cgga_logcpm = (
    X_cgga_logcpm
    .rename(
        columns=symbol_to_ensembl
    )
)


X_cgga_logcpm = (
    X_cgga_logcpm[
        VALIDATION_ENSEMBL
    ]
)


print(
    "CGGA log2(CPM+1) matrix:",
    X_cgga_logcpm.shape
)


# =============================================================================
# STEP 11 — TCGA vs CGGA SCALE AUDIT
# =============================================================================

section("STEP 11 — TCGA vs CGGA EXPRESSION SCALE AUDIT")


summarize_values(
    "TCGA DEVELOPMENT",
    X_tcga_df.to_numpy()
)


summarize_values(
    "CGGA RAW COUNTS",
    X_cgga_raw_symbol.to_numpy()
)


summarize_values(
    "CGGA LOG2(CPM+1)",
    X_cgga_logcpm.to_numpy()
)


audit_rows = []


for gene in VALIDATION_ENSEMBL:

    symbol = ensembl_symbol_map[gene]

    tcga_vals = (
        X_tcga_df[gene]
        .to_numpy(dtype=float)
    )

    raw_vals = (
        X_cgga_raw_symbol[gene]
        .to_numpy(dtype=float)
    )

    log_vals = (
        X_cgga_logcpm[gene]
        .to_numpy(dtype=float)
    )


    audit_rows.append(
        {
            "ensembl_id": gene,
            "gene_symbol": symbol,

            "TCGA_median":
                float(np.nanmedian(tcga_vals)),

            "TCGA_sd":
                float(np.nanstd(tcga_vals)),

            "CGGA_raw_median":
                float(np.nanmedian(raw_vals)),

            "CGGA_raw_sd":
                float(np.nanstd(raw_vals)),

            "CGGA_logCPM_median":
                float(np.nanmedian(log_vals)),

            "CGGA_logCPM_sd":
                float(np.nanstd(log_vals)),
        }
    )


audit_df = pd.DataFrame(
    audit_rows
)


print("\nPER-GENE AUDIT:")
print(
    audit_df.to_string(index=False)
)


# =============================================================================
# STEP 12 — FINAL MATCHED CGGA COHORT
# =============================================================================

section("STEP 12 — FINAL MATCHED CGGA COHORT")


valid_expression = (
    X_cgga_logcpm
    .notna()
    .all(axis=1)
)


valid_ids = (
    X_cgga_logcpm
    .index[valid_expression]
    .tolist()
)


cgga_final = (
    cgga.loc[
        valid_ids
    ]
    .copy()
)


X_cgga_logcpm = (
    X_cgga_logcpm.loc[
        valid_ids
    ]
)


X_cgga_raw_symbol = (
    X_cgga_raw_symbol.loc[
        valid_ids
    ]
)


print(
    "Final CGGA n:",
    len(cgga_final)
)

print(
    "Events:",
    int(cgga_final["OS_event"].sum())
)

print(
    "Censored:",
    int(
        len(cgga_final)
        - cgga_final["OS_event"].sum()
    )
)

print(
    "Median age:",
    float(cgga_final["Age"].median())
)

print(
    "Median OS months:",
    float(cgga_final["OS_months"].median())
)


# =============================================================================
# STEP 13 — TRAIN FROZEN TCGA V4 M2 MODEL
# =============================================================================

section("STEP 13 — TRAIN FROZEN TCGA V4 M2 MODEL")


X_train = (
    X_tcga_df
    .to_numpy(
        dtype=np.float64
    )
)


y_train = make_survival_y(
    tcga_survival.loc[
        development_ids
    ]
)


model = RandomSurvivalForest(
    **RSF_PARAMS
)


model.fit(
    X_train,
    y_train
)


print(
    "Frozen RSF fitted on:",
    len(X_train),
    "TCGA development patients"
)


# =============================================================================
# STEP 14 — CGGA EXTERNAL DIAGNOSTIC
# =============================================================================

section("STEP 14 — CGGA EXTERNAL VALIDATION DIAGNOSTIC")


y_cgga = make_survival_y(
    cgga_final
)


# RAW counts — diagnostic only
X_raw = (
    X_cgga_raw_symbol
    .to_numpy(dtype=np.float64)
)


risk_raw = model.predict(
    X_raw
)


c_raw = cindex(
    y_cgga,
    risk_raw
)


# LOG2 CPM — main compatibility candidate,
# but still requires audit before we call it valid.
X_log = (
    X_cgga_logcpm
    .to_numpy(dtype=np.float64)
)


risk_log = model.predict(
    X_log
)


c_log = cindex(
    y_cgga,
    risk_log
)


print("\n" + "*" * 90)

print(
    f"CGGA C-index — RAW COUNTS: "
    f"{c_raw:.4f}"
)

print(
    f"CGGA C-index — LOG2(CPM+1): "
    f"{c_log:.4f}"
)

print("*" * 90)


print(
    "\nDO NOT choose the higher C-index."
)

print(
    "The correct representation must be decided from "
    "preprocessing compatibility with the TCGA training data, "
    "not from CGGA survival performance."
)


# =============================================================================
# STEP 15 — SAVE RESULTS
# =============================================================================

section("STEP 15 — SAVE RESULTS")


OUTPUT_DIR = (
    STAGE_D_DIR
    / "CGGA_EXTERNAL_VALIDATION_SYMBOL_MAPPING"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


mapping_df.to_csv(
    OUTPUT_DIR
    / "V4_ENSEMBL_TO_CGGA_SYMBOL_MAPPING.csv",
    index=False
)


audit_df.to_csv(
    OUTPUT_DIR
    / "TCGA_CGGA_EXPRESSION_SCALE_AUDIT.csv",
    index=False
)


cgga_final.reset_index().to_csv(
    OUTPUT_DIR
    / "CGGA_FINAL_VALIDATION_COHORT.csv",
    index=False
)


prediction_df = pd.DataFrame(
    {
        "patient_id":
            cgga_final.index,

        "OS_months":
            cgga_final["OS_months"].values,

        "OS_event":
            cgga_final["OS_event"].values,

        "risk_raw_counts":
            risk_raw,

        "risk_log2CPM":
            risk_log,
    }
)


prediction_df.to_csv(
    OUTPUT_DIR
    / "CGGA_V4_M2_PREDICTIONS.csv",
    index=False
)


summary_df = pd.DataFrame(
    [
        {
            "model":
                "V4_M2_RSF",

            "dataset":
                "CGGA_mRNAseq_693",

            "eligible_clinical_n":
                104,

            "final_matched_n":
                len(cgga_final),

            "events":
                int(cgga_final["OS_event"].sum()),

            "requested_genes":
                25,

            "mapped_found_genes":
                len(VALIDATION_ENSEMBL),

            "cindex_raw_counts":
                c_raw,

            "cindex_log2CPM":
                c_log,
        }
    ]
)


summary_df.to_csv(
    OUTPUT_DIR
    / "CGGA_V4_M2_SUMMARY.csv",
    index=False
)


print("\nSaved to:")
print(OUTPUT_DIR)


section("DONE")

print("Check these sections:")
print("STEP 6")
print("STEP 7")
print("STEP 11")
print("STEP 12")
print("STEP 14")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DOWNLOADS = BASE

print("\n" + "=" * 80)
print("STEP 2B — MAP CGGA SYMBOLS TO THE FROZEN TCGA/STRING GENE SPACE")
print("=" * 80)

MAPPING_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
    / "TCGA_GBM_gene_to_STRING_mapping.csv"
)

FINAL_REF_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_FINAL_REFERENCE_EDGES_20PCT.csv"
)

checkpoint(MAPPING_FILE.exists(), "full TCGA↔STRING mapping file exists")
checkpoint(FINAL_REF_FILE.exists(), "final frozen 20% reference network exists")

mapping = pd.read_csv(MAPPING_FILE, low_memory=False)
ref20 = pd.read_csv(FINAL_REF_FILE, low_memory=False)

print("\nFull mapping shape:", mapping.shape)
print("Mapping columns:")
print(mapping.columns.tolist())

print("\nFrozen reference-network shape:", ref20.shape)
print("Reference-network columns:")
print(ref20.columns.tolist())

print("\nFirst 5 mapping rows:")
print(mapping.head())

print("\nFirst 5 frozen reference rows:")
print(ref20.head())

# ============================================================
# IDENTIFY MAPPING COLUMNS
# ============================================================

ensembl_col = next(
    (c for c in mapping.columns if c.lower() == "ensembl_id"),
    None
)

symbol_candidates = [
    c for c in mapping.columns
    if any(
        key in c.lower()
        for key in ["gene_name", "symbol", "preferred_name", "preferred"]
    )
]

print("\nEnsembl column:", ensembl_col)
print("Possible symbol columns:", symbol_candidates)

checkpoint(
    ensembl_col is not None,
    "mapping contains Ensembl IDs"
)

# show candidate symbol columns
for c in symbol_candidates:
    print(f"\nCOLUMN: {c}")
    print("Non-null:", mapping[c].notna().sum())
    print("Unique:", mapping[c].nunique(dropna=True))
    print("Examples:")
    print(mapping[c].dropna().astype(str).head(20).tolist())

# ============================================================
# ALSO AUDIT THE FINAL 90,959-EDGE UNIVERSE
# ============================================================

print("\n" + "-" * 80)
print("LOCATE FROZEN 90,959-EDGE FILE")
print("-" * 80)

candidate_files = list(
    (DOWNLOADS / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING").glob("*.csv")
)

for f in candidate_files:
    try:
        tmp = pd.read_csv(f, nrows=5)
        cols = tmp.columns.tolist()

        if {"gene1", "gene2"}.issubset(cols):
            nrows = sum(1 for _ in open(f, encoding="utf-8", errors="ignore")) - 1

            print(
                f.name,
                "| rows =", nrows,
                "| columns =", cols
            )
    except Exception:
        pass

print("\n" + "=" * 80)
print("STEP 2B COMPLETE — STOP HERE")
print("=" * 80)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pandas as pd
from pathlib import Path

DOWNLOADS = BASE

COUNTS_FILE = (
    DOWNLOADS
    / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"
    / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"
)

print("Counts file:", COUNTS_FILE)
print("Exists:", COUNTS_FILE.exists())
print("Is file:", COUNTS_FILE.is_file())

counts = pd.read_csv(
    COUNTS_FILE,
    sep="\t",
    low_memory=False
)

print("CGGA counts shape:", counts.shape)
print("Gene column:", counts.columns[0])

assert counts.shape[0] == 55523
assert counts.columns[0] == "gene_name"

print("[CHECKPOINT PASS] CGGA count matrix reloaded")
# ============================================================
# SETUP
# ============================================================

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

print("\n" + "=" * 80)
print("STEP 2C — FROZEN NETWORK COMPATIBILITY WITH CGGA")
print("=" * 80)

MAPPING_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
    / "TCGA_GBM_gene_to_STRING_mapping.csv"
)

STAGE_B_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)

EDGE_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_FINAL_FROZEN_EDGE_UNIVERSE.csv"
)

OUT_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"
OUT_DIR.mkdir(exist_ok=True)

# ============================================================
# LOAD FROZEN TRAINING-SIDE DEFINITIONS
# ============================================================

mapping = pd.read_csv(MAPPING_FILE, low_memory=False)
stage_b = pd.read_csv(STAGE_B_FILE, low_memory=False)
edges = pd.read_csv(EDGE_FILE, low_memory=False)

checkpoint(len(edges) == 90959, "frozen network contains exactly 90,959 edges")

checkpoint(
    {"gene1", "gene2"}.issubset(edges.columns),
    "frozen network contains gene1/gene2 Ensembl endpoints"
)

# clean identifiers
for df, col in [
    (mapping, "ensembl_id"),
    (stage_b, "ensembl_id"),
    (edges, "gene1"),
    (edges, "gene2"),
]:
    df[col] = (
        df[col]
        .astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

mapping["gene_name"] = mapping["gene_name"].astype(str).str.strip()

# ============================================================
# FROZEN NETWORK GENE UNIVERSE
# ============================================================

network_genes = sorted(
    set(edges["gene1"]) |
    set(edges["gene2"])
)

print("\nFrozen candidate edges:", len(edges))
print("Frozen network genes:", len(network_genes))

# ============================================================
# BUILD ORIGINAL Ensembl -> SYMBOL CROSSWALK
# ============================================================

map_network = (
    mapping[
        mapping["ensembl_id"].isin(network_genes)
    ][["ensembl_id", "gene_name"]]
    .drop_duplicates()
    .copy()
)

print("\nNetwork genes with mapping rows:",
      map_network["ensembl_id"].nunique())

# Ensembl ID mapping to >1 gene symbol?
ensembl_multi = (
    map_network
    .groupby("ensembl_id")["gene_name"]
    .nunique()
)

ensembl_multi = ensembl_multi[ensembl_multi > 1]

print("Ensembl IDs mapping to >1 symbol:",
      len(ensembl_multi))

checkpoint(
    len(ensembl_multi) == 0,
    "each frozen Ensembl ID maps to one gene symbol"
)

# ============================================================
# AUDIT SYMBOL -> MULTIPLE ENSEMBL IDs
# ============================================================

symbol_multi = (
    map_network
    .groupby("gene_name")["ensembl_id"]
    .nunique()
)

ambiguous_symbols = set(
    symbol_multi[symbol_multi > 1].index
)

print("\nAmbiguous gene symbols inside frozen network:",
      len(ambiguous_symbols))

if ambiguous_symbols:
    ambiguous_table = (
        map_network[
            map_network["gene_name"].isin(ambiguous_symbols)
        ]
        .sort_values(["gene_name", "ensembl_id"])
    )

    print("\nAmbiguous mappings:")
    print(ambiguous_table.to_string(index=False))

    ambiguous_table.to_csv(
        OUT_DIR / "CGGA_AMBIGUOUS_FROZEN_SYMBOLS.csv",
        index=False
    )

# ============================================================
# CGGA SYMBOL UNIVERSE
# ============================================================

cgga_symbols = set(
    counts["gene_name"]
    .astype(str)
    .str.strip()
)

print("\nCGGA unique symbols:", len(cgga_symbols))

# ============================================================
# BUILD FROZEN-GENE COMPATIBILITY TABLE
# ============================================================

gene_compat = (
    pd.DataFrame({"ensembl_id": network_genes})
    .merge(
        map_network,
        on="ensembl_id",
        how="left"
    )
)

gene_compat["has_symbol_mapping"] = (
    gene_compat["gene_name"].notna()
)

gene_compat["symbol_in_CGGA"] = (
    gene_compat["gene_name"].isin(cgga_symbols)
)

gene_compat["ambiguous_symbol"] = (
    gene_compat["gene_name"].isin(ambiguous_symbols)
)

# safest externally usable definition
gene_compat["externally_usable"] = (
    gene_compat["has_symbol_mapping"]
    & gene_compat["symbol_in_CGGA"]
    & ~gene_compat["ambiguous_symbol"]
)

n_network = len(gene_compat)
n_mapped = gene_compat["has_symbol_mapping"].sum()
n_in_cgga = gene_compat["symbol_in_CGGA"].sum()
n_usable = gene_compat["externally_usable"].sum()

print("\n" + "-" * 80)
print("GENE COVERAGE")
print("-" * 80)

print(f"Frozen network genes:      {n_network}")
print(f"With symbol mapping:       {n_mapped}")
print(f"Symbol present in CGGA:    {n_in_cgga}")
print(f"Externally usable genes:   {n_usable}")

print(
    f"Usable-gene coverage:      "
    f"{100*n_usable/n_network:.2f}%"
)

# ============================================================
# FROZEN ANCHORS
# ============================================================

anchor_mask = (
    stage_b["Is_DEG_Anchor"].astype(str)
    .str.lower()
    .isin(["true", "1"])
)

anchor_ids_stageb = set(
    stage_b.loc[anchor_mask, "ensembl_id"]
)

# Only anchors actually surviving into final frozen edge universe
frozen_anchor_ids = sorted(
    anchor_ids_stageb & set(network_genes)
)

print("\n" + "-" * 80)
print("DEG-ANCHOR COVERAGE")
print("-" * 80)

print("Stage-B connected DEG anchors:",
      len(anchor_ids_stageb))

print("DEG anchors in final frozen network:",
      len(frozen_anchor_ids))

checkpoint(
    len(frozen_anchor_ids) == 3992,
    "final frozen network contains 3,992 DEG anchors"
)

anchor_compat = gene_compat[
    gene_compat["ensembl_id"].isin(frozen_anchor_ids)
].copy()

n_anchor_usable = anchor_compat["externally_usable"].sum()

print("Anchors available in CGGA:",
      int(n_anchor_usable))

print(
    "Anchor coverage:",
    f"{100*n_anchor_usable/3992:.2f}%"
)

# ============================================================
# EDGE RECONSTRUCTABILITY
# ============================================================

usable_gene_set = set(
    gene_compat.loc[
        gene_compat["externally_usable"],
        "ensembl_id"
    ]
)

edges["gene1_available_CGGA"] = edges["gene1"].isin(usable_gene_set)
edges["gene2_available_CGGA"] = edges["gene2"].isin(usable_gene_set)

edges["reconstructable_CGGA"] = (
    edges["gene1_available_CGGA"]
    & edges["gene2_available_CGGA"]
)

n_edges_reconstructable = edges["reconstructable_CGGA"].sum()

print("\n" + "-" * 80)
print("EDGE COVERAGE")
print("-" * 80)

print("Frozen candidate edges:",
      len(edges))

print("Reconstructable CGGA edges:",
      int(n_edges_reconstructable))

print(
    "Edge coverage:",
    f"{100*n_edges_reconstructable/90959:.2f}%"
)

print(
    "Edges missing ≥1 endpoint:",
    int((~edges["reconstructable_CGGA"]).sum())
)

# ============================================================
# MISSING ANCHOR AUDIT
# ============================================================

missing_anchors = anchor_compat[
    ~anchor_compat["externally_usable"]
].copy()

print("\nMissing/unusable anchors:",
      len(missing_anchors))

if len(missing_anchors) > 0:
    print("\nFirst 30 missing/unusable anchors:")
    print(
        missing_anchors[
            [
                "ensembl_id",
                "gene_name",
                "has_symbol_mapping",
                "symbol_in_CGGA",
                "ambiguous_symbol",
            ]
        ]
        .head(30)
        .to_string(index=False)
    )

# ============================================================
# SAVE AUDIT
# ============================================================

gene_compat.to_csv(
    OUT_DIR / "CGGA_FROZEN_NETWORK_GENE_COMPATIBILITY.csv",
    index=False
)

anchor_compat.to_csv(
    OUT_DIR / "CGGA_FROZEN_ANCHOR_COMPATIBILITY.csv",
    index=False
)

missing_anchors.to_csv(
    OUT_DIR / "CGGA_MISSING_OR_AMBIGUOUS_ANCHORS.csv",
    index=False
)

edges[
    ~edges["reconstructable_CGGA"]
].to_csv(
    OUT_DIR / "CGGA_UNRECONSTRUCTABLE_FROZEN_EDGES.csv",
    index=False
)

summary = pd.DataFrame({
    "Metric": [
        "Frozen network genes",
        "Externally usable genes",
        "Gene coverage %",
        "Frozen DEG anchors",
        "Externally usable DEG anchors",
        "Anchor coverage %",
        "Frozen candidate edges",
        "Reconstructable CGGA edges",
        "Edge coverage %",
        "Ambiguous frozen-network symbols",
    ],
    "Value": [
        n_network,
        n_usable,
        100 * n_usable / n_network,
        3992,
        n_anchor_usable,
        100 * n_anchor_usable / 3992,
        90959,
        n_edges_reconstructable,
        100 * n_edges_reconstructable / 90959,
        len(ambiguous_symbols),
    ]
})

summary.to_csv(
    OUT_DIR / "CGGA_FROZEN_NETWORK_COMPATIBILITY_SUMMARY.csv",
    index=False
)

print("\n" + "=" * 80)
print("STEP 2C COMPLETE — STOP HERE")
print("=" * 80)

print("\nCompatibility summary:")
print(summary.to_string(index=False))

print("""
DO NOT normalize CGGA yet.
DO NOT calculate sPCC yet.
DO NOT calculate eNESH yet.
""")

### STEP 2D — HGNC ALIAS RECOVERY FOR CGGA

In [ ]:
# ============================================================
# STEP 2D — HGNC ALIAS RECOVERY FOR CGGA
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date
import urllib.request

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

OUT_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"
OUT_DIR.mkdir(exist_ok=True)

MAPPING_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STRING"
    / "TCGA_GBM_gene_to_STRING_mapping.csv"
)

EDGE_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_FINAL_FROZEN_EDGE_UNIVERSE.csv"
)

STAGE_B_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)

COUNTS_FILE = (
    DOWNLOADS
    / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"
    / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"
)

print("\n" + "=" * 80)
print("STEP 2D — HGNC ALIAS RECOVERY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load frozen data
# ------------------------------------------------------------

mapping = pd.read_csv(MAPPING_FILE, low_memory=False)
edges = pd.read_csv(EDGE_FILE, low_memory=False)
stage_b = pd.read_csv(STAGE_B_FILE, low_memory=False)

# Read only gene names from CGGA — faster
cgga_gene_df = pd.read_csv(
    COUNTS_FILE,
    sep="\t",
    usecols=["gene_name"],
    low_memory=False
)

cgga_symbols = set(
    cgga_gene_df["gene_name"]
    .astype(str)
    .str.strip()
)

for df, col in [
    (mapping, "ensembl_id"),
    (edges, "gene1"),
    (edges, "gene2"),
    (stage_b, "ensembl_id"),
]:
    df[col] = (
        df[col]
        .astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

mapping["gene_name"] = (
    mapping["gene_name"]
    .astype(str)
    .str.strip()
)

network_genes = sorted(
    set(edges["gene1"]) |
    set(edges["gene2"])
)

checkpoint(
    len(edges) == 90959,
    "frozen candidate network contains 90,959 edges"
)

checkpoint(
    len(network_genes) == 13039,
    "frozen network contains 13,039 genes"
)

# ------------------------------------------------------------
# 2. Direct mapping already established
# ------------------------------------------------------------

base_map = (
    mapping[
        mapping["ensembl_id"].isin(network_genes)
    ][["ensembl_id", "gene_name"]]
    .drop_duplicates("ensembl_id")
    .copy()
)

base_map["direct_present"] = (
    base_map["gene_name"].isin(cgga_symbols)
)

direct = base_map[
    base_map["direct_present"]
].copy()

direct = direct.rename(
    columns={"gene_name": "cgga_symbol"}
)

direct["mapping_type"] = "direct_frozen_symbol"

missing_ids = set(
    base_map.loc[
        ~base_map["direct_present"],
        "ensembl_id"
    ]
)

print("\nDirectly mapped frozen genes:", len(direct))
print("Genes requiring alias investigation:", len(missing_ids))

checkpoint(
    len(direct) == 12338,
    "direct CGGA mapping reproduces previous 12,338 genes"
)

# ------------------------------------------------------------
# 3. Download official HGNC complete set
# ------------------------------------------------------------

HGNC_URL = (
    "https://storage.googleapis.com/"
    "public-download-files/hgnc/tsv/tsv/"
    "hgnc_complete_set.txt"
)

HGNC_FILE = (
    OUT_DIR
    / f"hgnc_complete_set_accessed_{date.today().isoformat()}.txt"
)

if not HGNC_FILE.exists():
    print("\nDownloading official HGNC complete set...")
    urllib.request.urlretrieve(HGNC_URL, HGNC_FILE)

checkpoint(
    HGNC_FILE.exists(),
    "HGNC complete-set file available"
)

hgnc = pd.read_csv(
    HGNC_FILE,
    sep="\t",
    dtype=str,
    low_memory=False
)

print("\nHGNC shape:", hgnc.shape)

required_cols = {
    "symbol",
    "prev_symbol",
    "alias_symbol",
    "ensembl_gene_id"
}

print("Required HGNC columns available:",
      required_cols.issubset(hgnc.columns))

checkpoint(
    required_cols.issubset(hgnc.columns),
    "HGNC contains symbol / previous / alias / Ensembl fields"
)

# ------------------------------------------------------------
# 4. Expand HGNC Ensembl IDs
# ------------------------------------------------------------

hgnc2 = hgnc[
    [
        "symbol",
        "prev_symbol",
        "alias_symbol",
        "ensembl_gene_id"
    ]
].copy()

hgnc2["ensembl_gene_id"] = (
    hgnc2["ensembl_gene_id"]
    .fillna("")
    .str.split("|")
)

hgnc2 = hgnc2.explode("ensembl_gene_id")

hgnc2["ensembl_gene_id"] = (
    hgnc2["ensembl_gene_id"]
    .astype(str)
    .str.split(".").str[0]
    .str.strip()
)

hgnc_missing = hgnc2[
    hgnc2["ensembl_gene_id"].isin(missing_ids)
].copy()

print("\nMissing frozen genes represented in HGNC:",
      hgnc_missing["ensembl_gene_id"].nunique())

# ------------------------------------------------------------
# 5. Build all approved / previous / alias symbol candidates
# ------------------------------------------------------------

candidate_rows = []

def add_candidate(ensembl_id, value, source):
    if pd.isna(value):
        return

    for token in str(value).split("|"):
        token = token.strip()

        if token and token.lower() != "nan":
            candidate_rows.append({
                "ensembl_id": ensembl_id,
                "candidate_symbol": token,
                "source": source
            })

for _, row in hgnc_missing.iterrows():

    ens = row["ensembl_gene_id"]

    add_candidate(
        ens,
        row["symbol"],
        "HGNC_approved"
    )

    add_candidate(
        ens,
        row["prev_symbol"],
        "HGNC_previous"
    )

    add_candidate(
        ens,
        row["alias_symbol"],
        "HGNC_alias"
    )

candidates = (
    pd.DataFrame(candidate_rows)
    .drop_duplicates()
)

candidates["present_in_CGGA"] = (
    candidates["candidate_symbol"].isin(cgga_symbols)
)

hits = candidates[
    candidates["present_in_CGGA"]
].copy()

print("\nCandidate HGNC symbol hits in CGGA:", len(hits))
print(
    "Missing frozen genes with >=1 CGGA alias hit:",
    hits["ensembl_id"].nunique()
)

# ------------------------------------------------------------
# 6. Require exactly ONE CGGA symbol per frozen Ensembl gene
# ------------------------------------------------------------

n_symbols_per_gene = (
    hits
    .groupby("ensembl_id")["candidate_symbol"]
    .nunique()
)

unique_gene_ids = set(
    n_symbols_per_gene[
        n_symbols_per_gene == 1
    ].index
)

multi_gene_ids = set(
    n_symbols_per_gene[
        n_symbols_per_gene > 1
    ].index
)

print("\nGenes with exactly one CGGA alias candidate:",
      len(unique_gene_ids))

print("Genes with >1 CGGA alias candidate:",
      len(multi_gene_ids))

alias_unique = (
    hits[
        hits["ensembl_id"].isin(unique_gene_ids)
    ][
        [
            "ensembl_id",
            "candidate_symbol"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={"candidate_symbol": "cgga_symbol"}
    )
)

alias_unique["mapping_type"] = "HGNC_alias_or_previous"

# ------------------------------------------------------------
# 7. Check reverse collisions:
# one CGGA row cannot map to multiple frozen Ensembl genes
# ------------------------------------------------------------

tentative = pd.concat(
    [
        direct[
            [
                "ensembl_id",
                "cgga_symbol",
                "mapping_type"
            ]
        ],
        alias_unique
    ],
    ignore_index=True
)

symbol_collision_counts = (
    tentative
    .groupby("cgga_symbol")["ensembl_id"]
    .nunique()
)

collision_symbols = set(
    symbol_collision_counts[
        symbol_collision_counts > 1
    ].index
)

print("\nCGGA symbols mapping to >1 frozen Ensembl ID:",
      len(collision_symbols))

if collision_symbols:

    print("\nCollision examples:")
    print(
        tentative[
            tentative["cgga_symbol"].isin(collision_symbols)
        ]
        .sort_values("cgga_symbol")
        .head(50)
        .to_string(index=False)
    )

# reject all collided assignments
final_map = tentative[
    ~tentative["cgga_symbol"].isin(collision_symbols)
].copy()

checkpoint(
    final_map["ensembl_id"].is_unique,
    "final CGGA mapping has unique frozen Ensembl IDs"
)

checkpoint(
    final_map["cgga_symbol"].is_unique,
    "final CGGA mapping has unique CGGA gene symbols"
)

# ------------------------------------------------------------
# 8. Recalculate gene coverage
# ------------------------------------------------------------

usable_gene_set = set(final_map["ensembl_id"])

print("\n" + "-" * 80)
print("GENE COVERAGE AFTER HGNC RECOVERY")
print("-" * 80)

print("Frozen network genes:", len(network_genes))
print("Directly matched:", len(direct))
print(
    "Recovered through HGNC aliases:",
    int(
        (final_map["mapping_type"] ==
         "HGNC_alias_or_previous").sum()
    )
)

print("Total usable genes:", len(usable_gene_set))

print(
    "Gene coverage:",
    f"{100 * len(usable_gene_set) / 13039:.2f}%"
)

# ------------------------------------------------------------
# 9. Recalculate anchor coverage
# ------------------------------------------------------------

anchor_mask = (
    stage_b["Is_DEG_Anchor"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
)

stage_b_anchors = set(
    stage_b.loc[
        anchor_mask,
        "ensembl_id"
    ]
)

frozen_anchors = (
    stage_b_anchors &
    set(network_genes)
)

checkpoint(
    len(frozen_anchors) == 3992,
    "3,992 frozen DEG anchors identified"
)

usable_anchors = (
    frozen_anchors &
    usable_gene_set
)

print("\n" + "-" * 80)
print("ANCHOR COVERAGE AFTER HGNC RECOVERY")
print("-" * 80)

print("Frozen anchors:", len(frozen_anchors))
print("Usable anchors:", len(usable_anchors))

print(
    "Anchor coverage:",
    f"{100 * len(usable_anchors) / 3992:.2f}%"
)

# ------------------------------------------------------------
# 10. Recalculate edge coverage
# ------------------------------------------------------------

edges["reconstructable_after_alias"] = (
    edges["gene1"].isin(usable_gene_set)
    &
    edges["gene2"].isin(usable_gene_set)
)

n_edges = int(
    edges["reconstructable_after_alias"].sum()
)

print("\n" + "-" * 80)
print("EDGE COVERAGE AFTER HGNC RECOVERY")
print("-" * 80)

print("Frozen edges:", len(edges))
print("Reconstructable edges:", n_edges)
print(
    "Edge coverage:",
    f"{100 * n_edges / 90959:.2f}%"
)

print(
    "Still missing edges:",
    int(90959 - n_edges)
)

# ------------------------------------------------------------
# 11. Residual missing genes / anchors
# ------------------------------------------------------------

residual_genes = (
    set(network_genes) -
    usable_gene_set
)

residual_anchors = (
    frozen_anchors -
    usable_gene_set
)

print("\nResidual missing network genes:",
      len(residual_genes))

print("Residual missing anchors:",
      len(residual_anchors))

# ------------------------------------------------------------
# 12. Save everything
# ------------------------------------------------------------

final_map.to_csv(
    OUT_DIR / "CGGA_FINAL_FROZEN_GENE_MAPPING_AFTER_HGNC.csv",
    index=False
)

hits.to_csv(
    OUT_DIR / "CGGA_HGNC_ALIAS_CANDIDATE_HITS.csv",
    index=False
)

pd.DataFrame({
    "ensembl_id": sorted(residual_genes)
}).to_csv(
    OUT_DIR / "CGGA_RESIDUAL_MISSING_NETWORK_GENES.csv",
    index=False
)

pd.DataFrame({
    "ensembl_id": sorted(residual_anchors)
}).to_csv(
    OUT_DIR / "CGGA_RESIDUAL_MISSING_ANCHORS.csv",
    index=False
)

edges[
    ~edges["reconstructable_after_alias"]
].to_csv(
    OUT_DIR / "CGGA_RESIDUAL_UNRECONSTRUCTABLE_EDGES.csv",
    index=False
)

print("\n" + "=" * 80)
print("STEP 2D COMPLETE — STOP HERE")
print("=" * 80)

print("""
DO NOT normalize CGGA yet.
DO NOT compute sPCC yet.
DO NOT compute eNESH yet.
""")

### STEP 2E — FREEZE CGGA-COMPATIBLE EXTERNAL TOPOLOGY

In [ ]:
# ============================================================
# STEP 2E — FREEZE CGGA-COMPATIBLE EXTERNAL TOPOLOGY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

OUT_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"
OUT_DIR.mkdir(exist_ok=True)

EDGE_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_FINAL_FROZEN_EDGE_UNIVERSE.csv"
)

REF20_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_FINAL_REFERENCE_EDGES_20PCT.csv"
)

FINAL_MAP_FILE = (
    OUT_DIR
    / "CGGA_FINAL_FROZEN_GENE_MAPPING_AFTER_HGNC.csv"
)

STAGE_B_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_B_NETWORK"
    / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
)

print("\n" + "=" * 80)
print("STEP 2E — FREEZE CGGA-COMPATIBLE TOPOLOGY")
print("=" * 80)

edges = pd.read_csv(EDGE_FILE, low_memory=False)
ref20 = pd.read_csv(REF20_FILE, low_memory=False)
gene_map = pd.read_csv(FINAL_MAP_FILE, low_memory=False)
stage_b = pd.read_csv(STAGE_B_FILE, low_memory=False)

for df, col in [
    (edges, "gene1"),
    (edges, "gene2"),
    (ref20, "gene1"),
    (ref20, "gene2"),
    (gene_map, "ensembl_id"),
    (stage_b, "ensembl_id"),
]:
    df[col] = (
        df[col]
        .astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

checkpoint(len(edges) == 90959, "original frozen candidate network = 90,959 edges")
checkpoint(len(ref20) == 18192, "original frozen reference network = 18,192 edges")

usable_genes = set(gene_map["ensembl_id"])

# ============================================================
# 1. CGGA-compatible candidate universe
# ============================================================

compatible_edges = edges[
    edges["gene1"].isin(usable_genes)
    & edges["gene2"].isin(usable_genes)
].copy()

print("\nOriginal candidate edges:", len(edges))
print("CGGA-compatible candidate edges:", len(compatible_edges))
print(
    "Candidate edge retention:",
    f"{100 * len(compatible_edges) / len(edges):.4f}%"
)

checkpoint(
    len(compatible_edges) == 89551,
    "compatible candidate universe reproduces 89,551 edges"
)

# ============================================================
# 2. Project ORIGINAL frozen 20% reference into CGGA space
#    DO NOT re-rank reference PCCs.
# ============================================================

compatible_ref = ref20[
    ref20["gene1"].isin(usable_genes)
    & ref20["gene2"].isin(usable_genes)
].copy()

K_EXTERNAL = len(compatible_ref)

external_density = (
    K_EXTERNAL / len(compatible_edges)
)

print("\nOriginal reference edges:", len(ref20))
print("Reference edges surviving CGGA availability:", K_EXTERNAL)
print("Reference edges lost:", len(ref20) - K_EXTERNAL)

print(
    "External matched-network density:",
    f"{100 * external_density:.4f}%"
)

print(
    "Difference from frozen 20% rule:",
    f"{100 * (external_density - 0.20):+.4f} percentage points"
)

# ============================================================
# 3. Anchor degree impact
# ============================================================

anchor_mask = (
    stage_b["Is_DEG_Anchor"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
)

anchors = (
    set(stage_b.loc[anchor_mask, "ensembl_id"])
    & (
        set(edges["gene1"])
        | set(edges["gene2"])
    )
)

checkpoint(len(anchors) == 3992, "3,992 frozen DEG anchors recovered")

usable_anchors = sorted(
    anchors & usable_genes
)

checkpoint(
    len(usable_anchors) == 3944,
    "3,944 DEG anchors are externally measurable"
)

def degree_from_edges(df, anchor_set):
    counts = {g: 0 for g in anchor_set}

    for g in df["gene1"]:
        if g in counts:
            counts[g] += 1

    for g in df["gene2"]:
        if g in counts:
            counts[g] += 1

    return counts

orig_degree = degree_from_edges(
    ref20,
    usable_anchors
)

ext_degree = degree_from_edges(
    compatible_ref,
    usable_anchors
)

degree_audit = pd.DataFrame({
    "ensembl_id": usable_anchors,
    "original_reference_degree": [
        orig_degree[g] for g in usable_anchors
    ],
    "external_reference_degree": [
        ext_degree[g] for g in usable_anchors
    ],
})

degree_audit["edges_lost"] = (
    degree_audit["original_reference_degree"]
    - degree_audit["external_reference_degree"]
)

degree_audit["degree_retention"] = np.where(
    degree_audit["original_reference_degree"] > 0,
    degree_audit["external_reference_degree"]
    / degree_audit["original_reference_degree"],
    np.nan
)

original_connected = (
    degree_audit["original_reference_degree"] > 0
)

newly_isolated = (
    (degree_audit["original_reference_degree"] > 0)
    & (degree_audit["external_reference_degree"] == 0)
)

print("\n" + "-" * 80)
print("REFERENCE-NEIGHBORHOOD PRESERVATION")
print("-" * 80)

print(
    "Usable anchors connected in original reference:",
    int(original_connected.sum())
)

print(
    "Newly isolated because of CGGA availability:",
    int(newly_isolated.sum())
)

connected_audit = degree_audit[
    original_connected
].copy()

print(
    "Median reference-degree retention:",
    f"{100 * connected_audit['degree_retention'].median():.2f}%"
)

print(
    "Anchors with 100% degree retention:",
    int((connected_audit["degree_retention"] == 1).sum()),
    "/",
    len(connected_audit)
)

print(
    "Anchors with >=95% degree retention:",
    int((connected_audit["degree_retention"] >= 0.95).sum()),
    "/",
    len(connected_audit)
)

print(
    "Anchors with <80% degree retention:",
    int((connected_audit["degree_retention"] < 0.80).sum())
)

# ============================================================
# 4. dmax audit
# ============================================================

original_dmax = int(
    max(orig_degree.values())
)

external_dmax = int(
    max(ext_degree.values())
)

print("\nOriginal reference dmax:", original_dmax)
print("CGGA-compatible reference dmax:", external_dmax)

# ============================================================
# 5. Save frozen external topology
# ============================================================

compatible_edges.to_csv(
    OUT_DIR / "CGGA_FROZEN_COMPATIBLE_EDGE_UNIVERSE.csv",
    index=False
)

compatible_ref.to_csv(
    OUT_DIR / "CGGA_FROZEN_COMPATIBLE_REFERENCE_NETWORK.csv",
    index=False
)

degree_audit.to_csv(
    OUT_DIR / "CGGA_REFERENCE_NEIGHBORHOOD_RETENTION_AUDIT.csv",
    index=False
)

summary = pd.DataFrame({
    "Metric": [
        "Original candidate edges",
        "CGGA-compatible candidate edges",
        "Candidate-edge retention %",
        "Original frozen reference edges",
        "CGGA-compatible reference edges",
        "External network density %",
        "Externally measurable anchors",
        "Newly isolated anchors",
        "Median reference-degree retention %",
        "Original dmax",
        "External dmax",
    ],
    "Value": [
        len(edges),
        len(compatible_edges),
        100 * len(compatible_edges) / len(edges),
        len(ref20),
        len(compatible_ref),
        100 * external_density,
        len(usable_anchors),
        int(newly_isolated.sum()),
        100 * connected_audit["degree_retention"].median(),
        original_dmax,
        external_dmax,
    ]
})

summary.to_csv(
    OUT_DIR / "CGGA_EXTERNAL_TOPOLOGY_FREEZE_SUMMARY.csv",
    index=False
)

print("\n" + "=" * 80)
print("STEP 2E COMPLETE — STOP HERE")
print("=" * 80)

print("\nExternal topology summary:")
print(summary.to_string(index=False))

print(
    "\nFrozen patient rule for CGGA will be:"
)
print(
    f"Each patient retains its own top {K_EXTERNAL:,} "
    f"|sPCC| edges from the {len(compatible_edges):,} "
    "compatible candidate interactions."
)

print("\nDO NOT compute sPCC yet.")

### STEP 3A — LOAD AND AUDIT CGGA 20 NON-GLIOMA CONTROLS

In [ ]:
# ============================================================
# STEP 3A — LOAD AND AUDIT CGGA 20 NON-GLIOMA CONTROLS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

print("\n" + "=" * 80)
print("STEP 3A — CGGA NON-GLIOMA CONTROL AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# Locate extracted control file automatically
# ------------------------------------------------------------

matches = [
    x for x in DOWNLOADS.rglob("*normal*20*Read*Counts*")
    if x.is_file()
    and not x.name.lower().endswith(".zip")
]

print("\nPossible extracted control files:")

for x in matches:
    print(
        x,
        "|",
        round(x.stat().st_size / (1024 ** 2), 2),
        "MB"
    )

checkpoint(
    len(matches) >= 1,
    "at least one extracted CGGA normal-20 read-count file found"
)

# Prefer exact expected filename if present
expected = [
    x for x in matches
    if x.name == "CGGA.normal_20.Read_Counts-genes.20230104.txt"
]

if len(expected) == 1:
    CONTROL_FILE = expected[0]
elif len(matches) == 1:
    CONTROL_FILE = matches[0]
else:
    raise RuntimeError(
        "More than one possible control file found. "
        "Please send the printed paths before proceeding."
    )

print("\nUsing control file:")
print(CONTROL_FILE)

# ------------------------------------------------------------
# Load controls
# ------------------------------------------------------------

controls = pd.read_csv(
    CONTROL_FILE,
    sep="\t",
    low_memory=False
)

print("\nRaw control matrix shape:", controls.shape)

print("\nFirst 10 columns:")
print(controls.columns[:10].tolist())

print("\nFirst 5 rows / first 6 columns:")
print(controls.iloc[:5, :6])

gene_col = controls.columns[0]

print("\nGene identifier column:", gene_col)

checkpoint(
    gene_col == "gene_name",
    "control matrix uses gene_name as first column"
)

control_sample_cols = [
    c for c in controls.columns
    if c != gene_col
]

print("Control samples:", len(control_sample_cols))

checkpoint(
    len(control_sample_cols) == 20,
    "exactly 20 CGGA non-glioma controls"
)

# ------------------------------------------------------------
# Gene-symbol sanity
# ------------------------------------------------------------

control_symbols = (
    controls["gene_name"]
    .astype(str)
    .str.strip()
)

print("\nControl gene rows:", len(control_symbols))
print("Unique control symbols:", control_symbols.nunique())
print("Duplicated symbols:", control_symbols.duplicated().sum())
print(
    "Missing/blank symbols:",
    (
        (control_symbols == "")
        | (control_symbols.str.lower() == "nan")
    ).sum()
)

checkpoint(
    control_symbols.duplicated().sum() == 0,
    "control gene symbols are unique"
)

# ------------------------------------------------------------
# Compare with mRNAseq_693 gene universe
# counts must already be loaded from Step 1
# ------------------------------------------------------------

TUMOR_FILE = (
    DOWNLOADS
    / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"
    / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"
)

tumor_gene_df = pd.read_csv(
    TUMOR_FILE,
    sep="\t",
    usecols=["gene_name"],
    low_memory=False
)

tumor_symbols = set(
    tumor_gene_df["gene_name"]
    .astype(str)
    .str.strip()
)

control_symbol_set = set(control_symbols)

print("\n" + "-" * 80)
print("CONTROL ↔ mRNAseq_693 GENE UNIVERSE")
print("-" * 80)

print("mRNAseq_693 symbols:", len(tumor_symbols))
print("Control symbols:", len(control_symbol_set))

print(
    "Shared symbols:",
    len(tumor_symbols & control_symbol_set)
)

print(
    "Tumor-only symbols:",
    len(tumor_symbols - control_symbol_set)
)

print(
    "Control-only symbols:",
    len(control_symbol_set - tumor_symbols)
)

# ------------------------------------------------------------
# Check frozen externally usable genes
# ------------------------------------------------------------

MAP_FILE = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
    / "CGGA_FINAL_FROZEN_GENE_MAPPING_AFTER_HGNC.csv"
)

gene_map = pd.read_csv(MAP_FILE)

needed_symbols = set(
    gene_map["cgga_symbol"]
    .astype(str)
    .str.strip()
)

available_in_controls = (
    needed_symbols & control_symbol_set
)

missing_from_controls = (
    needed_symbols - control_symbol_set
)

print("\n" + "-" * 80)
print("FROZEN NETWORK GENE COVERAGE IN CONTROLS")
print("-" * 80)

print("Externally mapped frozen genes:", len(needed_symbols))

print(
    "Present in CGGA controls:",
    len(available_in_controls)
)

print(
    "Missing from CGGA controls:",
    len(missing_from_controls)
)

print(
    "Control coverage:",
    f"{100 * len(available_in_controls) / len(needed_symbols):.2f}%"
)

if missing_from_controls:
    print("\nFirst 30 missing symbols:")
    print(sorted(missing_from_controls)[:30])

print("\n" + "=" * 80)
print("STEP 3A COMPLETE — STOP HERE")
print("=" * 80)

print("""
DO NOT normalize yet.
DO NOT transform CGGA tumors yet.
DO NOT compute sPCC/eNESH yet.
""")

### STEP 3B — CGGA TUMOR + CONTROL NORMALIZATION

In [ ]:
# ============================================================
# STEP 3B — CGGA TUMOR + CONTROL NORMALIZATION
# log2(CPM + 1), matched to original preprocessing convention
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

OUT_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"
OUT_DIR.mkdir(exist_ok=True)

TUMOR_FILE = (
    DOWNLOADS
    / "CGGA.mRNAseq_693.Read_Counts-genes.20220620"
    / "CGGA.mRNAseq_693.Read_Counts-genes.20220620.txt"
)

CONTROL_FILE = (
    DOWNLOADS
    / "CGGA.normal_20.Read_Counts-genes.20230104"
    / "CGGA.normal_20.Read_Counts-genes.20230104.txt"
)

CLINICAL_FILE = (
    DOWNLOADS
    / "CGGA.mRNAseq_693_clinical.20200506.txt"
    / "CGGA.mRNAseq_693_clinical.20200506.txt"
)

MAP_FILE = (
    OUT_DIR
    / "CGGA_FINAL_FROZEN_GENE_MAPPING_AFTER_HGNC.csv"
)

print("\n" + "=" * 80)
print("STEP 3B — NORMALIZE CGGA TUMORS + CONTROLS")
print("=" * 80)

# ============================================================
# 1. LOAD CLINICAL AND RECONSTRUCT THE FROZEN 104 PATIENTS
# ============================================================

clinical = pd.read_csv(
    CLINICAL_FILE,
    sep="\t",
    low_memory=False
)

clinical["Age_num"] = pd.to_numeric(
    clinical["Age"],
    errors="coerce"
)

clinical["OS_days"] = pd.to_numeric(
    clinical["OS"],
    errors="coerce"
)

clinical["OS_event"] = pd.to_numeric(
    clinical["Censor (alive=0; dead=1)"],
    errors="coerce"
)

eligible = clinical[
    (clinical["PRS_type"] == "Primary")
    & (clinical["Histology"] == "GBM")
    & (clinical["Grade"] == "WHO IV")
    & (clinical["IDH_mutation_status"] == "Wildtype")
    & (clinical["Age_num"] >= 18)
    & (clinical["OS_days"] > 0)
    & (clinical["OS_event"].isin([0, 1]))
].copy()

eligible_ids = eligible["CGGA_ID"].astype(str).tolist()

checkpoint(
    len(eligible_ids) == 104,
    "exactly 104 frozen CGGA validation patients recovered"
)

# ============================================================
# 2. LOAD ONLY THE 104 TUMORS + ALL 20 CONTROLS
# ============================================================

tumor_usecols = ["gene_name"] + eligible_ids

tumor_counts = pd.read_csv(
    TUMOR_FILE,
    sep="\t",
    usecols=tumor_usecols,
    low_memory=False
)

control_counts = pd.read_csv(
    CONTROL_FILE,
    sep="\t",
    low_memory=False
)

print("\nTumor count matrix:", tumor_counts.shape)
print("Control count matrix:", control_counts.shape)

checkpoint(
    tumor_counts.shape == (55523, 105),
    "tumor matrix = 55,523 genes × 104 patients"
)

checkpoint(
    control_counts.shape == (55523, 21),
    "control matrix = 55,523 genes × 20 controls"
)

# ============================================================
# 3. VERIFY IDENTICAL GENE SPACE / ORDER
# ============================================================

tumor_genes = tumor_counts["gene_name"].astype(str).str.strip()
control_genes = control_counts["gene_name"].astype(str).str.strip()

checkpoint(
    tumor_genes.equals(control_genes),
    "CGGA tumor and control gene rows are identical and in the same order"
)

checkpoint(
    tumor_genes.is_unique,
    "CGGA gene symbols are unique"
)

print("\nShared gene rows:", len(tumor_genes))

# ============================================================
# 4. VERIFY RAW COUNTS
# ============================================================

tumor_X = tumor_counts.drop(columns="gene_name")
control_X = control_counts.drop(columns="gene_name")

checkpoint(
    (tumor_X.to_numpy() >= 0).all(),
    "tumor counts are non-negative"
)

checkpoint(
    (control_X.to_numpy() >= 0).all(),
    "control counts are non-negative"
)

checkpoint(
    np.isfinite(tumor_X.to_numpy()).all(),
    "tumor counts are finite"
)

checkpoint(
    np.isfinite(control_X.to_numpy()).all(),
    "control counts are finite"
)

# ============================================================
# 5. MITOCHONDRIAL GENES EXCLUDED FROM CPM DENOMINATOR
#    Same 15-gene convention as original TCGA calibration
# ============================================================

MT_GENES = {
    "MT-RNR1",
    "MT-RNR2",
    "MT-ND1",
    "MT-ND2",
    "MT-CO1",
    "MT-CO2",
    "MT-ATP8",
    "MT-ATP6",
    "MT-CO3",
    "MT-ND3",
    "MT-ND4L",
    "MT-ND4",
    "MT-ND5",
    "MT-ND6",
    "MT-CYB",
}

present_mt = set(tumor_genes) & MT_GENES

print("\nCanonical mitochondrial genes present:", len(present_mt))
print(sorted(present_mt))

checkpoint(
    len(present_mt) == 15,
    "all 15 canonical mitochondrial genes are present"
)

non_mt_mask = ~tumor_genes.isin(MT_GENES)

# ============================================================
# 6. COMPUTE LIBRARY SIZES EXCLUDING THE 15 MT GENES
# ============================================================

tumor_libsize = (
    tumor_X.loc[non_mt_mask]
    .sum(axis=0)
    .astype(float)
)

control_libsize = (
    control_X.loc[non_mt_mask]
    .sum(axis=0)
    .astype(float)
)

checkpoint(
    (tumor_libsize > 0).all(),
    "all tumor library sizes are positive"
)

checkpoint(
    (control_libsize > 0).all(),
    "all control library sizes are positive"
)

print("\nTumor library sizes:")
print(tumor_libsize.describe())

print("\nControl library sizes:")
print(control_libsize.describe())

# ============================================================
# 7. CPM -> log2(CPM + 1)
# ============================================================

tumor_cpm = tumor_X.divide(
    tumor_libsize,
    axis=1
) * 1e6

control_cpm = control_X.divide(
    control_libsize,
    axis=1
) * 1e6

tumor_logcpm = np.log2(
    tumor_cpm + 1.0
)

control_logcpm = np.log2(
    control_cpm + 1.0
)

tumor_logcpm.index = tumor_genes
control_logcpm.index = control_genes

checkpoint(
    np.isfinite(tumor_logcpm.to_numpy()).all(),
    "normalized tumor matrix is finite"
)

checkpoint(
    np.isfinite(control_logcpm.to_numpy()).all(),
    "normalized control matrix is finite"
)

# ============================================================
# 8. MAP TO FROZEN EXTERNAL NETWORK GENE SPACE
# ============================================================

gene_map = pd.read_csv(
    MAP_FILE,
    low_memory=False
)

gene_map["ensembl_id"] = (
    gene_map["ensembl_id"]
    .astype(str)
    .str.split(".").str[0]
    .str.strip()
)

gene_map["cgga_symbol"] = (
    gene_map["cgga_symbol"]
    .astype(str)
    .str.strip()
)

checkpoint(
    len(gene_map) == 12888,
    "final external mapping contains 12,888 genes"
)

checkpoint(
    gene_map["ensembl_id"].is_unique,
    "mapped Ensembl IDs are unique"
)

checkpoint(
    gene_map["cgga_symbol"].is_unique,
    "mapped CGGA symbols are unique"
)

mapped_symbols = gene_map["cgga_symbol"].tolist()

checkpoint(
    set(mapped_symbols).issubset(set(tumor_logcpm.index)),
    "all frozen mapped genes are present in tumors"
)

checkpoint(
    set(mapped_symbols).issubset(set(control_logcpm.index)),
    "all frozen mapped genes are present in controls"
)

# reorder by mapping table
tumor_network = (
    tumor_logcpm
    .loc[mapped_symbols]
    .copy()
)

control_network = (
    control_logcpm
    .loc[mapped_symbols]
    .copy()
)

# replace CGGA symbols with frozen Ensembl IDs
symbol_to_ens = dict(
    zip(
        gene_map["cgga_symbol"],
        gene_map["ensembl_id"]
    )
)

tumor_network.index = [
    symbol_to_ens[g]
    for g in tumor_network.index
]

control_network.index = [
    symbol_to_ens[g]
    for g in control_network.index
]

checkpoint(
    tumor_network.index.equals(control_network.index),
    "tumor/control frozen gene order is identical"
)

print("\nFrozen tumor expression matrix:",
      tumor_network.shape)

print("Frozen control expression matrix:",
      control_network.shape)

# ============================================================
# 9. SAVE NORMALIZED MATRICES
# ============================================================

tumor_network.to_csv(
    OUT_DIR / "CGGA_104_LOG2CPM_FROZEN_GENES.csv"
)

control_network.to_csv(
    OUT_DIR / "CGGA_NORMAL20_LOG2CPM_FROZEN_GENES.csv"
)

pd.DataFrame({
    "sample": tumor_libsize.index,
    "nonMT_library_size": tumor_libsize.values,
    "cohort": "CGGA_GBM"
}).to_csv(
    OUT_DIR / "CGGA_104_LIBRARY_SIZES.csv",
    index=False
)

pd.DataFrame({
    "sample": control_libsize.index,
    "nonMT_library_size": control_libsize.values,
    "cohort": "CGGA_normal20"
}).to_csv(
    OUT_DIR / "CGGA_NORMAL20_LIBRARY_SIZES.csv",
    index=False
)

print("\n" + "=" * 80)
print("STEP 3B COMPLETE — STOP HERE")
print("=" * 80)

print("""
DO NOT calibrate yet.
DO NOT compute sPCC yet.
DO NOT compute eNESH yet.
""")

### STEP 3C — CGGA NORMAL-20 vs GTEx-334

In [ ]:
# ============================================================
# STEP 3C — CGGA NORMAL-20 vs GTEx-334
# PRE-HARMONIZATION AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

OUT_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"
OUT_DIR.mkdir(exist_ok=True)

GTEX_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
)

CGGA_CONTROL_FILE = (
    OUT_DIR
    / "CGGA_NORMAL20_LOG2CPM_FROZEN_GENES.csv"
)

CGGA_TUMOR_FILE = (
    OUT_DIR
    / "CGGA_104_LOG2CPM_FROZEN_GENES.csv"
)

print("\n" + "=" * 80)
print("STEP 3C — CGGA NORMAL-20 vs GTEx-334 PRE-HARMONIZATION AUDIT")
print("=" * 80)

# ============================================================
# 1. LOAD MATRICES
# ============================================================

gtex = pd.read_csv(
    GTEX_FILE,
    index_col=0,
    low_memory=False
)

cgga_ctrl = pd.read_csv(
    CGGA_CONTROL_FILE,
    index_col=0,
    low_memory=False
)

cgga_tumor = pd.read_csv(
    CGGA_TUMOR_FILE,
    index_col=0,
    low_memory=False
)

# clean Ensembl version suffix if present
for df in [gtex, cgga_ctrl, cgga_tumor]:
    df.index = (
        df.index.astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

print("\nGTEx matrix:", gtex.shape)
print("CGGA controls:", cgga_ctrl.shape)
print("CGGA tumors:", cgga_tumor.shape)

checkpoint(
    gtex.shape == (13101, 334),
    "GTEx processed reference = 13,101 genes × 334 samples"
)

checkpoint(
    cgga_ctrl.shape == (12888, 20),
    "CGGA controls = 12,888 genes × 20 samples"
)

checkpoint(
    cgga_tumor.shape == (12888, 104),
    "CGGA tumors = 12,888 genes × 104 patients"
)

checkpoint(gtex.index.is_unique, "GTEx gene IDs are unique")
checkpoint(cgga_ctrl.index.is_unique, "CGGA control gene IDs are unique")

# ============================================================
# 2. EXACT GENE-SPACE ALIGNMENT
# ============================================================

missing_in_gtex = [
    g for g in cgga_ctrl.index
    if g not in set(gtex.index)
]

print("\nCGGA frozen genes missing from GTEx:",
      len(missing_in_gtex))

if len(missing_in_gtex) > 0:
    print("First missing genes:", missing_in_gtex[:20])

common = [
    g for g in cgga_ctrl.index
    if g in set(gtex.index)
]

print("Common genes:", len(common))

checkpoint(
    len(common) == 12888,
    "all 12,888 CGGA-compatible frozen genes are present in GTEx"
)

G = gtex.loc[common].astype(float)
C = cgga_ctrl.loc[common].astype(float)

checkpoint(
    G.index.equals(C.index),
    "GTEx and CGGA-control gene order is identical"
)

# ============================================================
# 3. VALUE-SCALE AUDIT
# ============================================================

def describe_matrix(name, X):
    vals = X.to_numpy(dtype=float)

    print(f"\n{name} value summary")
    print("-" * 50)
    print(f"min:       {np.min(vals):.4f}")
    print(f"median:    {np.median(vals):.4f}")
    print(f"mean:      {np.mean(vals):.4f}")
    print(f"p95:       {np.percentile(vals,95):.4f}")
    print(f"p99:       {np.percentile(vals,99):.4f}")
    print(f"max:       {np.max(vals):.4f}")
    print(f"zero %:    {100*np.mean(vals == 0):.2f}")

describe_matrix("GTEx-334", G)
describe_matrix("CGGA normal-20", C)

checkpoint(
    np.isfinite(G.to_numpy()).all(),
    "GTEx values are finite"
)

checkpoint(
    np.isfinite(C.to_numpy()).all(),
    "CGGA-control values are finite"
)

# ============================================================
# 4. GENE-WISE NORMAL REFERENCE PROFILES
# ============================================================

gtex_mean = G.mean(axis=1)
cgga_mean = C.mean(axis=1)

gtex_median = G.median(axis=1)
cgga_median = C.median(axis=1)

gtex_sd = G.std(axis=1, ddof=1)
cgga_sd = C.std(axis=1, ddof=1)

# ============================================================
# 5. GLOBAL AGREEMENT
# ============================================================

mean_pearson = pearsonr(
    gtex_mean.values,
    cgga_mean.values
).statistic

mean_spearman = spearmanr(
    gtex_mean.values,
    cgga_mean.values
).statistic

rmse = np.sqrt(
    np.mean(
        (cgga_mean.values - gtex_mean.values) ** 2
    )
)

mae = np.mean(
    np.abs(
        cgga_mean.values - gtex_mean.values
    )
)

median_shift = np.median(
    cgga_mean.values - gtex_mean.values
)

print("\n" + "=" * 80)
print("GENE-WISE NORMAL PROFILE AGREEMENT")
print("=" * 80)

print(f"Pearson r:              {mean_pearson:.4f}")
print(f"Spearman rho:           {mean_spearman:.4f}")
print(f"RMSE:                   {rmse:.4f}")
print(f"MAE:                    {mae:.4f}")
print(f"Median CGGA-GTEx shift: {median_shift:.4f}")

# ============================================================
# 6. SAMPLE-LEVEL AGREEMENT
# ============================================================

sample_rows = []

for sample in C.columns:

    x = C[sample].values
    y = gtex_mean.values

    sample_rows.append({
        "sample": sample,

        "pearson_to_GTEx_mean":
            pearsonr(x, y).statistic,

        "spearman_to_GTEx_mean":
            spearmanr(x, y).statistic,

        "rmse_to_GTEx_mean":
            np.sqrt(np.mean((x - y) ** 2))
    })

sample_audit = pd.DataFrame(sample_rows)

print("\n" + "=" * 80)
print("CGGA CONTROL → GTEx MEAN")
print("=" * 80)

print("\nPearson:")
print(sample_audit[
    "pearson_to_GTEx_mean"
].describe())

print("\nSpearman:")
print(sample_audit[
    "spearman_to_GTEx_mean"
].describe())

print("\nRMSE:")
print(sample_audit[
    "rmse_to_GTEx_mean"
].describe())

# ============================================================
# 7. GENE-WISE LOCATION DIFFERENCE
# ============================================================

gene_audit = pd.DataFrame({
    "ensembl_id": common,
    "GTEx_mean": gtex_mean.values,
    "CGGA_mean": cgga_mean.values,
    "CGGA_minus_GTEx":
        cgga_mean.values - gtex_mean.values,
    "abs_mean_difference":
        np.abs(cgga_mean.values - gtex_mean.values),
    "GTEx_sd": gtex_sd.values,
    "CGGA_sd": cgga_sd.values
})

valid_sd = (
    (gene_audit["GTEx_sd"] > 1e-8)
    & (gene_audit["CGGA_sd"] > 1e-8)
)

gene_audit["sd_ratio_CGGA_over_GTEx"] = np.nan

gene_audit.loc[
    valid_sd,
    "sd_ratio_CGGA_over_GTEx"
] = (
    gene_audit.loc[valid_sd, "CGGA_sd"]
    /
    gene_audit.loc[valid_sd, "GTEx_sd"]
)

print("\n" + "=" * 80)
print("GENE-WISE LOCATION / SCALE DIFFERENCES")
print("=" * 80)

print(
    "Genes with non-zero SD in both:",
    valid_sd.sum(),
    "/",
    len(gene_audit)
)

print("\nAbsolute mean difference:")
print(
    gene_audit[
        "abs_mean_difference"
    ].describe(
        percentiles=[.25,.5,.75,.9,.95,.99]
    )
)

print("\nSD ratio CGGA / GTEx:")
print(
    gene_audit.loc[
        valid_sd,
        "sd_ratio_CGGA_over_GTEx"
    ].describe(
        percentiles=[.25,.5,.75,.9,.95,.99]
    )
)

# ============================================================
# 8. SOME USEFUL THRESHOLD COUNTS
# ============================================================

for threshold in [0.5, 1.0, 1.5, 2.0]:

    n = (
        gene_audit["abs_mean_difference"]
        > threshold
    ).sum()

    print(
        f"|CGGA-GTEx mean| > {threshold}: "
        f"{n} genes "
        f"({100*n/len(gene_audit):.2f}%)"
    )

# ============================================================
# 9. SAVE EVERYTHING
# ============================================================

gene_audit.to_csv(
    OUT_DIR
    / "CGGA_NORMAL20_vs_GTEX334_PREHARMONIZATION_GENE_AUDIT.csv",
    index=False
)

sample_audit.to_csv(
    OUT_DIR
    / "CGGA_NORMAL20_vs_GTEX334_PREHARMONIZATION_SAMPLE_AUDIT.csv",
    index=False
)

summary = pd.DataFrame({
    "Metric": [
        "Common genes",
        "Mean-profile Pearson",
        "Mean-profile Spearman",
        "Mean-profile RMSE",
        "Mean-profile MAE",
        "Median CGGA-GTEx shift",
        "Median control Pearson",
        "Median control Spearman",
        "Median control RMSE"
    ],

    "Value": [
        len(common),
        mean_pearson,
        mean_spearman,
        rmse,
        mae,
        median_shift,
        sample_audit[
            "pearson_to_GTEx_mean"
        ].median(),
        sample_audit[
            "spearman_to_GTEx_mean"
        ].median(),
        sample_audit[
            "rmse_to_GTEx_mean"
        ].median()
    ]
})

summary.to_csv(
    OUT_DIR
    / "CGGA_NORMAL20_vs_GTEX334_PREHARMONIZATION_SUMMARY.csv",
    index=False
)

print("\n" + "=" * 80)
print("STEP 3C COMPLETE — STOP HERE")
print("=" * 80)

print("\nFINAL SUMMARY")
print(summary.to_string(index=False))

print("""
DO NOT harmonize yet.
DO NOT compute sPCC.
DO NOT compute eNESH.
""")

### STEP 3D — LEAVE-ONE-CONTROL-OUT VALIDATION

In [ ]:
# ============================================================
# STEP 3D — LEAVE-ONE-CONTROL-OUT VALIDATION
# Conservative gene-wise LOCATION harmonization
# No survival outcomes used
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

OUT_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"

GTEX_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
)

CONTROL_FILE = (
    OUT_DIR
    / "CGGA_NORMAL20_LOG2CPM_FROZEN_GENES.csv"
)

print("\n" + "=" * 80)
print("STEP 3D — LOOCV VALIDATION OF CONTROL-ANCHORED LOCATION HARMONIZATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load
# ------------------------------------------------------------

gtex = pd.read_csv(
    GTEX_FILE,
    index_col=0,
    low_memory=False
)

controls = pd.read_csv(
    CONTROL_FILE,
    index_col=0,
    low_memory=False
)

for df in [gtex, controls]:
    df.index = (
        df.index.astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

genes = controls.index.tolist()

checkpoint(
    set(genes).issubset(set(gtex.index)),
    "all CGGA-compatible genes exist in GTEx"
)

G = gtex.loc[genes].astype(float)
C = controls.loc[genes].astype(float)

checkpoint(
    G.shape == (12888, 334),
    "GTEx aligned matrix = 12,888 × 334"
)

checkpoint(
    C.shape == (12888, 20),
    "CGGA controls = 12,888 × 20"
)

# Fixed GTEx reference mean
mu_gtex = G.mean(axis=1)

# ------------------------------------------------------------
# 2. Leave-one-control-out validation
# ------------------------------------------------------------

results = []

for held_out in C.columns:

    train_cols = [
        c for c in C.columns
        if c != held_out
    ]

    C_train = C[train_cols]

    # CGGA normal mean estimated WITHOUT the held-out sample
    mu_cgga_train = C_train.mean(axis=1)

    # Gene-wise location correction
    # target GTEx mean - estimated CGGA control mean
    delta = mu_gtex - mu_cgga_train

    raw = C[held_out]

    corrected = raw + delta

    # ----------------------------------------
    # Before
    # ----------------------------------------
    raw_pearson = pearsonr(
        raw.values,
        mu_gtex.values
    ).statistic

    raw_spearman = spearmanr(
        raw.values,
        mu_gtex.values
    ).statistic

    raw_rmse = np.sqrt(
        np.mean(
            (raw.values - mu_gtex.values) ** 2
        )
    )

    raw_mae = np.mean(
        np.abs(
            raw.values - mu_gtex.values
        )
    )

    # ----------------------------------------
    # After location correction
    # ----------------------------------------
    corr_pearson = pearsonr(
        corrected.values,
        mu_gtex.values
    ).statistic

    corr_spearman = spearmanr(
        corrected.values,
        mu_gtex.values
    ).statistic

    corr_rmse = np.sqrt(
        np.mean(
            (corrected.values - mu_gtex.values) ** 2
        )
    )

    corr_mae = np.mean(
        np.abs(
            corrected.values - mu_gtex.values
        )
    )

    results.append({
        "sample": held_out,

        "raw_pearson": raw_pearson,
        "corrected_pearson": corr_pearson,

        "raw_spearman": raw_spearman,
        "corrected_spearman": corr_spearman,

        "raw_rmse": raw_rmse,
        "corrected_rmse": corr_rmse,

        "raw_mae": raw_mae,
        "corrected_mae": corr_mae,

        "rmse_improvement":
            raw_rmse - corr_rmse,

        "mae_improvement":
            raw_mae - corr_mae
    })

results = pd.DataFrame(results)

# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LOOCV RESULTS")
print("=" * 80)

for metric in [
    "pearson",
    "spearman",
    "rmse",
    "mae"
]:
    print(f"\n{metric.upper()}")

    print(
        "Before median:",
        results[f"raw_{metric}"].median()
    )

    print(
        "After median:",
        results[f"corrected_{metric}"].median()
    )

print("\nControls with improved RMSE:",
      (results["corrected_rmse"]
       < results["raw_rmse"]).sum(),
      "/ 20")

print("Controls with improved MAE:",
      (results["corrected_mae"]
       < results["raw_mae"]).sum(),
      "/ 20")

print("Controls with improved Pearson:",
      (results["corrected_pearson"]
       > results["raw_pearson"]).sum(),
      "/ 20")

print("Controls with improved Spearman:",
      (results["corrected_spearman"]
       > results["raw_spearman"]).sum(),
      "/ 20")

print("\nRMSE improvement:")
print(
    results["rmse_improvement"].describe()
)

print("\nMAE improvement:")
print(
    results["mae_improvement"].describe()
)

# ------------------------------------------------------------
# 4. Save audit
# ------------------------------------------------------------

results.to_csv(
    OUT_DIR
    / "CGGA_NORMAL20_LOCATION_HARMONIZATION_LOOCV.csv",
    index=False
)

summary = pd.DataFrame({
    "Metric": [
        "Median Pearson before",
        "Median Pearson after",
        "Median Spearman before",
        "Median Spearman after",
        "Median RMSE before",
        "Median RMSE after",
        "Median MAE before",
        "Median MAE after",
        "RMSE improved controls",
        "MAE improved controls",
        "Pearson improved controls",
        "Spearman improved controls"
    ],

    "Value": [
        results["raw_pearson"].median(),
        results["corrected_pearson"].median(),

        results["raw_spearman"].median(),
        results["corrected_spearman"].median(),

        results["raw_rmse"].median(),
        results["corrected_rmse"].median(),

        results["raw_mae"].median(),
        results["corrected_mae"].median(),

        (
            results["corrected_rmse"]
            < results["raw_rmse"]
        ).sum(),

        (
            results["corrected_mae"]
            < results["raw_mae"]
        ).sum(),

        (
            results["corrected_pearson"]
            > results["raw_pearson"]
        ).sum(),

        (
            results["corrected_spearman"]
            > results["raw_spearman"]
        ).sum()
    ]
})

summary.to_csv(
    OUT_DIR
    / "CGGA_NORMAL20_LOCATION_HARMONIZATION_LOOCV_SUMMARY.csv",
    index=False
)

print("\n" + "=" * 80)
print("FINAL LOOCV SUMMARY")
print("=" * 80)

print(summary.to_string(index=False))

print("""
STOP HERE.

Do NOT fit the final transformation yet.
Do NOT transform the 104 tumors yet.
Do NOT compute sPCC/eNESH yet.
""")

### STEP 3E — FIT FINAL LOCATION-ONLY HARMONIZATION

In [ ]:
# ============================================================
# STEP 3E — FIT FINAL LOCATION-ONLY HARMONIZATION
# using ALL 20 CGGA non-glioma controls
# and apply the SAME frozen transformation to 104 tumors
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

OUT_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"
OUT_DIR.mkdir(exist_ok=True)

GTEX_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
)

CONTROL_FILE = (
    OUT_DIR
    / "CGGA_NORMAL20_LOG2CPM_FROZEN_GENES.csv"
)

TUMOR_FILE = (
    OUT_DIR
    / "CGGA_104_LOG2CPM_FROZEN_GENES.csv"
)

print("\n" + "=" * 80)
print("STEP 3E — FINAL CGGA → GTEx LOCATION HARMONIZATION")
print("=" * 80)

# ============================================================
# 1. LOAD
# ============================================================

gtex = pd.read_csv(
    GTEX_FILE,
    index_col=0,
    low_memory=False
)

controls = pd.read_csv(
    CONTROL_FILE,
    index_col=0,
    low_memory=False
)

tumors = pd.read_csv(
    TUMOR_FILE,
    index_col=0,
    low_memory=False
)

for df in [gtex, controls, tumors]:
    df.index = (
        df.index.astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

genes = controls.index.tolist()

checkpoint(
    tumors.index.equals(controls.index),
    "CGGA tumor and control frozen gene order is identical"
)

checkpoint(
    set(genes).issubset(set(gtex.index)),
    "all 12,888 frozen genes exist in GTEx"
)

G = gtex.loc[genes].astype(float)
C = controls.loc[genes].astype(float)
T = tumors.loc[genes].astype(float)

checkpoint(
    G.shape == (12888, 334),
    "GTEx aligned matrix = 12,888 × 334"
)

checkpoint(
    C.shape == (12888, 20),
    "CGGA controls = 12,888 × 20"
)

checkpoint(
    T.shape == (12888, 104),
    "CGGA tumors = 12,888 × 104"
)

# ============================================================
# 2. FIT FINAL GENE-WISE LOCATION SHIFT
#    NO survival data are used
# ============================================================

mu_gtex = G.mean(axis=1)
mu_cgga_control = C.mean(axis=1)

delta = (
    mu_gtex
    - mu_cgga_control
)

print("\nLocation-shift summary (GTEx mean - CGGA-control mean):")
print(
    delta.describe(
        percentiles=[.01, .05, .25, .5, .75, .95, .99]
    )
)

# ============================================================
# 3. APPLY SAME FROZEN SHIFT
# ============================================================

C_harm = C.add(delta, axis=0)
T_harm = T.add(delta, axis=0)

checkpoint(
    np.isfinite(C_harm.to_numpy()).all(),
    "harmonized controls are finite"
)

checkpoint(
    np.isfinite(T_harm.to_numpy()).all(),
    "harmonized tumors are finite"
)

# IMPORTANT:
# Do NOT clip negative values.
# These are harmonized transformed values,
# not raw counts or literal CPM values.

# ============================================================
# 4. AUDIT FINAL CONTROLS
# ============================================================

gtex_mean = G.mean(axis=1)

sample_rows = []

for sample in C.columns:

    raw = C[sample]
    harm = C_harm[sample]

    sample_rows.append({
        "sample": sample,

        "raw_pearson":
            pearsonr(
                raw.values,
                gtex_mean.values
            ).statistic,

        "harm_pearson":
            pearsonr(
                harm.values,
                gtex_mean.values
            ).statistic,

        "raw_spearman":
            spearmanr(
                raw.values,
                gtex_mean.values
            ).statistic,

        "harm_spearman":
            spearmanr(
                harm.values,
                gtex_mean.values
            ).statistic,

        "raw_rmse":
            np.sqrt(
                np.mean(
                    (raw.values - gtex_mean.values) ** 2
                )
            ),

        "harm_rmse":
            np.sqrt(
                np.mean(
                    (harm.values - gtex_mean.values) ** 2
                )
            )
    })

sample_audit = pd.DataFrame(sample_rows)

print("\n" + "=" * 80)
print("FINAL CONTROL ALIGNMENT — AUDIT ONLY")
print("=" * 80)

print("\nPearson before:")
print(sample_audit["raw_pearson"].describe())

print("\nPearson after:")
print(sample_audit["harm_pearson"].describe())

print("\nSpearman before:")
print(sample_audit["raw_spearman"].describe())

print("\nSpearman after:")
print(sample_audit["harm_spearman"].describe())

print("\nRMSE before:")
print(sample_audit["raw_rmse"].describe())

print("\nRMSE after:")
print(sample_audit["harm_rmse"].describe())

# ============================================================
# 5. AUDIT VALUE RANGES
# ============================================================

def matrix_summary(name, X):
    vals = X.to_numpy(dtype=float)

    print(f"\n{name}")
    print("-" * 50)
    print(f"min:          {np.min(vals):.4f}")
    print(f"median:       {np.median(vals):.4f}")
    print(f"mean:         {np.mean(vals):.4f}")
    print(f"p95:          {np.percentile(vals,95):.4f}")
    print(f"p99:          {np.percentile(vals,99):.4f}")
    print(f"max:          {np.max(vals):.4f}")
    print(
        f"negative %:   "
        f"{100*np.mean(vals < 0):.4f}"
    )

matrix_summary(
    "CGGA controls BEFORE harmonization",
    C
)

matrix_summary(
    "CGGA controls AFTER harmonization",
    C_harm
)

matrix_summary(
    "CGGA tumors BEFORE harmonization",
    T
)

matrix_summary(
    "CGGA tumors AFTER harmonization",
    T_harm
)

# ============================================================
# 6. VERIFY FINAL REFERENCE MEAN
# ============================================================

final_control_mean = C_harm.mean(axis=1)

mean_rmse = np.sqrt(
    np.mean(
        (final_control_mean.values
         - mu_gtex.values) ** 2
    )
)

print(
    "\nRMSE between FINAL harmonized CGGA-control "
    "mean and GTEx mean:",
    mean_rmse
)

# This should be ~ numerical zero by construction.
checkpoint(
    mean_rmse < 1e-10,
    "final CGGA-control mean matches GTEx mean by construction"
)

# ============================================================
# 7. SAVE FROZEN HARMONIZATION PARAMETERS
# ============================================================

harm_params = pd.DataFrame({
    "ensembl_id": genes,
    "GTEx_mean": mu_gtex.values,
    "CGGA_normal20_mean": mu_cgga_control.values,
    "location_shift_delta":
        delta.values
})

harm_params.to_csv(
    OUT_DIR
    / "CGGA_TO_GTEX_FINAL_LOCATION_HARMONIZATION_PARAMETERS.csv",
    index=False
)

C_harm.to_csv(
    OUT_DIR
    / "CGGA_NORMAL20_HARMONIZED_TO_GTEX.csv"
)

T_harm.to_csv(
    OUT_DIR
    / "CGGA_104_HARMONIZED_TO_GTEX.csv"
)

sample_audit.to_csv(
    OUT_DIR
    / "CGGA_NORMAL20_FINAL_HARMONIZATION_AUDIT.csv",
    index=False
)

# ============================================================
# 8. SAVE FREEZE NOTE
# ============================================================

freeze_summary = pd.DataFrame({
    "Item": [
        "Harmonization method",
        "Reference controls",
        "External controls",
        "External tumors",
        "Genes",
        "Survival information used",
        "LOOCV RMSE improved controls",
        "LOOCV Pearson improved controls"
    ],

    "Value": [
        "gene-wise location-only additive correction",
        "GTEx cortex n=334",
        "CGGA non-glioma n=20",
        "CGGA eligible IDH-WT GBM n=104",
        "12888",
        "No",
        "20/20",
        "20/20"
    ]
})

freeze_summary.to_csv(
    OUT_DIR
    / "CGGA_EXTERNAL_HARMONIZATION_FREEZE_SUMMARY.csv",
    index=False
)

print("\n" + "=" * 80)
print("STEP 3E COMPLETE")
print("=" * 80)

print("""
FINAL LOCATION-ONLY TRANSFORMATION HAS BEEN FIT.

DO NOT modify the transformation after this point.
DO NOT use CGGA survival outcomes for further harmonization decisions.

STOP BEFORE sPCC/eNESH.
""")

### STEP 4A — CGGA PATIENT-SPECIFIC sPCC NETWORKS

In [ ]:
# ============================================================
# STEP 4A — CGGA PATIENT-SPECIFIC sPCC NETWORKS
#
# Frozen rules:
#   GTEx reference controls        = 334
#   compatible candidate edges    = 89,551
#   projected reference edges     = 17,858
#   each patient retains          = 17,858 highest |sPCC|
#
# IMPORTANT:
#   - Reference network is NOT re-ranked.
#   - No survival outcomes are used.
#   - No CGGA-dependent tuning is performed.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# helper
# ------------------------------------------------------------

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)


# ============================================================
# 0. PATHS / CONSTANTS
# ============================================================

DOWNLOADS = BASE

OUT_DIR = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
)

GTEX_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
)

TUMOR_FILE = (
    OUT_DIR
    / "CGGA_104_HARMONIZED_TO_GTEX.csv"
)

EDGE_FILE = (
    OUT_DIR
    / "CGGA_FROZEN_COMPATIBLE_EDGE_UNIVERSE.csv"
)

REF_FILE = (
    OUT_DIR
    / "CGGA_FROZEN_COMPATIBLE_REFERENCE_NETWORK.csv"
)

N_REF = 334
N_KEEP = 17858

print("\n" + "=" * 80)
print("STEP 4A — CGGA PATIENT-SPECIFIC sPCC NETWORK RECONSTRUCTION")
print("=" * 80)


# ============================================================
# 1. LOAD
# ============================================================

gtex = pd.read_csv(
    GTEX_FILE,
    index_col=0,
    low_memory=False
)

tumors = pd.read_csv(
    TUMOR_FILE,
    index_col=0,
    low_memory=False
)

edges = pd.read_csv(
    EDGE_FILE,
    low_memory=False
)

ref = pd.read_csv(
    REF_FILE,
    low_memory=False
)


# clean Ensembl IDs
for df in [gtex, tumors]:
    df.index = (
        df.index.astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

for col in ["gene1", "gene2"]:
    edges[col] = (
        edges[col].astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

    ref[col] = (
        ref[col].astype(str)
        .str.split(".").str[0]
        .str.strip()
    )


print("\nGTEx:", gtex.shape)
print("CGGA tumors:", tumors.shape)
print("Candidate edges:", edges.shape)
print("Projected reference:", ref.shape)


checkpoint(
    gtex.shape[1] == 334,
    "GTEx reference contains exactly 334 samples"
)

checkpoint(
    tumors.shape == (12888, 104),
    "harmonized CGGA tumor matrix = 12,888 × 104"
)

checkpoint(
    len(edges) == 89551,
    "compatible candidate edge universe = 89,551"
)

checkpoint(
    len(ref) == 17858,
    "projected frozen reference network = 17,858 edges"
)

checkpoint(
    edges["edge_key"].is_unique,
    "candidate edge keys are unique"
)


# ============================================================
# 2. ALIGN GTEx TO EXACT EXTERNAL GENE SPACE
# ============================================================

genes = tumors.index.tolist()

checkpoint(
    set(genes).issubset(set(gtex.index)),
    "all harmonized CGGA genes exist in GTEx"
)

G = (
    gtex
    .loc[genes]
    .astype(np.float64)
)

T = tumors.astype(np.float64)

checkpoint(
    G.index.equals(T.index),
    "GTEx and CGGA gene order is identical"
)

candidate_genes = (
    set(edges["gene1"])
    | set(edges["gene2"])
)

checkpoint(
    candidate_genes.issubset(set(genes)),
    "every candidate-edge endpoint exists in harmonized expression"
)


# ============================================================
# 3. BUILD GENE -> MATRIX POSITION LOOKUP
# ============================================================

gene_to_pos = {
    gene: i
    for i, gene in enumerate(genes)
}

edge_i = np.array(
    [gene_to_pos[g] for g in edges["gene1"]],
    dtype=np.int32
)

edge_j = np.array(
    [gene_to_pos[g] for g in edges["gene2"]],
    dtype=np.int32
)

checkpoint(
    len(edge_i) == 89551,
    "all candidate edge endpoints mapped"
)


# ============================================================
# 4. PRECOMPUTE GTEx REFERENCE STATISTICS
# ============================================================

G_arr = G.to_numpy(dtype=np.float64)

mu = G_arr.mean(axis=1)

G_centered = (
    G_arr
    - mu[:, None]
)

SS = np.einsum(
    "ij,ij->i",
    G_centered,
    G_centered
)

checkpoint(
    np.all(SS[edge_i] > 1e-12),
    "all gene1 endpoints have non-zero GTEx variance"
)

checkpoint(
    np.all(SS[edge_j] > 1e-12),
    "all gene2 endpoints have non-zero GTEx variance"
)


# ============================================================
# 5. FROZEN REFERENCE PCC
# ============================================================

r_ref = (
    edges["reference_PCC"]
    .to_numpy(dtype=np.float64)
)

ref_sign_candidate = (
    edges["reference_sign"]
    .to_numpy(dtype=np.int8)
)

checkpoint(
    np.isfinite(r_ref).all(),
    "all frozen reference PCC values are finite"
)

checkpoint(
    np.all(np.abs(r_ref) <= 1.0 + 1e-12),
    "all frozen reference PCC values lie in [-1,1]"
)

checkpoint(
    np.array_equal(
        np.sign(r_ref).astype(np.int8),
        ref_sign_candidate
    ),
    "reference signs match reference PCC values"
)


# ============================================================
# 6. VERIFY REFERENCE PCC DIRECTLY FROM GTEx
#
# Do this over ALL 89,551 edges in manageable batches.
# ============================================================

den_ref = np.sqrt(
    SS[edge_i]
    * SS[edge_j]
)

max_abs_diff = 0.0
mean_abs_diff_sum = 0.0
n_checked = 0

BATCH = 5000

for start in range(0, len(edges), BATCH):

    stop = min(
        start + BATCH,
        len(edges)
    )

    ii = edge_i[start:stop]
    jj = edge_j[start:stop]

    cross = np.einsum(
        "ij,ij->i",
        G_centered[ii],
        G_centered[jj]
    )

    direct_r = (
        cross
        / den_ref[start:stop]
    )

    diff = np.abs(
        direct_r
        - r_ref[start:stop]
    )

    max_abs_diff = max(
        max_abs_diff,
        float(diff.max())
    )

    mean_abs_diff_sum += float(
        diff.sum()
    )

    n_checked += len(diff)


mean_abs_diff = (
    mean_abs_diff_sum
    / n_checked
)

print("\nReference PCC reconstruction audit:")
print("Max absolute difference:", max_abs_diff)
print("Mean absolute difference:", mean_abs_diff)

checkpoint(
    max_abs_diff < 1e-8,
    "frozen reference PCC matches direct GTEx recomputation"
)


# ============================================================
# 7. RECONSTRUCT EXACT PROJECTED REFERENCE EDGE SET
#
# IMPORTANT:
# This is ONLY to identify the existing frozen reference edges.
# We DO NOT rank reference PCC again.
# ============================================================

g1 = edges["gene1"].to_numpy(dtype=str)
g2 = edges["gene2"].to_numpy(dtype=str)

candidate_key_check = np.where(
    g1 <= g2,
    np.char.add(
        np.char.add(g1, "||"),
        g2
    ),
    np.char.add(
        np.char.add(g2, "||"),
        g1
    )
)

checkpoint(
    np.array_equal(
        candidate_key_check,
        edges["edge_key"].astype(str).to_numpy()
    ),
    "candidate edge_key uses canonical undirected gene ordering"
)


rg1 = ref["gene1"].to_numpy(dtype=str)
rg2 = ref["gene2"].to_numpy(dtype=str)

ref_keys = np.where(
    rg1 <= rg2,
    np.char.add(
        np.char.add(rg1, "||"),
        rg2
    ),
    np.char.add(
        np.char.add(rg2, "||"),
        rg1
    )
)

candidate_index_by_key = pd.Series(
    np.arange(len(edges), dtype=np.int32),
    index=edges["edge_key"].astype(str)
)

checkpoint(
    set(ref_keys).issubset(
        set(candidate_index_by_key.index)
    ),
    "all projected reference edges exist in candidate universe"
)

ref_candidate_idx = (
    candidate_index_by_key
    .loc[ref_keys]
    .to_numpy(dtype=np.int32)
)

checkpoint(
    len(np.unique(ref_candidate_idx)) == 17858,
    "reference projection contains 17,858 unique candidate edges"
)

ref_mask = np.zeros(
    len(edges),
    dtype=bool
)

ref_mask[ref_candidate_idx] = True

checkpoint(
    ref_mask.sum() == 17858,
    "reference mask contains exactly 17,858 edges"
)


# ============================================================
# 8. PRECOMPUTE REFERENCE CROSS-PRODUCTS
#
# Since:
#
# r = Sxy / sqrt(Sxx*Syy)
#
# Sxy = r * sqrt(Sxx*Syy)
# ============================================================

SXY_ref = (
    r_ref
    * den_ref
)

factor = (
    N_REF
    / (N_REF + 1.0)
)


# ============================================================
# 9. PREPARE OUTPUT ARRAYS
# ============================================================

patient_ids = np.array(
    T.columns.astype(str),
    dtype=str
)

n_patients = len(patient_ids)

selected_idx = np.empty(
    (n_patients, N_KEEP),
    dtype=np.int32
)

selected_sign = np.empty(
    (n_patients, N_KEEP),
    dtype=np.int8
)

selected_spcc = np.empty(
    (n_patients, N_KEEP),
    dtype=np.float32
)

audit_rows = []

T_arr = T.to_numpy(dtype=np.float64)


# ============================================================
# 10. COMPUTE EACH CGGA PATIENT NETWORK
# ============================================================

print("\nComputing 104 patient-specific networks...\n")

for p_idx, patient in enumerate(patient_ids):

    x_patient = T_arr[:, p_idx]

    xp = x_patient[edge_i]
    yp = x_patient[edge_j]

    dx = (
        xp
        - mu[edge_i]
    )

    dy = (
        yp
        - mu[edge_j]
    )

    # --------------------------------------------------------
    # updated centered sums after adding one patient
    # --------------------------------------------------------

    SXX_aug = (
        SS[edge_i]
        + factor * dx * dx
    )

    SYY_aug = (
        SS[edge_j]
        + factor * dy * dy
    )

    SXY_aug = (
        SXY_ref
        + factor * dx * dy
    )

    r_aug = (
        SXY_aug
        / np.sqrt(
            SXX_aug
            * SYY_aug
        )
    )

    # only numerical protection
    r_aug = np.clip(
        r_aug,
        -1.0,
        1.0
    )

    spcc = (
        (N_REF + 1.0) * r_aug
        - N_REF * r_ref
    )

    checkpoint(
        np.isfinite(spcc).all(),
        f"{patient}: all sPCC values finite"
    )

    abs_spcc = np.abs(spcc)

    # deterministic ranking:
    # stable sort preserves frozen candidate-file order for exact ties
    order = np.argsort(
        -abs_spcc,
        kind="mergesort"
    )

    sel = order[:N_KEEP]

    sel_sign = (
        np.sign(spcc[sel])
        .astype(np.int8)
    )

    checkpoint(
        len(sel) == N_KEEP,
        f"{patient}: exactly 17,858 edges retained"
    )

    checkpoint(
        len(np.unique(sel)) == N_KEEP,
        f"{patient}: retained edges are unique"
    )

    checkpoint(
        np.all(sel_sign != 0),
        f"{patient}: no zero-sPCC edge retained"
    )

    selected_idx[p_idx] = sel
    selected_sign[p_idx] = sel_sign
    selected_spcc[p_idx] = (
        spcc[sel]
        .astype(np.float32)
    )

    # --------------------------------------------------------
    # audit against projected healthy reference
    # --------------------------------------------------------

    sel_ref = ref_mask[sel]

    n_overlap = int(
        sel_ref.sum()
    )

    n_same_sign_overlap = int(
        np.sum(
            sel_sign[sel_ref]
            ==
            ref_sign_candidate[
                sel[sel_ref]
            ]
        )
    )

    n_sign_flip_overlap = (
        n_overlap
        - n_same_sign_overlap
    )

    audit_rows.append({
        "patient": patient,

        "candidate_edges":
            len(edges),

        "retained_edges":
            len(sel),

        "min_abs_sPCC_retained":
            float(abs_spcc[sel].min()),

        "median_abs_sPCC_retained":
            float(np.median(abs_spcc[sel])),

        "max_abs_sPCC_retained":
            float(abs_spcc[sel].max()),

        "positive_retained":
            int(np.sum(sel_sign == 1)),

        "negative_retained":
            int(np.sum(sel_sign == -1)),

        "reference_edge_overlap":
            n_overlap,

        "reference_overlap_fraction":
            n_overlap / N_KEEP,

        "same_sign_reference_overlap":
            n_same_sign_overlap,

        "sign_flip_reference_overlap":
            n_sign_flip_overlap
    })

    if (
        (p_idx + 1) % 10 == 0
        or p_idx == 0
        or p_idx == n_patients - 1
    ):
        print(
            f"Completed "
            f"{p_idx + 1}/{n_patients}: "
            f"{patient}"
        )


# ============================================================
# 11. FINAL AUDIT
# ============================================================

audit = pd.DataFrame(
    audit_rows
)

print("\n" + "=" * 80)
print("PATIENT-NETWORK AUDIT")
print("=" * 80)

print(
    "\nMinimum retained |sPCC|:"
)
print(
    audit[
        "min_abs_sPCC_retained"
    ].describe()
)

print(
    "\nReference-edge overlap:"
)
print(
    audit[
        "reference_edge_overlap"
    ].describe()
)

print(
    "\nReference-overlap fraction:"
)
print(
    audit[
        "reference_overlap_fraction"
    ].describe()
)

print(
    "\nSign flips among overlapping reference edges:"
)
print(
    audit[
        "sign_flip_reference_overlap"
    ].describe()
)

checkpoint(
    (audit["retained_edges"] == 17858).all(),
    "all 104 patients retain exactly 17,858 edges"
)

checkpoint(
    selected_idx.shape == (104, 17858),
    "selected-index matrix = 104 × 17,858"
)

checkpoint(
    selected_sign.shape == (104, 17858),
    "selected-sign matrix = 104 × 17,858"
)


# ============================================================
# 12. SAVE COMPACT FROZEN PATIENT NETWORK REPRESENTATION
#
# Avoid a huge 1.86-million-row CSV.
# ============================================================

np.savez_compressed(
    OUT_DIR
    / "CGGA_104_FROZEN_TOP17858_SPCC_NETWORKS.npz",

    patient_ids=patient_ids,

    selected_candidate_indices=
        selected_idx,

    selected_signs=
        selected_sign,

    selected_spcc=
        selected_spcc
)

audit.to_csv(
    OUT_DIR
    / "CGGA_104_SPCC_NETWORK_AUDIT.csv",
    index=False
)


# ============================================================
# 13. SAVE FREEZE SUMMARY
# ============================================================

summary = pd.DataFrame({
    "Item": [
        "GTEx reference N",
        "Compatible candidate edges",
        "Projected reference edges",
        "Edges retained per CGGA patient",
        "CGGA patients",
        "Ranking rule",
        "Reference re-ranked",
        "Survival outcomes used"
    ],

    "Value": [
        334,
        89551,
        17858,
        17858,
        104,
        "descending absolute sPCC",
        "No",
        "No"
    ]
})

summary.to_csv(
    OUT_DIR
    / "CGGA_SPCC_NETWORK_FREEZE_SUMMARY.csv",
    index=False
)


print("\n" + "=" * 80)
print("STEP 4A COMPLETE — STOP HERE")
print("=" * 80)

print("\nFrozen network representation:")
print(
    selected_idx.shape,
    "patient × retained-edge indices"
)

print("""
DO NOT compute eNESH yet.

Send the final audit output first.
""")

### STEP 4A-CONTROL — NEGATIVE-CONTROL NETWORK TRANSPORT AUDIT

In [ ]:
# ============================================================
# STEP 4A-CONTROL — NEGATIVE-CONTROL NETWORK TRANSPORT AUDIT
#
# Each CGGA non-glioma control:
#   1. is held out
#   2. is harmonized using the OTHER 19 controls only
#   3. is added to GTEx-334
#   4. gets an sPCC network
#   5. retains top 17,858 |sPCC| edges
#
# No survival information is used.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)

DOWNLOADS = BASE

OUT_DIR = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
)

GTEX_FILE = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
    / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
)

CONTROL_FILE = (
    OUT_DIR
    / "CGGA_NORMAL20_LOG2CPM_FROZEN_GENES.csv"
)

EDGE_FILE = (
    OUT_DIR
    / "CGGA_FROZEN_COMPATIBLE_EDGE_UNIVERSE.csv"
)

REF_FILE = (
    OUT_DIR
    / "CGGA_FROZEN_COMPATIBLE_REFERENCE_NETWORK.csv"
)

TUMOR_AUDIT_FILE = (
    OUT_DIR
    / "CGGA_104_SPCC_NETWORK_AUDIT.csv"
)

N_REF = 334
N_KEEP = 17858

print("\n" + "=" * 80)
print("STEP 4A-CONTROL — LOOCV NEGATIVE-CONTROL NETWORK AUDIT")
print("=" * 80)

# ============================================================
# 1. LOAD
# ============================================================

gtex = pd.read_csv(
    GTEX_FILE,
    index_col=0,
    low_memory=False
)

controls = pd.read_csv(
    CONTROL_FILE,
    index_col=0,
    low_memory=False
)

edges = pd.read_csv(
    EDGE_FILE,
    low_memory=False
)

ref = pd.read_csv(
    REF_FILE,
    low_memory=False
)

tumor_audit = pd.read_csv(
    TUMOR_AUDIT_FILE
)

for df in [gtex, controls]:
    df.index = (
        df.index.astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

for col in ["gene1", "gene2"]:
    edges[col] = (
        edges[col].astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

    ref[col] = (
        ref[col].astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

checkpoint(
    controls.shape == (12888, 20),
    "raw normalized CGGA controls = 12,888 × 20"
)

checkpoint(
    len(edges) == 89551,
    "candidate edge universe = 89,551"
)

checkpoint(
    len(ref) == 17858,
    "projected reference = 17,858"
)

# ============================================================
# 2. ALIGN GTEx
# ============================================================

genes = controls.index.tolist()

G = (
    gtex
    .loc[genes]
    .astype(np.float64)
)

C = controls.astype(np.float64)

checkpoint(
    G.shape == (12888, 334),
    "aligned GTEx = 12,888 × 334"
)

# ============================================================
# 3. EDGE ENDPOINT LOOKUP
# ============================================================

gene_to_pos = {
    gene: i
    for i, gene in enumerate(genes)
}

edge_i = np.array(
    [gene_to_pos[g] for g in edges["gene1"]],
    dtype=np.int32
)

edge_j = np.array(
    [gene_to_pos[g] for g in edges["gene2"]],
    dtype=np.int32
)

# ============================================================
# 4. GTEx REFERENCE STATISTICS
# ============================================================

G_arr = G.to_numpy(dtype=np.float64)

mu = G_arr.mean(axis=1)

G_centered = (
    G_arr
    - mu[:, None]
)

SS = np.einsum(
    "ij,ij->i",
    G_centered,
    G_centered
)

den_ref = np.sqrt(
    SS[edge_i]
    * SS[edge_j]
)

r_ref = (
    edges["reference_PCC"]
    .to_numpy(dtype=np.float64)
)

ref_sign_candidate = (
    edges["reference_sign"]
    .to_numpy(dtype=np.int8)
)

SXY_ref = (
    r_ref
    * den_ref
)

factor = (
    N_REF
    / (N_REF + 1.0)
)

# ============================================================
# 5. FROZEN REFERENCE MASK
# ============================================================

g1 = edges["gene1"].to_numpy(dtype=str)
g2 = edges["gene2"].to_numpy(dtype=str)

candidate_keys = np.where(
    g1 <= g2,
    np.char.add(
        np.char.add(g1, "||"),
        g2
    ),
    np.char.add(
        np.char.add(g2, "||"),
        g1
    )
)

candidate_index = pd.Series(
    np.arange(len(edges)),
    index=candidate_keys
)

rg1 = ref["gene1"].to_numpy(dtype=str)
rg2 = ref["gene2"].to_numpy(dtype=str)

ref_keys = np.where(
    rg1 <= rg2,
    np.char.add(
        np.char.add(rg1, "||"),
        rg2
    ),
    np.char.add(
        np.char.add(rg2, "||"),
        rg1
    )
)

ref_idx = (
    candidate_index
    .loc[ref_keys]
    .to_numpy(dtype=np.int32)
)

ref_mask = np.zeros(
    len(edges),
    dtype=bool
)

ref_mask[ref_idx] = True

checkpoint(
    ref_mask.sum() == 17858,
    "reference mask = 17,858 edges"
)

# ============================================================
# 6. LOOCV CONTROL NETWORKS
# ============================================================

C_arr = C.to_numpy(dtype=np.float64)

rows = []

for held_idx, held_sample in enumerate(C.columns):

    train_cols = [
        c for c in C.columns
        if c != held_sample
    ]

    # --------------------------------------------------------
    # location correction estimated from OTHER 19 controls
    # --------------------------------------------------------

    mu_cgga_19 = (
        C[train_cols]
        .mean(axis=1)
        .to_numpy(dtype=np.float64)
    )

    delta = (
        mu
        - mu_cgga_19
    )

    x_raw = (
        C[held_sample]
        .to_numpy(dtype=np.float64)
    )

    x_harm = (
        x_raw
        + delta
    )

    # --------------------------------------------------------
    # sPCC
    # --------------------------------------------------------

    xp = x_harm[edge_i]
    yp = x_harm[edge_j]

    dx = (
        xp
        - mu[edge_i]
    )

    dy = (
        yp
        - mu[edge_j]
    )

    SXX_aug = (
        SS[edge_i]
        + factor * dx * dx
    )

    SYY_aug = (
        SS[edge_j]
        + factor * dy * dy
    )

    SXY_aug = (
        SXY_ref
        + factor * dx * dy
    )

    r_aug = (
        SXY_aug
        / np.sqrt(
            SXX_aug * SYY_aug
        )
    )

    r_aug = np.clip(
        r_aug,
        -1.0,
        1.0
    )

    spcc = (
        (N_REF + 1.0) * r_aug
        - N_REF * r_ref
    )

    checkpoint(
        np.isfinite(spcc).all(),
        f"{held_sample}: finite sPCC"
    )

    # --------------------------------------------------------
    # exact same top-K rule
    # --------------------------------------------------------

    abs_spcc = np.abs(spcc)

    order = np.argsort(
        -abs_spcc,
        kind="mergesort"
    )

    sel = order[:N_KEEP]

    sel_sign = (
        np.sign(spcc[sel])
        .astype(np.int8)
    )

    # --------------------------------------------------------
    # comparison with frozen healthy reference
    # --------------------------------------------------------

    in_ref = ref_mask[sel]

    overlap = int(
        in_ref.sum()
    )

    same_sign = int(
        np.sum(
            sel_sign[in_ref]
            ==
            ref_sign_candidate[
                sel[in_ref]
            ]
        )
    )

    flips = (
        overlap
        - same_sign
    )

    union_size = (
        N_KEEP
        + N_KEEP
        - overlap
    )

    rows.append({
        "control": held_sample,

        "retained_edges":
            N_KEEP,

        "min_abs_sPCC_retained":
            float(
                abs_spcc[sel].min()
            ),

        "reference_edge_overlap":
            overlap,

        "reference_overlap_fraction":
            overlap / N_KEEP,

        "same_sign_reference_overlap":
            same_sign,

        "same_sign_reference_fraction":
            same_sign / N_KEEP,

        "sign_flip_reference_overlap":
            flips,

        "sign_flip_fraction_among_overlap":
            flips / overlap
            if overlap > 0 else np.nan,

        "edge_jaccard_with_reference":
            overlap / union_size
    })

control_audit = pd.DataFrame(rows)

# ============================================================
# 7. SUMMARIZE NEGATIVE CONTROLS
# ============================================================

print("\n" + "=" * 80)
print("NEGATIVE-CONTROL NETWORK SUMMARY")
print("=" * 80)

for col in [
    "min_abs_sPCC_retained",
    "reference_edge_overlap",
    "reference_overlap_fraction",
    "same_sign_reference_fraction",
    "sign_flip_fraction_among_overlap",
    "edge_jaccard_with_reference"
]:
    print(f"\n{col}:")
    print(control_audit[col].describe())

# ============================================================
# 8. COMPARE AGAINST 104 GBM TUMORS
# ============================================================

tumor_flip_fraction = (
    tumor_audit["sign_flip_reference_overlap"]
    /
    tumor_audit["reference_edge_overlap"]
)

print("\n" + "=" * 80)
print("CONTROL vs GBM — MEDIAN COMPARISON")
print("=" * 80)

print(
    "Reference overlap fraction:"
)
print(
    "  controls median =",
    control_audit[
        "reference_overlap_fraction"
    ].median()
)
print(
    "  tumors median   =",
    tumor_audit[
        "reference_overlap_fraction"
    ].median()
)

print(
    "\nSign-flip fraction among overlapping edges:"
)
print(
    "  controls median =",
    control_audit[
        "sign_flip_fraction_among_overlap"
    ].median()
)
print(
    "  tumors median   =",
    tumor_flip_fraction.median()
)

print(
    "\nMinimum retained |sPCC|:"
)
print(
    "  controls median =",
    control_audit[
        "min_abs_sPCC_retained"
    ].median()
)
print(
    "  tumors median   =",
    tumor_audit[
        "min_abs_sPCC_retained"
    ].median()
)

# ============================================================
# 9. SAVE
# ============================================================

control_audit.to_csv(
    OUT_DIR
    / "CGGA_NORMAL20_LOOCV_SPCC_NETWORK_AUDIT.csv",
    index=False
)

print("\n" + "=" * 80)
print("STEP 4A-CONTROL COMPLETE — STOP HERE")
print("=" * 80)

print("""
Do NOT compute eNESH yet.

We first need to verify that held-out non-glioma controls
are more reference-like than GBM tumors.
""")

### STEP 4B — EXTERNAL CGGA eNESH RECONSTRUCTION

In [ ]:
# ============================================================
# STEP 4B — EXTERNAL CGGA eNESH RECONSTRUCTION
#
# Uses:
#   - frozen projected CGGA-compatible reference network
#   - frozen 104 patient top-17,858 sPCC networks
#   - exact original TCGA eNESH definition
#   - original frozen dmax = 166
#
# NO survival outcomes are used here.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ------------------------------------------------------------
# helper
# ------------------------------------------------------------

def checkpoint(condition, message):
    if condition:
        print(f"[CHECKPOINT PASS] {message}")
    else:
        print(f"[CHECKPOINT FAIL] {message}")
        raise AssertionError(message)


# ============================================================
# 0. PATHS
# ============================================================

DOWNLOADS = BASE

OUT_DIR = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
)

TCGA_STAGE_C = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)

EDGE_FILE = (
    OUT_DIR
    / "CGGA_FROZEN_COMPATIBLE_EDGE_UNIVERSE.csv"
)

REF_FILE = (
    OUT_DIR
    / "CGGA_FROZEN_COMPATIBLE_REFERENCE_NETWORK.csv"
)

NETWORK_FILE = (
    OUT_DIR
    / "CGGA_104_FROZEN_TOP17858_SPCC_NETWORKS.npz"
)

TUMOR_FILE = (
    OUT_DIR
    / "CGGA_104_HARMONIZED_TO_GTEX.csv"
)

TCGA_ENESH_FILE = (
    TCGA_STAGE_C
    / "STAGE_C_FINAL_REWIRING_MATRIX_144xANCHORS_20PCT.csv"
)

ORIGINAL_REF_FILE = (
    TCGA_STAGE_C
    / "STAGE_C_FINAL_REFERENCE_EDGES_20PCT.csv"
)

DMAX = 166

print("\n" + "=" * 80)
print("STEP 4B — CGGA EXTERNAL eNESH RECONSTRUCTION")
print("=" * 80)


# ============================================================
# 1. CHECK FILES
# ============================================================

for path, name in [
    (EDGE_FILE, "compatible edge universe"),
    (REF_FILE, "projected reference network"),
    (NETWORK_FILE, "104 patient sPCC networks"),
    (TUMOR_FILE, "harmonized tumor matrix"),
    (TCGA_ENESH_FILE, "original TCGA eNESH matrix"),
    (ORIGINAL_REF_FILE, "original TCGA reference network")
]:
    checkpoint(
        path.exists(),
        f"{name} exists"
    )


# ============================================================
# 2. LOAD TOPOLOGY
# ============================================================

edges = pd.read_csv(
    EDGE_FILE,
    low_memory=False
)

ref = pd.read_csv(
    REF_FILE,
    low_memory=False
)

original_ref = pd.read_csv(
    ORIGINAL_REF_FILE,
    low_memory=False
)

tumors = pd.read_csv(
    TUMOR_FILE,
    index_col=0,
    low_memory=False
)

for df in [tumors]:
    df.index = (
        df.index.astype(str)
        .str.split(".").str[0]
        .str.strip()
    )

for df in [edges, ref, original_ref]:
    for col in ["gene1", "gene2"]:
        df[col] = (
            df[col].astype(str)
            .str.split(".").str[0]
            .str.strip()
        )

checkpoint(
    len(edges) == 89551,
    "compatible edge universe = 89,551"
)

checkpoint(
    len(ref) == 17858,
    "projected reference network = 17,858"
)

checkpoint(
    len(original_ref) == 18192,
    "original frozen TCGA reference = 18,192"
)


# ============================================================
# 3. LOAD EXACT ORIGINAL 3,992 ANCHOR IDENTITIES
#
# We use the ORIGINAL TCGA feature matrix header so that
# external anchor identities come directly from the frozen
# training representation.
# ============================================================

tcga_header = pd.read_csv(
    TCGA_ENESH_FILE,
    index_col=0,
    nrows=0
)

tcga_anchors = [
    str(g).split(".")[0].strip()
    for g in tcga_header.columns
]

checkpoint(
    len(tcga_anchors) == 3992,
    "original frozen TCGA eNESH representation has 3,992 anchors"
)

checkpoint(
    len(set(tcga_anchors)) == 3992,
    "original TCGA anchor identities are unique"
)


# ============================================================
# 4. DEFINE EXTERNALLY MEASURABLE ANCHORS
# ============================================================

available_genes = set(
    tumors.index.astype(str)
)

external_anchors = [
    g for g in tcga_anchors
    if g in available_genes
]

missing_anchors = [
    g for g in tcga_anchors
    if g not in available_genes
]

print("\nOriginal TCGA anchors:", len(tcga_anchors))
print("Externally measurable:", len(external_anchors))
print("Unavailable in CGGA:", len(missing_anchors))

checkpoint(
    len(external_anchors) == 3944,
    "exactly 3,944 frozen anchors are measurable in CGGA"
)

checkpoint(
    len(missing_anchors) == 48,
    "exactly 48 frozen anchors are unavailable externally"
)

anchor_set = set(external_anchors)


# save unavailable anchor list
pd.DataFrame({
    "missing_external_anchor": missing_anchors
}).to_csv(
    OUT_DIR
    / "CGGA_48_UNAVAILABLE_FROZEN_ANCHORS.csv",
    index=False
)


# ============================================================
# 5. BUILD ORIGINAL REFERENCE CONNECTIVITY AUDIT
# ============================================================

original_neighbors = defaultdict(set)

for row in original_ref.itertuples(index=False):

    g1 = row.gene1
    g2 = row.gene2

    if g1 in anchor_set:
        original_neighbors[g1].add(g2)

    if g2 in anchor_set:
        original_neighbors[g2].add(g1)

original_connected = sum(
    len(original_neighbors[g]) > 0
    for g in external_anchors
)

print(
    "\nExternally measurable anchors connected "
    "in ORIGINAL reference:",
    original_connected
)

checkpoint(
    original_connected == 2604,
    "2,604 measurable anchors were connected in original reference"
)


# ============================================================
# 6. BUILD PROJECTED SIGNED REFERENCE NEIGHBORHOODS
# ============================================================

ref_pos = {
    g: set()
    for g in external_anchors
}

ref_neg = {
    g: set()
    for g in external_anchors
}

for row in ref.itertuples(index=False):

    g1 = row.gene1
    g2 = row.gene2
    sign = int(row.reference_sign)

    checkpoint(
        sign in (-1, 1),
        "projected reference signs are ±1"
    )

    if g1 in anchor_set:

        if sign == 1:
            ref_pos[g1].add(g2)
        else:
            ref_neg[g1].add(g2)

    if g2 in anchor_set:

        if sign == 1:
            ref_pos[g2].add(g1)
        else:
            ref_neg[g2].add(g1)


ref_degree = {
    g:
    len(ref_pos[g])
    + len(ref_neg[g])

    for g in external_anchors
}

projected_connected = sum(
    ref_degree[g] > 0
    for g in external_anchors
)

newly_isolated = sum(
    (len(original_neighbors[g]) > 0)
    and
    (ref_degree[g] == 0)

    for g in external_anchors
)

projected_max_degree = max(
    ref_degree.values()
)

print("\nProjected reference audit:")
print(
    "Connected measurable anchors:",
    projected_connected
)
print(
    "Newly isolated after availability projection:",
    newly_isolated
)
print(
    "Projected maximum anchor degree:",
    projected_max_degree
)
print(
    "Frozen eNESH dmax:",
    DMAX
)

checkpoint(
    projected_connected == 2598,
    "2,598 measurable anchors remain reference-connected"
)

checkpoint(
    newly_isolated == 6,
    "exactly 6 anchors became isolated after CGGA availability projection"
)

checkpoint(
    projected_max_degree <= DMAX,
    "projected reference degree does not exceed frozen dmax=166"
)

# IMPORTANT:
# DO NOT replace DMAX=166 with projected_max_degree.
# dmax is part of the frozen TCGA eNESH definition.


# ============================================================
# 7. LOAD FROZEN PATIENT NETWORKS
# ============================================================

net = np.load(
    NETWORK_FILE,
    allow_pickle=True
)

print("\nNetwork cache keys:")
for k in net.files:
    print(
        k,
        net[k].shape,
        net[k].dtype
    )

required = {
    "patient_ids",
    "selected_candidate_indices",
    "selected_signs",
    "selected_spcc"
}

checkpoint(
    required.issubset(set(net.files)),
    "all required patient-network arrays exist"
)

patient_ids = (
    net["patient_ids"]
    .astype(str)
)

selected_idx = (
    net["selected_candidate_indices"]
    .astype(np.int32)
)

selected_sign = (
    net["selected_signs"]
    .astype(np.int8)
)

checkpoint(
    patient_ids.shape == (104,),
    "104 external patient IDs"
)

checkpoint(
    selected_idx.shape == (104, 17858),
    "patient selected-edge matrix = 104 × 17,858"
)

checkpoint(
    selected_sign.shape == (104, 17858),
    "patient sign matrix = 104 × 17,858"
)

checkpoint(
    np.all(
        (selected_idx >= 0)
        &
        (selected_idx < len(edges))
    ),
    "all stored patient edge indices are valid"
)

checkpoint(
    np.all(
        np.isin(
            selected_sign,
            [-1, 1]
        )
    ),
    "all retained patient edge signs are ±1"
)


# ============================================================
# 8. EDGE ARRAYS
# ============================================================

edge_g1 = (
    edges["gene1"]
    .astype(str)
    .to_numpy()
)

edge_g2 = (
    edges["gene2"]
    .astype(str)
    .to_numpy()
)


# ============================================================
# 9. COMPUTE EXACT EXTERNAL eNESH
#
# P = same-sign preserved reference relationships
# L = reference relationships NOT retained with same sign
#
# f = |P| / |NR union NP|
# g = |L| / 166
# h = |L| / |NR union NP|
#
# eNESH = 1 - [ f - (g + h) ]
#
# Empty union -> 0
# Reference empty + patient nonempty -> 1
# ============================================================

n_patients = len(patient_ids)
n_anchors = len(external_anchors)

enesh = np.zeros(
    (n_patients, n_anchors),
    dtype=np.float64
)

patient_summary_rows = []

EMPTY = frozenset()

print("\nComputing external eNESH...\n")

for p in range(n_patients):

    # patient signed neighborhoods only for anchors
    p_pos = defaultdict(set)
    p_neg = defaultdict(set)

    for idx, sign in zip(
        selected_idx[p],
        selected_sign[p]
    ):

        g1 = edge_g1[idx]
        g2 = edge_g2[idx]

        if g1 in anchor_set:

            if sign == 1:
                p_pos[g1].add(g2)
            else:
                p_neg[g1].add(g2)

        if g2 in anchor_set:

            if sign == 1:
                p_pos[g2].add(g1)
            else:
                p_neg[g2].add(g1)

    patient_values = enesh[p]

    for a_idx, anchor in enumerate(
        external_anchors
    ):

        Rpos = ref_pos[anchor]
        Rneg = ref_neg[anchor]

        Ppos = p_pos.get(
            anchor,
            EMPTY
        )

        Pneg = p_neg.get(
            anchor,
            EMPTY
        )

        NR = (
            Rpos
            | Rneg
        )

        NP = (
            Ppos
            | Pneg
        )

        union = (
            NR
            | NP
        )

        union_n = len(union)

        # exact predefined rule
        if union_n == 0:

            patient_values[a_idx] = 0.0
            continue

        # same-sign preserved relationships
        preserved = (
            (Rpos & Ppos)
            |
            (Rneg & Pneg)
        )

        # reference relationships not retained
        # with the SAME sign.
        #
        # A sign flip is automatically included here.
        lost = (
            (Rpos - Ppos)
            |
            (Rneg - Pneg)
        )

        n_preserved = len(preserved)
        n_lost = len(lost)

        f = (
            n_preserved
            / union_n
        )

        g = (
            n_lost
            / DMAX
        )

        h = (
            n_lost
            / union_n
        )

        patient_values[a_idx] = (
            1.0
            -
            (
                f
                -
                (g + h)
            )
        )

    patient_summary_rows.append({
        "patient":
            patient_ids[p],

        "mean_eNESH":
            float(
                patient_values.mean()
            ),

        "median_eNESH":
            float(
                np.median(
                    patient_values
                )
            ),

        "max_eNESH":
            float(
                patient_values.max()
            ),

        "zero_fraction":
            float(
                np.mean(
                    patient_values == 0
                )
            ),

        "one_fraction":
            float(
                np.mean(
                    patient_values == 1
                )
            )
    })

    if (
        (p + 1) % 10 == 0
        or p == 0
        or p == n_patients - 1
    ):

        print(
            f"Completed "
            f"{p+1}/{n_patients}: "
            f"{patient_ids[p]}"
        )


# ============================================================
# 10. HARD MATHEMATICAL CHECKS
# ============================================================

checkpoint(
    np.isfinite(enesh).all(),
    "all external eNESH values are finite"
)

checkpoint(
    np.min(enesh) >= -1e-12,
    "all external eNESH values are non-negative"
)

checkpoint(
    np.max(enesh) <= 3.0 + 1e-12,
    "all external eNESH values respect theoretical upper bound ≤3"
)


# ============================================================
# 11. REFERENCE-ISOLATED RULE CHECK
# ============================================================

isolated_anchor_idx = [
    i
    for i, g in enumerate(external_anchors)
    if ref_degree[g] == 0
]

connected_anchor_idx = [
    i
    for i, g in enumerate(external_anchors)
    if ref_degree[g] > 0
]

isolated_vals = (
    enesh[
        :,
        isolated_anchor_idx
    ]
)

checkpoint(
    np.all(
        np.isin(
            isolated_vals,
            [0.0, 1.0]
        )
    ),
    "reference-isolated anchors take only eNESH 0 or 1"
)


# ============================================================
# 12. GLOBAL eNESH AUDIT
# ============================================================

print("\n" + "=" * 80)
print("EXTERNAL eNESH GLOBAL AUDIT")
print("=" * 80)

print("\nMatrix shape:")
print(enesh.shape)

print("\nAll values:")
print(
    pd.Series(
        enesh.ravel()
    ).describe(
        percentiles=[
            .01,
            .05,
            .25,
            .50,
            .75,
            .95,
            .99
        ]
    )
)

zero_rate = (
    100
    * np.mean(
        enesh == 0
    )
)

one_rate = (
    100
    * np.mean(
        enesh == 1
    )
)

print(
    "\nExact-zero rate:",
    f"{zero_rate:.4f}%"
)

print(
    "Exact-one rate:",
    f"{one_rate:.4f}%"
)

print(
    "Overall mean:",
    float(
        enesh.mean()
    )
)

print(
    "Overall median:",
    float(
        np.median(enesh)
    )
)

print(
    "Overall maximum:",
    float(
        enesh.max()
    )
)


# ============================================================
# 13. PATIENT-TO-PATIENT FEATURE VARIATION
# ============================================================

anchor_sd = (
    enesh.std(
        axis=0,
        ddof=1
    )
)

varying_mask = (
    anchor_sd > 1e-12
)

n_varying = int(
    varying_mask.sum()
)

n_constant = (
    n_anchors
    - n_varying
)

print("\nAnchor variability:")
print(
    "Varying anchors:",
    n_varying,
    "/",
    n_anchors
)

print(
    "Constant anchors:",
    n_constant,
    "/",
    n_anchors
)

print(
    "Percent varying:",
    100
    * n_varying
    / n_anchors
)


# ============================================================
# 14. ANCHOR-LEVEL AUDIT TABLE
# ============================================================

anchor_audit = pd.DataFrame({
    "ensembl_id":
        external_anchors,

    "projected_reference_degree":
        [
            ref_degree[g]
            for g in external_anchors
        ],

    "reference_connected":
        [
            ref_degree[g] > 0
            for g in external_anchors
        ],

    "original_reference_connected":
        [
            len(
                original_neighbors[g]
            ) > 0
            for g in external_anchors
        ],

    "became_isolated_after_projection":
        [
            (
                len(
                    original_neighbors[g]
                ) > 0
            )
            and
            (
                ref_degree[g] == 0
            )
            for g in external_anchors
        ],

    "mean_eNESH":
        enesh.mean(
            axis=0
        ),

    "median_eNESH":
        np.median(
            enesh,
            axis=0
        ),

    "sd_eNESH":
        anchor_sd,

    "min_eNESH":
        enesh.min(
            axis=0
        ),

    "max_eNESH":
        enesh.max(
            axis=0
        ),

    "zero_fraction":
        np.mean(
            enesh == 0,
            axis=0
        ),

    "one_fraction":
        np.mean(
            enesh == 1,
            axis=0
        )
})


# ============================================================
# 15. SAVE EXTERNAL FEATURE MATRIX
# ============================================================

external_enesh = pd.DataFrame(
    enesh,
    index=patient_ids,
    columns=external_anchors
)

external_enesh.index.name = "patient"

external_enesh.to_csv(
    OUT_DIR
    / "CGGA_104_EXTERNAL_ENESH_3944.csv"
)

anchor_audit.to_csv(
    OUT_DIR
    / "CGGA_EXTERNAL_ENESH_ANCHOR_AUDIT.csv",
    index=False
)

patient_summary = pd.DataFrame(
    patient_summary_rows
)

patient_summary.to_csv(
    OUT_DIR
    / "CGGA_EXTERNAL_ENESH_PATIENT_AUDIT.csv",
    index=False
)


# ============================================================
# 16. SAVE FREEZE SUMMARY
# ============================================================

freeze_summary = pd.DataFrame({
    "Item": [
        "CGGA patients",
        "Original frozen anchors",
        "Externally measurable anchors",
        "Unavailable anchors",
        "Original reference-connected measurable anchors",
        "Projected reference-connected anchors",
        "Newly isolated after projection",
        "Frozen dmax",
        "Candidate edges",
        "Patient edges",
        "eNESH zero rate percent",
        "Varying external anchors",
        "Survival outcomes used"
    ],

    "Value": [
        104,
        3992,
        3944,
        48,
        original_connected,
        projected_connected,
        newly_isolated,
        DMAX,
        89551,
        17858,
        zero_rate,
        n_varying,
        "No"
    ]
})

freeze_summary.to_csv(
    OUT_DIR
    / "CGGA_EXTERNAL_ENESH_FREEZE_SUMMARY.csv",
    index=False
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("STEP 4B COMPLETE — EXTERNAL eNESH RECONSTRUCTED")
print("=" * 80)

print("\nFeature matrix:")
print(
    external_enesh.shape
)

print("\nReference anchor accounting:")
print(
    "Original frozen anchors:       ",
    3992
)
print(
    "Externally measurable:         ",
    3944
)
print(
    "Unavailable externally:        ",
    48
)
print(
    "Original connected measurable: ",
    original_connected
)
print(
    "Projected connected:           ",
    projected_connected
)
print(
    "Newly isolated:                ",
    newly_isolated
)
print(
    "Frozen dmax:                   ",
    DMAX
)

print("\neNESH:")
print(
    "Zero rate:     ",
    f"{zero_rate:.4f}%"
)
print(
    "Mean:          ",
    f"{enesh.mean():.6f}"
)
print(
    "Median:        ",
    f"{np.median(enesh):.6f}"
)
print(
    "Maximum:       ",
    f"{enesh.max():.6f}"
)
print(
    "Varying genes: ",
    f"{n_varying}/{n_anchors}"
)

print("""
STOP HERE.

Do NOT fit or tune any survival model yet.

Next:
verify that the exact frozen TCGA model features required
by M3/M5/M6 are present in the external eNESH matrix.
""")

### RECONSTRUCT EXACT FINAL V4 M3

In [ ]:
# ============================================================
# RECONSTRUCT EXACT FINAL V4 M3
# + AUDIT USE OF THE ONE CGGA-MISSING FEATURE
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored


# ============================================================
# 0. PATHS
# ============================================================

DOWNLOADS = BASE

STAGE_D = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

MODEL_DIR = (
    STAGE_D
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)

SPLIT_FILE = (
    STAGE_D
    / "STAGE_D_LOCKED_SPLIT.csv"
)

SURVIVAL_FILE = (
    STAGE_D
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

REWIRE_FILE = (
    STAGE_D
    / "STAGE_D_M3_REWIRING_RAW.csv"
)

FEATURE_FILE = (
    MODEL_DIR
    / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv"
)

SUSPECT = "ENSG00000145107"

RANDOM_SEED = 42


def checkpoint(cond, msg):
    if cond:
        print(f"[CHECKPOINT PASS] {msg}")
    else:
        print(f"[CHECKPOINT FAIL] {msg}")
        raise AssertionError(msg)


def make_y(df):
    return np.array(
        [
            (bool(e), float(t))
            for e, t in zip(
                df["OS_event"],
                df["OS_months"]
            )
        ],
        dtype=[
            ("event", "?"),
            ("time", "<f8")
        ]
    )


print("=" * 80)
print("EXACT FINAL V4 M3 RECONSTRUCTION")
print("=" * 80)


# ============================================================
# 1. LOAD
# ============================================================

split_df = pd.read_csv(SPLIT_FILE)

survival = (
    pd.read_csv(SURVIVAL_FILE)
    .set_index("patient_id")
)

rewiring = pd.read_csv(
    REWIRE_FILE,
    index_col=0
)

selected = pd.read_csv(
    FEATURE_FILE
)

# force patient IDs to strings consistently
survival.index = survival.index.astype(str)
rewiring.index = rewiring.index.astype(str)
split_df["patient_id"] = split_df["patient_id"].astype(str)


# ============================================================
# 2. LOCKED 115 / 29 SPLIT
# ============================================================

development_ids = (
    split_df.loc[
        split_df["split"] == "development",
        "patient_id"
    ]
    .tolist()
)

test_ids = (
    split_df.loc[
        split_df["split"] != "development",
        "patient_id"
    ]
    .tolist()
)

print("\nSplit labels:")
print(
    split_df["split"]
    .value_counts()
)

checkpoint(
    len(development_ids) == 115,
    "115 development patients"
)

checkpoint(
    len(test_ids) == 29,
    "29 internal-test patients"
)

checkpoint(
    set(development_ids).issubset(rewiring.index),
    "all development patients present in rewiring matrix"
)

checkpoint(
    set(test_ids).issubset(rewiring.index),
    "all test patients present in rewiring matrix"
)


# ============================================================
# 3. EXACT FROZEN M3 TOP-50 FEATURES
# ============================================================

m3_features = (
    selected.loc[
        selected["model"] == "M3"
    ]
    .sort_values("rank")["feature"]
    .astype(str)
    .str.replace(
        "REWIRE::",
        "",
        regex=False
    )
    .tolist()
)

checkpoint(
    len(m3_features) == 50,
    "M3 has exactly 50 frozen rewiring features"
)

checkpoint(
    SUSPECT in m3_features,
    "CGGA-missing feature is in frozen M3"
)

checkpoint(
    set(m3_features).issubset(
        rewiring.columns.astype(str)
    ),
    "all 50 frozen M3 features exist in TCGA rewiring matrix"
)

suspect_idx = m3_features.index(SUSPECT)

print(
    "\nSuspect feature position "
    "(0-based):",
    suspect_idx
)

print(
    "Suspect feature rank "
    "(1-based):",
    suspect_idx + 1
)


# ============================================================
# 4. EXACT TRAIN / TEST MATRICES
# ============================================================

X_train = (
    rewiring
    .loc[
        development_ids,
        m3_features
    ]
    .to_numpy(
        dtype=np.float64
    )
)

X_test = (
    rewiring
    .loc[
        test_ids,
        m3_features
    ]
    .to_numpy(
        dtype=np.float64
    )
)

y_train = make_y(
    survival.loc[
        development_ids
    ]
)

y_test = make_y(
    survival.loc[
        test_ids
    ]
)

checkpoint(
    X_train.shape == (115, 50),
    "M3 training matrix = 115 × 50"
)

checkpoint(
    X_test.shape == (29, 50),
    "M3 internal-test matrix = 29 × 50"
)

checkpoint(
    np.isfinite(X_train).all(),
    "training features finite"
)

checkpoint(
    np.isfinite(X_test).all(),
    "test features finite"
)


# ============================================================
# 5. EXACT FROZEN M3 GBSA
#
# Frozen:
#   n_estimators = 100
#   learning_rate = 0.1
#   max_depth = 3
#   subsample = 0.8
#   random_state = 42
# ============================================================

m3_model = GradientBoostingSurvivalAnalysis(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

m3_model.fit(
    X_train,
    y_train
)

test_risk = m3_model.predict(
    X_test
)

test_cindex = float(
    concordance_index_censored(
        y_test["event"],
        y_test["time"],
        test_risk
    )[0]
)

print("\n" + "=" * 80)
print("RECONSTRUCTION CHECK")
print("=" * 80)

print(
    "Reconstructed internal-test C-index:",
    f"{test_cindex:.12f}"
)

print(
    "Published/saved V4 value approximately:",
    "0.640127"
)

print(
    "Absolute difference:",
    abs(
        test_cindex
        - 0.640127
    )
)


# ============================================================
# 6. INSPECT ALL GBSA TREE SPLITS
# ============================================================

estimators = np.asarray(
    m3_model.estimators_,
    dtype=object
).ravel()

split_indices = []
trees_using_suspect = []

for tree_number, estimator in enumerate(
    estimators,
    start=1
):

    feature_indices = (
        estimator
        .tree_
        .feature
    )

    used = feature_indices[
        feature_indices >= 0
    ]

    split_indices.extend(
        used.tolist()
    )

    if suspect_idx in used:
        trees_using_suspect.append(
            tree_number
        )

split_indices = np.asarray(
    split_indices,
    dtype=int
)

n_suspect_splits = int(
    np.sum(
        split_indices
        == suspect_idx
    )
)

n_trees_suspect = len(
    trees_using_suspect
)

print("\n" + "=" * 80)
print("MISSING-FEATURE TREE AUDIT")
print("=" * 80)

print(
    "Feature:",
    SUSPECT
)

print(
    "M3 rank:",
    suspect_idx + 1
)

print(
    "Total GBSA trees:",
    len(estimators)
)

print(
    "Total internal tree splits:",
    len(split_indices)
)

print(
    "Number of splits using suspect feature:",
    n_suspect_splits
)

print(
    "Number of trees using suspect feature:",
    n_trees_suspect
)

print(
    "Trees using suspect feature:",
    trees_using_suspect
)


# ============================================================
# 7. FEATURE IMPORTANCE
# ============================================================

importance = getattr(
    m3_model,
    "feature_importances_",
    None
)

if importance is not None:

    print(
        "\nSuspect feature importance:",
        float(
            importance[
                suspect_idx
            ]
        )
    )

    importance_table = (
        pd.DataFrame({
            "feature":
                m3_features,

            "importance":
                importance
        })
        .sort_values(
            "importance",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    suspect_importance_rank = int(
        importance_table
        .index[
            importance_table[
                "feature"
            ] == SUSPECT
        ][0]
        + 1
    )

    print(
        "Importance rank:",
        suspect_importance_rank,
        "/ 50"
    )


# ============================================================
# 8. DIRECT PREDICTION-INVARIANCE CHECK
#
# If the model NEVER uses this feature,
# radically changing its value should leave
# every prediction identical.
# ============================================================

X_low = X_test.copy()
X_high = X_test.copy()

X_low[
    :,
    suspect_idx
] = -999.0

X_high[
    :,
    suspect_idx
] = 999.0

risk_original = m3_model.predict(
    X_test
)

risk_low = m3_model.predict(
    X_low
)

risk_high = m3_model.predict(
    X_high
)

max_diff_low = float(
    np.max(
        np.abs(
            risk_original
            - risk_low
        )
    )
)

max_diff_high = float(
    np.max(
        np.abs(
            risk_original
            - risk_high
        )
    )
)

print("\n" + "=" * 80)
print("PREDICTION INVARIANCE TEST")
print("=" * 80)

print(
    "Max prediction difference "
    "after setting feature to -999:",
    max_diff_low
)

print(
    "Max prediction difference "
    "after setting feature to +999:",
    max_diff_high
)


# ============================================================
# 9. FINAL INTERPRETATION
# ============================================================

print("\n" + "=" * 80)
print("FINAL AUDIT RESULT")
print("=" * 80)

if (
    n_suspect_splits == 0
    and
    max_diff_low == 0.0
    and
    max_diff_high == 0.0
):

    print(
        "PASS: ENSG00000145107 is NOT used "
        "by the reconstructed final M3 model."
    )

    print(
        "Its external absence does not affect "
        "M3 predictions."
    )

else:

    print(
        "IMPORTANT: ENSG00000145107 IS used "
        "by the final M3 model."
    )

    print(
        "Exact M3 external prediction cannot "
        "simply ignore this feature."
    )

print("\nSTOP HERE.")

### AVAILABILITY-ADAPTED M3 — INTERNAL IMPACT AUDIT

In [ ]:
# ============================================================
# AVAILABILITY-ADAPTED M3 — INTERNAL IMPACT AUDIT
#
# Remove ONLY the externally unavailable frozen feature:
# ENSG00000145107
#
# NO re-ranking.
# NO replacement gene.
# NO hyperparameter tuning.
# NO CGGA outcomes.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored


# ============================================================
# 0. PATHS / CONSTANTS
# ============================================================

DOWNLOADS = BASE

STAGE_D = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

MODEL_DIR = (
    STAGE_D
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)

SPLIT_FILE = (
    STAGE_D
    / "STAGE_D_LOCKED_SPLIT.csv"
)

SURVIVAL_FILE = (
    STAGE_D
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

REWIRE_FILE = (
    STAGE_D
    / "STAGE_D_M3_REWIRING_RAW.csv"
)

FEATURE_FILE = (
    MODEL_DIR
    / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv"
)

MISSING_GENE = "ENSG00000145107"


def checkpoint(cond, msg):
    if cond:
        print(f"[CHECKPOINT PASS] {msg}")
    else:
        print(f"[CHECKPOINT FAIL] {msg}")
        raise AssertionError(msg)


def make_y(df):
    return np.array(
        [
            (bool(e), float(t))
            for e, t in zip(
                df["OS_event"],
                df["OS_months"]
            )
        ],
        dtype=[
            ("event", "?"),
            ("time", "<f8")
        ]
    )


print("=" * 80)
print("AVAILABILITY-ADAPTED M3 — INTERNAL IMPACT AUDIT")
print("=" * 80)


# ============================================================
# 1. LOAD
# ============================================================

split_df = pd.read_csv(SPLIT_FILE)
survival = pd.read_csv(SURVIVAL_FILE).set_index("patient_id")
rewiring = pd.read_csv(REWIRE_FILE, index_col=0)
selected = pd.read_csv(FEATURE_FILE)

split_df["patient_id"] = split_df["patient_id"].astype(str)
survival.index = survival.index.astype(str)
rewiring.index = rewiring.index.astype(str)

development_ids = (
    split_df.loc[
        split_df["split"] == "development",
        "patient_id"
    ].tolist()
)

test_ids = (
    split_df.loc[
        split_df["split"] == "locked_test",
        "patient_id"
    ].tolist()
)

checkpoint(
    len(development_ids) == 115,
    "115 development patients"
)

checkpoint(
    len(test_ids) == 29,
    "29 internal-test patients"
)


# ============================================================
# 2. EXACT FROZEN M3 FEATURE LIST
# ============================================================

m3_50 = (
    selected.loc[
        selected["model"] == "M3"
    ]
    .sort_values("rank")["feature"]
    .astype(str)
    .str.replace(
        "REWIRE::",
        "",
        regex=False
    )
    .tolist()
)

checkpoint(
    len(m3_50) == 50,
    "original M3 has 50 frozen features"
)

checkpoint(
    MISSING_GENE in m3_50,
    "unavailable CGGA gene is in original M3"
)

m3_49 = [
    g for g in m3_50
    if g != MISSING_GENE
]

checkpoint(
    len(m3_49) == 49,
    "availability-adapted M3 contains exactly 49 features"
)

# IMPORTANT:
# No feature is promoted from rank 51.
# We simply remove the unavailable frozen feature.


# ============================================================
# 3. DATA MATRICES
# ============================================================

X_dev_50 = (
    rewiring.loc[
        development_ids,
        m3_50
    ]
    .to_numpy(dtype=np.float64)
)

X_test_50 = (
    rewiring.loc[
        test_ids,
        m3_50
    ]
    .to_numpy(dtype=np.float64)
)

X_dev_49 = (
    rewiring.loc[
        development_ids,
        m3_49
    ]
    .to_numpy(dtype=np.float64)
)

X_test_49 = (
    rewiring.loc[
        test_ids,
        m3_49
    ]
    .to_numpy(dtype=np.float64)
)

y_dev = make_y(
    survival.loc[development_ids]
)

y_test = make_y(
    survival.loc[test_ids]
)

checkpoint(
    X_dev_50.shape == (115, 50),
    "original M3 development matrix = 115 × 50"
)

checkpoint(
    X_dev_49.shape == (115, 49),
    "adapted M3 development matrix = 115 × 49"
)


# ============================================================
# 4. ORIGINAL EXACT 50-FEATURE MODEL
# ============================================================

original_m3 = GradientBoostingSurvivalAnalysis(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

original_m3.fit(
    X_dev_50,
    y_dev
)

risk_50 = original_m3.predict(
    X_test_50
)

c50 = float(
    concordance_index_censored(
        y_test["event"],
        y_test["time"],
        risk_50
    )[0]
)

checkpoint(
    abs(c50 - 0.640127388535) < 1e-10,
    "original 50-feature M3 reproduces frozen internal C-index"
)


# ============================================================
# 5. AVAILABILITY-ADAPTED 49-FEATURE MODEL
# ============================================================

adapted_m3 = GradientBoostingSurvivalAnalysis(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

adapted_m3.fit(
    X_dev_49,
    y_dev
)

risk_49 = adapted_m3.predict(
    X_test_49
)

c49 = float(
    concordance_index_censored(
        y_test["event"],
        y_test["time"],
        risk_49
    )[0]
)


# ============================================================
# 6. PREDICTION AGREEMENT
# ============================================================

pearson_r = float(
    pearsonr(
        risk_50,
        risk_49
    ).statistic
)

spearman_r = float(
    spearmanr(
        risk_50,
        risk_49
    ).statistic
)

mean_abs_diff = float(
    np.mean(
        np.abs(
            risk_50 - risk_49
        )
    )
)

max_abs_diff = float(
    np.max(
        np.abs(
            risk_50 - risk_49
        )
    )
)


# ============================================================
# 7. OUTPUT
# ============================================================

print("\n" + "=" * 80)
print("INTERNAL PERFORMANCE COMPARISON")
print("=" * 80)

print(
    "Original frozen M3 (50 features):",
    f"{c50:.12f}"
)

print(
    "Availability-adapted M3 (49 features):",
    f"{c49:.12f}"
)

print(
    "C-index difference (49 - 50):",
    f"{c49 - c50:+.12f}"
)


print("\n" + "=" * 80)
print("PREDICTION AGREEMENT")
print("=" * 80)

print(
    "Pearson correlation:",
    f"{pearson_r:.12f}"
)

print(
    "Spearman correlation:",
    f"{spearman_r:.12f}"
)

print(
    "Mean absolute risk difference:",
    f"{mean_abs_diff:.12f}"
)

print(
    "Maximum absolute risk difference:",
    f"{max_abs_diff:.12f}"
)


# ============================================================
# 8. SAVE EXACT ADAPTED FEATURE LIST
# ============================================================

pd.DataFrame({
    "rank_original": [
        m3_50.index(g) + 1
        for g in m3_49
    ],
    "feature": m3_49
}).to_csv(
    MODEL_DIR
    / "M3_EXTERNAL_AVAILABILITY_ADAPTED_49_FEATURES.csv",
    index=False
)

print("\n" + "=" * 80)
print("ADAPTATION RULE")
print("=" * 80)

print(
    "Removed feature:",
    MISSING_GENE
)

print(
    "Replacement feature:",
    "NONE"
)

print(
    "Feature re-ranking:",
    "NO"
)

print(
    "Hyperparameter re-tuning:",
    "NO"
)

print(
    "CGGA survival used:",
    "NO"
)

print("\nSTOP HERE.")

### CGGA EXTERNAL TRANSPORT EVALUATION

In [ ]:
# ============================================================
# CGGA EXTERNAL TRANSPORT EVALUATION
# AVAILABILITY-ADAPTED M3 (49/50 frozen rewiring features)
#
# NO CGGA tuning
# NO CGGA feature ranking
# NO replacement feature
# NO hyperparameter modification
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored


# ============================================================
# 0. PATHS
# ============================================================

DOWNLOADS = BASE

STAGE_D = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

MODEL_DIR = (
    STAGE_D
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)

OUT_DIR = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
)

TCGA_SPLIT_FILE = (
    STAGE_D
    / "STAGE_D_LOCKED_SPLIT.csv"
)

TCGA_SURVIVAL_FILE = (
    STAGE_D
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

TCGA_REWIRE_FILE = (
    STAGE_D
    / "STAGE_D_M3_REWIRING_RAW.csv"
)

ADAPTED_FEATURE_FILE = (
    MODEL_DIR
    / "M3_EXTERNAL_AVAILABILITY_ADAPTED_49_FEATURES.csv"
)

CGGA_ENESH_FILE = (
    OUT_DIR
    / "CGGA_104_EXTERNAL_ENESH_3944.csv"
)

CGGA_CLINICAL_FILE = (
    DOWNLOADS
    / "CGGA.mRNAseq_693_clinical.20200506.txt"
    / "CGGA.mRNAseq_693_clinical.20200506.txt"
)

# If clinical file is directly in Downloads instead of folder
if not CGGA_CLINICAL_FILE.exists():

    alt = (
        DOWNLOADS
        / "CGGA.mRNAseq_693_clinical.20200506.txt"
    )

    if alt.is_file():
        CGGA_CLINICAL_FILE = alt

    else:
        candidates = [
            p for p in DOWNLOADS.rglob(
                "CGGA.mRNAseq_693_clinical.20200506.txt"
            )
            if p.is_file()
        ]

        if len(candidates) == 0:
            raise FileNotFoundError(
                "CGGA clinical file not found."
            )

        CGGA_CLINICAL_FILE = candidates[0]


def checkpoint(cond, msg):
    if cond:
        print(f"[CHECKPOINT PASS] {msg}")
    else:
        print(f"[CHECKPOINT FAIL] {msg}")
        raise AssertionError(msg)


def make_y(df):
    return np.array(
        [
            (bool(e), float(t))
            for e, t in zip(
                df["OS_event"],
                df["OS_time"]
            )
        ],
        dtype=[
            ("event", "?"),
            ("time", "<f8")
        ]
    )


print("=" * 80)
print("CGGA EXTERNAL — AVAILABILITY-ADAPTED M3")
print("=" * 80)


# ============================================================
# 1. LOAD TCGA DEVELOPMENT DATA
# ============================================================

split_df = pd.read_csv(
    TCGA_SPLIT_FILE
)

tcga_survival = (
    pd.read_csv(
        TCGA_SURVIVAL_FILE
    )
    .set_index("patient_id")
)

tcga_rewire = pd.read_csv(
    TCGA_REWIRE_FILE,
    index_col=0
)

adapted_features = pd.read_csv(
    ADAPTED_FEATURE_FILE
)["feature"].astype(str).tolist()

split_df["patient_id"] = (
    split_df["patient_id"]
    .astype(str)
)

tcga_survival.index = (
    tcga_survival.index
    .astype(str)
)

tcga_rewire.index = (
    tcga_rewire.index
    .astype(str)
)

development_ids = (
    split_df.loc[
        split_df["split"] == "development",
        "patient_id"
    ]
    .tolist()
)

checkpoint(
    len(development_ids) == 115,
    "TCGA development cohort = 115"
)

checkpoint(
    len(adapted_features) == 49,
    "availability-adapted feature set = 49"
)

checkpoint(
    set(adapted_features).issubset(
        set(tcga_rewire.columns.astype(str))
    ),
    "all 49 features exist in TCGA"
)


# ============================================================
# 2. TRAIN FIXED 49-FEATURE MODEL ON TCGA DEVELOPMENT
# ============================================================

X_train = (
    tcga_rewire.loc[
        development_ids,
        adapted_features
    ]
    .to_numpy(dtype=np.float64)
)

y_train = np.array(
    [
        (bool(e), float(t))
        for e, t in zip(
            tcga_survival.loc[
                development_ids,
                "OS_event"
            ],
            tcga_survival.loc[
                development_ids,
                "OS_months"
            ]
        )
    ],
    dtype=[
        ("event", "?"),
        ("time", "<f8")
    ]
)

model = GradientBoostingSurvivalAnalysis(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print(
    "\nFixed model fitted on "
    "115 TCGA development patients."
)


# ============================================================
# 3. LOAD CGGA EXTERNAL eNESH
# ============================================================

cgga_enesh = pd.read_csv(
    CGGA_ENESH_FILE,
    index_col=0
)

cgga_enesh.index = (
    cgga_enesh.index
    .astype(str)
)

checkpoint(
    cgga_enesh.shape == (104, 3944),
    "CGGA external eNESH = 104 × 3944"
)

checkpoint(
    set(adapted_features).issubset(
        set(cgga_enesh.columns.astype(str))
    ),
    "all 49 adapted M3 features exist in CGGA"
)


# ============================================================
# 4. RECONSTRUCT EXACT CGGA ELIGIBLE CLINICAL COHORT
# ============================================================

clinical = pd.read_csv(
    CGGA_CLINICAL_FILE,
    sep="\t",
    low_memory=False
)

print(
    "\nCGGA clinical source:",
    CGGA_CLINICAL_FILE
)

# numeric conversion
clinical["Age"] = pd.to_numeric(
    clinical["Age"],
    errors="coerce"
)

clinical["OS"] = pd.to_numeric(
    clinical["OS"],
    errors="coerce"
)

EVENT_COL = (
    "Censor (alive=0; dead=1)"
)

clinical[EVENT_COL] = pd.to_numeric(
    clinical[EVENT_COL],
    errors="coerce"
)

eligible = clinical.loc[
    (clinical["PRS_type"] == "Primary")
    &
    (clinical["Histology"] == "GBM")
    &
    (clinical["Grade"] == "WHO IV")
    &
    (
        clinical[
            "IDH_mutation_status"
        ] == "Wildtype"
    )
    &
    (clinical["Age"] >= 18)
    &
    (clinical["OS"] > 0)
    &
    (
        clinical[EVENT_COL]
        .isin([0, 1])
    )
].copy()

eligible["CGGA_ID"] = (
    eligible["CGGA_ID"]
    .astype(str)
)

print(
    "\nEligible clinical patients:",
    len(eligible)
)

print(
    "Events:",
    int(eligible[EVENT_COL].sum())
)

print(
    "Censored:",
    int(
        len(eligible)
        - eligible[EVENT_COL].sum()
    )
)

checkpoint(
    len(eligible) == 104,
    "CGGA eligible cohort = 104"
)

checkpoint(
    int(eligible[EVENT_COL].sum()) == 92,
    "CGGA deaths/events = 92"
)

checkpoint(
    set(eligible["CGGA_ID"])
    ==
    set(cgga_enesh.index),
    "clinical 104 IDs exactly match eNESH 104 IDs"
)


# ============================================================
# 5. ALIGN CGGA TO eNESH PATIENT ORDER
# ============================================================

eligible = (
    eligible
    .set_index("CGGA_ID")
    .loc[cgga_enesh.index]
)

X_external = (
    cgga_enesh.loc[
        :,
        adapted_features
    ]
    .to_numpy(dtype=np.float64)
)

checkpoint(
    X_external.shape == (104, 49),
    "external model matrix = 104 × 49"
)

checkpoint(
    np.isfinite(X_external).all(),
    "all external M3 inputs finite"
)

external_event = (
    eligible[EVENT_COL]
    .to_numpy(dtype=bool)
)

external_time = (
    eligible["OS"]
    .to_numpy(dtype=float)
)


# ============================================================
# 6. EXTERNAL RISK PREDICTIONS
# ============================================================

external_risk = model.predict(
    X_external
)

checkpoint(
    np.isfinite(external_risk).all(),
    "all external risk predictions finite"
)

external_cindex = float(
    concordance_index_censored(
        external_event,
        external_time,
        external_risk
    )[0]
)


# ============================================================
# 7. PATIENT-LEVEL BOOTSTRAP 95% CI
# ============================================================

N_BOOT = 2000
RNG = np.random.default_rng(42)

bootstrap_c = []

n = len(external_risk)

for b in range(N_BOOT):

    idx = RNG.integers(
        0,
        n,
        size=n
    )

    try:

        c = float(
            concordance_index_censored(
                external_event[idx],
                external_time[idx],
                external_risk[idx]
            )[0]
        )

        if np.isfinite(c):
            bootstrap_c.append(c)

    except Exception:
        pass

bootstrap_c = np.asarray(
    bootstrap_c,
    dtype=float
)

checkpoint(
    len(bootstrap_c) >= 1900,
    "sufficient valid bootstrap replicates"
)

ci_low = float(
    np.percentile(
        bootstrap_c,
        2.5
    )
)

ci_high = float(
    np.percentile(
        bootstrap_c,
        97.5
    )
)


# ============================================================
# 8. RESULTS
# ============================================================

print("\n" + "=" * 80)
print("EXTERNAL TRANSPORT RESULT")
print("=" * 80)

print(
    "Patients:",
    n
)

print(
    "Events:",
    int(
        external_event.sum()
    )
)

print(
    "Censored:",
    int(
        n - external_event.sum()
    )
)

print(
    "\nAvailability-adapted M3 "
    "external Harrell C-index:",
    f"{external_cindex:.6f}"
)

print(
    "Bootstrap 95% CI:",
    f"[{ci_low:.6f}, {ci_high:.6f}]"
)

print(
    "Valid bootstrap replicates:",
    len(bootstrap_c)
)


# ============================================================
# 9. SAVE PATIENT-LEVEL EXTERNAL PREDICTIONS
# ============================================================

prediction_table = pd.DataFrame({
    "CGGA_ID":
        cgga_enesh.index,

    "OS_days":
        external_time,

    "event":
        external_event.astype(int),

    "M3_availability_adapted_risk":
        external_risk
})

prediction_table.to_csv(
    OUT_DIR
    / "CGGA_104_M3_AVAILABILITY_ADAPTED_PREDICTIONS.csv",
    index=False
)

pd.DataFrame({
    "metric": [
        "n_patients",
        "n_events",
        "harrell_cindex",
        "bootstrap_ci_low",
        "bootstrap_ci_high",
        "bootstrap_valid_replicates",
        "n_frozen_features_original",
        "n_external_features_used",
        "missing_feature"
    ],

    "value": [
        n,
        int(external_event.sum()),
        external_cindex,
        ci_low,
        ci_high,
        len(bootstrap_c),
        50,
        49,
        "ENSG00000145107"
    ]
}).to_csv(
    OUT_DIR
    / "CGGA_M3_AVAILABILITY_ADAPTED_EXTERNAL_RESULT.csv",
    index=False
)


print("\n" + "=" * 80)
print("FREEZE NOTE")
print("=" * 80)

print(
    "Original frozen M3 features: 50"
)

print(
    "Externally usable frozen features: 49"
)

print(
    "Unavailable feature:",
    "ENSG00000145107 / TM4SF19"
)

print(
    "Replacement feature: NONE"
)

print(
    "CGGA feature ranking: NO"
)

print(
    "CGGA tuning: NO"
)

print(
    "CGGA outcomes used before final scoring: NO"
)

print("\nSTOP HERE.")

In [ ]:
from pathlib import Path
import pandas as pd

DOWNLOADS = BASE

MODEL_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)

FEATURE_FILE = (
    MODEL_DIR
    / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv"
)

CGGA_EXPR_FILE = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
    / "CGGA_104_HARMONIZED_TO_GTEX.csv"
)

features = pd.read_csv(FEATURE_FILE)

expr = pd.read_csv(
    CGGA_EXPR_FILE,
    index_col=0
)

# CGGA expression matrix = genes × patients
cgga_genes = set(
    expr.index.astype(str)
)

m2 = (
    features.loc[
        features["model"] == "M2",
        ["rank", "feature"]
    ]
    .sort_values("rank")
    .copy()
)

m2["gene"] = (
    m2["feature"]
    .astype(str)
    .str.replace(
        "EXPR::",
        "",
        regex=False
    )
)

m2["available"] = (
    m2["gene"]
    .isin(cgga_genes)
)

print("=" * 80)
print("M2 EXTERNAL FEATURE AVAILABILITY")
print("=" * 80)

print(
    "Frozen M2 features:",
    len(m2)
)

print(
    "Available in CGGA:",
    int(m2["available"].sum())
)

print(
    "Missing:",
    int((~m2["available"]).sum())
)

print("\nMissing features:")
print(
    m2.loc[
        ~m2["available"],
        ["rank", "gene"]
    ].to_string(index=False)
)

print("\nFull audit:")
print(
    m2[
        ["rank", "gene", "available"]
    ].to_string(index=False)
)

print("\nSTOP HERE.")

### EXACT FROZEN M2 — RECONSTRUCTION + CGGA EXTERNAL EVALUATION

In [ ]:
# ============================================================
# EXACT FROZEN M2 — RECONSTRUCTION + CGGA EXTERNAL EVALUATION
#
# Frozen:
#   top 25 expression features
#   RSF 300 trees
#   min_samples_leaf = 5
#   min_samples_split = 10
#   max_features = "sqrt"
#   final prediction = mean across seeds 42,43,44,45,46
#
# NO CGGA feature selection
# NO CGGA tuning
# NO replacement features
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored


# ============================================================
# 0. PATHS
# ============================================================

DOWNLOADS = BASE

STAGE_D = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

MODEL_DIR = (
    STAGE_D
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)

CGGA_DIR = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
)

SPLIT_FILE = (
    STAGE_D
    / "STAGE_D_LOCKED_SPLIT.csv"
)

SURVIVAL_FILE = (
    STAGE_D
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

TCGA_EXPR_FILE = (
    STAGE_D
    / "STAGE_D_M2_EXPRESSION_RAW.csv"
)

FEATURE_FILE = (
    MODEL_DIR
    / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv"
)

CGGA_EXPR_FILE = (
    CGGA_DIR
    / "CGGA_104_HARMONIZED_TO_GTEX.csv"
)


# ------------------------------------------------------------
# locate clinical file automatically
# ------------------------------------------------------------

clinical_candidates = [
    p for p in DOWNLOADS.rglob(
        "CGGA.mRNAseq_693_clinical.20200506.txt"
    )
    if p.is_file()
]

if len(clinical_candidates) == 0:
    raise FileNotFoundError(
        "CGGA clinical file not found."
    )

CGGA_CLINICAL_FILE = clinical_candidates[0]


# ============================================================
# HELPERS
# ============================================================

def checkpoint(cond, msg):
    if cond:
        print(f"[CHECKPOINT PASS] {msg}")
    else:
        print(f"[CHECKPOINT FAIL] {msg}")
        raise AssertionError(msg)


def make_y(event, time):
    return np.array(
        [
            (bool(e), float(t))
            for e, t in zip(event, time)
        ],
        dtype=[
            ("event", "?"),
            ("time", "<f8")
        ]
    )


def fit_rsf_ensemble(X_train, y_train, X_target):

    predictions = []

    for seed in range(42, 47):

        model = RandomSurvivalForest(
            n_estimators=300,
            min_samples_leaf=5,
            min_samples_split=10,
            max_features="sqrt",
            random_state=seed,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        predictions.append(
            model.predict(
                X_target
            )
        )

    return np.mean(
        np.vstack(predictions),
        axis=0
    )


print("=" * 80)
print("EXACT FROZEN M2 — TCGA RECONSTRUCTION + CGGA EXTERNAL")
print("=" * 80)


# ============================================================
# 1. LOAD TCGA DATA
# ============================================================

split_df = pd.read_csv(
    SPLIT_FILE
)

survival = (
    pd.read_csv(
        SURVIVAL_FILE
    )
    .set_index("patient_id")
)

tcga_expr = pd.read_csv(
    TCGA_EXPR_FILE,
    index_col=0
)

features = pd.read_csv(
    FEATURE_FILE
)

split_df["patient_id"] = (
    split_df["patient_id"]
    .astype(str)
)

survival.index = (
    survival.index
    .astype(str)
)

tcga_expr.index = (
    tcga_expr.index
    .astype(str)
)


development_ids = (
    split_df.loc[
        split_df["split"] == "development",
        "patient_id"
    ]
    .tolist()
)

test_ids = (
    split_df.loc[
        split_df["split"] == "locked_test",
        "patient_id"
    ]
    .tolist()
)

checkpoint(
    len(development_ids) == 115,
    "115 TCGA development patients"
)

checkpoint(
    len(test_ids) == 29,
    "29 TCGA internal-test patients"
)


# ============================================================
# 2. EXACT FROZEN M2 FEATURES
# ============================================================

m2_features = (
    features.loc[
        features["model"] == "M2"
    ]
    .sort_values("rank")["feature"]
    .astype(str)
    .str.replace(
        "EXPR::",
        "",
        regex=False
    )
    .tolist()
)

checkpoint(
    len(m2_features) == 25,
    "M2 has exactly 25 frozen expression features"
)

checkpoint(
    set(m2_features).issubset(
        set(tcga_expr.columns.astype(str))
    ),
    "all 25 M2 features present in TCGA expression matrix"
)


# ============================================================
# 3. TCGA TRAIN / INTERNAL TEST
# ============================================================

X_dev = (
    tcga_expr.loc[
        development_ids,
        m2_features
    ]
    .to_numpy(dtype=np.float64)
)

X_test = (
    tcga_expr.loc[
        test_ids,
        m2_features
    ]
    .to_numpy(dtype=np.float64)
)

y_dev = make_y(
    survival.loc[
        development_ids,
        "OS_event"
    ],
    survival.loc[
        development_ids,
        "OS_months"
    ]
)

y_test = make_y(
    survival.loc[
        test_ids,
        "OS_event"
    ],
    survival.loc[
        test_ids,
        "OS_months"
    ]
)

checkpoint(
    X_dev.shape == (115, 25),
    "TCGA M2 development matrix = 115 × 25"
)

checkpoint(
    X_test.shape == (29, 25),
    "TCGA M2 internal-test matrix = 29 × 25"
)

checkpoint(
    np.isfinite(X_dev).all(),
    "TCGA development expression finite"
)

checkpoint(
    np.isfinite(X_test).all(),
    "TCGA internal expression finite"
)


# ============================================================
# 4. RECONSTRUCT FROZEN M2 INTERNAL RESULT
# ============================================================

internal_risk = fit_rsf_ensemble(
    X_dev,
    y_dev,
    X_test
)

internal_c = float(
    concordance_index_censored(
        y_test["event"],
        y_test["time"],
        internal_risk
    )[0]
)

print("\n" + "=" * 80)
print("M2 RECONSTRUCTION CHECK")
print("=" * 80)

print(
    "Reconstructed internal C-index:",
    f"{internal_c:.12f}"
)

print(
    "Saved V4 internal C-index:",
    "0.643312"
)

print(
    "Absolute difference:",
    f"{abs(internal_c - 0.643312):.12f}"
)

checkpoint(
    abs(internal_c - 0.643312) < 1e-5,
    "frozen M2 internal performance reproduced"
)


# ============================================================
# 5. LOAD CGGA HARMONIZED EXPRESSION
#
# Matrix orientation:
# genes × patients
# ============================================================

cgga_expr = pd.read_csv(
    CGGA_EXPR_FILE,
    index_col=0
)

cgga_expr.index = (
    cgga_expr.index
    .astype(str)
)

cgga_expr.columns = (
    cgga_expr.columns
    .astype(str)
)

checkpoint(
    len(cgga_expr.columns) == 104,
    "CGGA harmonized expression contains 104 patients"
)

checkpoint(
    set(m2_features).issubset(
        set(cgga_expr.index)
    ),
    "all 25 frozen M2 genes available in CGGA"
)


# ============================================================
# 6. CGGA CLINICAL COHORT
# ============================================================

clinical = pd.read_csv(
    CGGA_CLINICAL_FILE,
    sep="\t",
    low_memory=False
)

clinical["Age"] = pd.to_numeric(
    clinical["Age"],
    errors="coerce"
)

clinical["OS"] = pd.to_numeric(
    clinical["OS"],
    errors="coerce"
)

EVENT_COL = (
    "Censor (alive=0; dead=1)"
)

clinical[EVENT_COL] = pd.to_numeric(
    clinical[EVENT_COL],
    errors="coerce"
)

eligible = clinical.loc[
    (clinical["PRS_type"] == "Primary")
    &
    (clinical["Histology"] == "GBM")
    &
    (clinical["Grade"] == "WHO IV")
    &
    (
        clinical["IDH_mutation_status"]
        == "Wildtype"
    )
    &
    (clinical["Age"] >= 18)
    &
    (clinical["OS"] > 0)
    &
    (
        clinical[EVENT_COL]
        .isin([0, 1])
    )
].copy()

eligible["CGGA_ID"] = (
    eligible["CGGA_ID"]
    .astype(str)
)

checkpoint(
    len(eligible) == 104,
    "CGGA external cohort = 104"
)

checkpoint(
    int(eligible[EVENT_COL].sum()) == 92,
    "CGGA events = 92"
)

checkpoint(
    set(eligible["CGGA_ID"])
    ==
    set(cgga_expr.columns),
    "CGGA clinical IDs exactly match expression IDs"
)


# ============================================================
# 7. ALIGN CGGA
# ============================================================

patient_order = (
    eligible["CGGA_ID"]
    .tolist()
)

eligible = (
    eligible
    .set_index("CGGA_ID")
    .loc[patient_order]
)

X_external = (
    cgga_expr.loc[
        m2_features,
        patient_order
    ]
    .T
    .to_numpy(dtype=np.float64)
)

checkpoint(
    X_external.shape == (104, 25),
    "CGGA M2 external matrix = 104 × 25"
)

checkpoint(
    np.isfinite(X_external).all(),
    "all CGGA M2 values finite"
)

external_event = (
    eligible[EVENT_COL]
    .to_numpy(dtype=bool)
)

external_time = (
    eligible["OS"]
    .to_numpy(dtype=float)
)


# ============================================================
# 8. EXACT FROZEN M2 EXTERNAL PREDICTION
# ============================================================

external_risk = fit_rsf_ensemble(
    X_dev,
    y_dev,
    X_external
)

checkpoint(
    np.isfinite(external_risk).all(),
    "all M2 external risk predictions finite"
)

external_c = float(
    concordance_index_censored(
        external_event,
        external_time,
        external_risk
    )[0]
)


# ============================================================
# 9. PATIENT-LEVEL BOOTSTRAP 95% CI
# ============================================================

N_BOOT = 2000
rng = np.random.default_rng(42)

bootstrap_c = []

n = len(external_risk)

for _ in range(N_BOOT):

    idx = rng.integers(
        0,
        n,
        size=n
    )

    try:

        c = float(
            concordance_index_censored(
                external_event[idx],
                external_time[idx],
                external_risk[idx]
            )[0]
        )

        if np.isfinite(c):
            bootstrap_c.append(c)

    except Exception:
        pass


bootstrap_c = np.asarray(
    bootstrap_c,
    dtype=float
)

checkpoint(
    len(bootstrap_c) >= 1900,
    "sufficient valid bootstrap replicates"
)

ci_low = float(
    np.percentile(
        bootstrap_c,
        2.5
    )
)

ci_high = float(
    np.percentile(
        bootstrap_c,
        97.5
    )
)


# ============================================================
# 10. RESULTS
# ============================================================

print("\n" + "=" * 80)
print("M2 EXACT EXTERNAL RESULT")
print("=" * 80)

print(
    "Patients:",
    n
)

print(
    "Events:",
    int(external_event.sum())
)

print(
    "Censored:",
    int(n - external_event.sum())
)

print(
    "\nFrozen M2 external Harrell C-index:",
    f"{external_c:.6f}"
)

print(
    "Bootstrap 95% CI:",
    f"[{ci_low:.6f}, {ci_high:.6f}]"
)

print(
    "Valid bootstrap replicates:",
    len(bootstrap_c)
)


# ============================================================
# 11. SAVE
# ============================================================

pd.DataFrame({
    "CGGA_ID": patient_order,
    "OS_days": external_time,
    "event": external_event.astype(int),
    "M2_frozen_risk": external_risk
}).to_csv(
    CGGA_DIR
    / "CGGA_104_M2_EXACT_FROZEN_PREDICTIONS.csv",
    index=False
)

pd.DataFrame({
    "metric": [
        "n_patients",
        "n_events",
        "harrell_cindex",
        "bootstrap_ci_low",
        "bootstrap_ci_high",
        "bootstrap_valid_replicates",
        "n_frozen_features",
        "missing_features",
        "feature_replacement",
        "external_tuning"
    ],

    "value": [
        n,
        int(external_event.sum()),
        external_c,
        ci_low,
        ci_high,
        len(bootstrap_c),
        25,
        0,
        "NONE",
        "NO"
    ]
}).to_csv(
    CGGA_DIR
    / "CGGA_M2_EXACT_FROZEN_EXTERNAL_RESULT.csv",
    index=False
)


print("\n" + "=" * 80)
print("M2 FREEZE NOTE")
print("=" * 80)

print("Frozen features: 25 / 25 available")
print("Feature replacement: NONE")
print("Feature re-ranking in CGGA: NO")
print("Hyperparameter tuning in CGGA: NO")
print("RSF final seeds: 42, 43, 44, 45, 46")
print("CGGA outcomes used before scoring: NO")

print("\nSTOP HERE.")

### EXACT CORRECTED-V4 STACKING RECONSTRUCTION

In [ ]:
# ============================================================
# EXACT CORRECTED-V4 STACKING RECONSTRUCTION
# M4 / M5 / M6
#
# IMPORTANT FIX:
# - clinical preprocessing INSIDE each meta fold
# - molecular ranking INSIDE each meta fold
# - final target prediction uses full 115-development fit
# ============================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
META_FOLDS = 3


# ============================================================
# 0. PATHS
# ============================================================

DOWNLOADS = BASE

STAGE_D = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

MODEL_DIR = (
    STAGE_D
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)

SPLIT_FILE = (
    STAGE_D
    / "STAGE_D_LOCKED_SPLIT.csv"
)

SURVIVAL_FILE = (
    STAGE_D
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

CLIN_FILE = (
    STAGE_D
    / "STAGE_D_M1_CLINICAL_RAW.csv"
)

EXPR_FILE = (
    STAGE_D
    / "STAGE_D_M2_EXPRESSION_RAW.csv"
)

REWIRE_FILE = (
    STAGE_D
    / "STAGE_D_M3_REWIRING_RAW.csv"
)

FEATURE_FILE = (
    MODEL_DIR
    / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv"
)


# ============================================================
# 1. HELPERS
# ============================================================

def checkpoint(cond, msg):
    if cond:
        print(f"[CHECKPOINT PASS] {msg}")
    else:
        print(f"[CHECKPOINT FAIL] {msg}")
        raise AssertionError(msg)


def make_y(df):
    return np.array(
        [
            (bool(e), float(t))
            for e, t in zip(
                df["OS_event"],
                df["OS_months"]
            )
        ],
        dtype=[
            ("event", "?"),
            ("time", "<f8")
        ]
    )


def cindex(y, risk):
    return float(
        concordance_index_censored(
            y["event"],
            y["time"],
            risk
        )[0]
    )


def make_onehot():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )


def build_clinical_preprocessor(df):

    numeric_cols = [
        c for c in [
            "Diagnosis Age",
            "Karnofsky Performance Score"
        ]
        if c in df.columns
    ]

    categorical_cols = [
        c for c in [
            "Sex",
            "MGMT Status"
        ]
        if c in df.columns
    ]

    transformers = []

    if numeric_cols:
        transformers.append(
            (
                "numeric",
                SimpleImputer(
                    strategy="median"
                ),
                numeric_cols
            )
        )

    if categorical_cols:

        cat_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                make_onehot()
            )
        ])

        transformers.append(
            (
                "categorical",
                cat_pipe,
                categorical_cols
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


def fit_gbsa_predict(
    X_train,
    y_train,
    X_valid
):

    model = GradientBoostingSurvivalAnalysis(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    return model.predict(
        X_valid
    )


# ============================================================
# 2. EXACT MOLECULAR RANKING
# ============================================================

def rank_molecular_features(
    X_train,
    y_train
):

    feature_names = list(
        X_train.columns
    )

    values = X_train.to_numpy(
        dtype=np.float64
    )

    scores = np.full(
        len(feature_names),
        0.5,
        dtype=float
    )

    for j in range(
        values.shape[1]
    ):

        x = values[:, j]

        if np.nanstd(x) <= 1e-12:
            continue

        try:

            c = concordance_index_censored(
                y_train["event"],
                y_train["time"],
                x
            )[0]

            scores[j] = (
                0.5
                if np.isnan(c)
                else max(c, 1.0 - c)
            )

        except Exception:
            pass

    ranking = pd.DataFrame({
        "feature": feature_names,
        "univariate_c": scores
    })

    return (
        ranking
        .sort_values(
            ["univariate_c", "feature"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )


def select_molecular(
    molecular,
    train_ids,
    y_train,
    top_k
):

    ranking = rank_molecular_features(
        molecular.loc[
            train_ids
        ],
        y_train
    )

    return (
        ranking["feature"]
        .head(top_k)
        .tolist()
    )


# ============================================================
# 3. LOAD DATA
# ============================================================

split_df = pd.read_csv(
    SPLIT_FILE
)

survival = (
    pd.read_csv(
        SURVIVAL_FILE
    )
    .set_index("patient_id")
)

clinical = pd.read_csv(
    CLIN_FILE,
    index_col=0
)

expression = pd.read_csv(
    EXPR_FILE,
    index_col=0
)

rewiring = pd.read_csv(
    REWIRE_FILE,
    index_col=0
)

selected_saved = pd.read_csv(
    FEATURE_FILE
)


split_df["patient_id"] = (
    split_df["patient_id"]
    .astype(str)
)

for df in [
    survival,
    clinical,
    expression,
    rewiring
]:
    df.index = df.index.astype(str)


development_ids = (
    split_df.loc[
        split_df["split"]
        == "development",
        "patient_id"
    ]
    .tolist()
)

test_ids = (
    split_df.loc[
        split_df["split"]
        == "locked_test",
        "patient_id"
    ]
    .tolist()
)


checkpoint(
    len(development_ids) == 115,
    "115 development patients"
)

checkpoint(
    len(test_ids) == 29,
    "29 internal-test patients"
)


y_dev = make_y(
    survival.loc[
        development_ids
    ]
)

y_test = make_y(
    survival.loc[
        test_ids
    ]
)


# ============================================================
# 4. CLINICAL BLOCK
#
# PREPROCESSING IS REFIT INSIDE EACH META FOLD
# ============================================================

def clinical_oof_and_test():

    n = len(
        development_ids
    )

    oof = np.full(
        n,
        np.nan
    )

    meta_cv = StratifiedKFold(
        n_splits=META_FOLDS,
        shuffle=True,
        random_state=42
    )

    events = (
        y_dev["event"]
        .astype(int)
    )

    for train_idx, hold_idx in (
        meta_cv.split(
            np.zeros(n),
            events
        )
    ):

        meta_train_ids = [
            development_ids[i]
            for i in train_idx
        ]

        meta_hold_ids = [
            development_ids[i]
            for i in hold_idx
        ]

        y_meta_train = (
            make_y(
                survival.loc[
                    meta_train_ids
                ]
            )
        )

        processor = (
            build_clinical_preprocessor(
                clinical.loc[
                    meta_train_ids
                ]
            )
        )

        X_train = (
            processor
            .fit_transform(
                clinical.loc[
                    meta_train_ids
                ]
            )
        )

        X_hold = (
            processor
            .transform(
                clinical.loc[
                    meta_hold_ids
                ]
            )
        )

        oof[hold_idx] = (
            fit_gbsa_predict(
                X_train,
                y_meta_train,
                X_hold
            )
        )

    checkpoint(
        np.isfinite(oof).all(),
        "clinical OOF risks finite"
    )


    # Full development -> internal test
    processor = (
        build_clinical_preprocessor(
            clinical.loc[
                development_ids
            ]
        )
    )

    X_dev = (
        processor.fit_transform(
            clinical.loc[
                development_ids
            ]
        )
    )

    X_test = (
        processor.transform(
            clinical.loc[
                test_ids
            ]
        )
    )

    test_risk = (
        fit_gbsa_predict(
            X_dev,
            y_dev,
            X_test
        )
    )

    return (
        oof,
        test_risk
    )


# ============================================================
# 5. MOLECULAR BLOCK
#
# FEATURE RANKING IS REPEATED INSIDE EACH META FOLD
# ============================================================

def molecular_oof_and_test(
    molecular,
    top_k
):

    n = len(
        development_ids
    )

    oof = np.full(
        n,
        np.nan
    )

    meta_cv = StratifiedKFold(
        n_splits=META_FOLDS,
        shuffle=True,
        random_state=42
    )

    events = (
        y_dev["event"]
        .astype(int)
    )

    for train_idx, hold_idx in (
        meta_cv.split(
            np.zeros(n),
            events
        )
    ):

        meta_train_ids = [
            development_ids[i]
            for i in train_idx
        ]

        meta_hold_ids = [
            development_ids[i]
            for i in hold_idx
        ]

        y_meta_train = (
            make_y(
                survival.loc[
                    meta_train_ids
                ]
            )
        )

        # CRITICAL:
        # rank only on this meta-training subset
        fold_selected = (
            select_molecular(
                molecular,
                meta_train_ids,
                y_meta_train,
                top_k
            )
        )

        X_train = (
            molecular.loc[
                meta_train_ids,
                fold_selected
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        X_hold = (
            molecular.loc[
                meta_hold_ids,
                fold_selected
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        oof[hold_idx] = (
            fit_gbsa_predict(
                X_train,
                y_meta_train,
                X_hold
            )
        )

    checkpoint(
        np.isfinite(oof).all(),
        "molecular OOF risks finite"
    )


    # Full 115-development ranking
    final_selected = (
        select_molecular(
            molecular,
            development_ids,
            y_dev,
            top_k
        )
    )

    X_dev = (
        molecular.loc[
            development_ids,
            final_selected
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    X_test = (
        molecular.loc[
            test_ids,
            final_selected
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    test_risk = (
        fit_gbsa_predict(
            X_dev,
            y_dev,
            X_test
        )
    )

    return (
        oof,
        test_risk,
        final_selected
    )


# ============================================================
# 6. META COXNET
# ============================================================

def meta_predict(
    oof_blocks,
    test_blocks
):

    meta_X_train = (
        np.column_stack(
            oof_blocks
        )
    )

    meta_X_test = (
        np.column_stack(
            test_blocks
        )
    )

    model = CoxnetSurvivalAnalysis(
        l1_ratio=0.01,
        alpha_min_ratio=0.5
    )

    model.fit(
        meta_X_train,
        y_dev
    )

    return model.predict(
        meta_X_test
    )


# ============================================================
# 7. GENERATE CLINICAL OOF ONCE
# ============================================================

print(
    "\nGenerating clinical OOF..."
)

clin_oof, clin_test = (
    clinical_oof_and_test()
)

print(
    "Clinical OOF mean/sd:",
    float(np.mean(clin_oof)),
    float(np.std(clin_oof))
)

print(
    "Clinical TEST mean/sd:",
    float(np.mean(clin_test)),
    float(np.std(clin_test))
)


# ============================================================
# 8. M4
# ============================================================

print("\n" + "=" * 80)
print("M4")
print("=" * 80)

expr_oof, expr_test, m4_selected = (
    molecular_oof_and_test(
        expression,
        top_k=10
    )
)

saved_m4 = (
    selected_saved.loc[
        selected_saved["model"]
        == "M4"
    ]
    .sort_values("rank")[
        "feature"
    ]
    .astype(str)
    .str.replace(
        "EXPR::",
        "",
        regex=False
    )
    .tolist()
)

checkpoint(
    m4_selected == saved_m4,
    "M4 final selected features exactly reproduced"
)

risk_m4 = meta_predict(
    [
        clin_oof,
        expr_oof
    ],
    [
        clin_test,
        expr_test
    ]
)

c_m4 = cindex(
    y_test,
    risk_m4
)

print(
    "M4 reconstructed C-index:",
    f"{c_m4:.12f}"
)


# ============================================================
# 9. M5
# ============================================================

print("\n" + "=" * 80)
print("M5")
print("=" * 80)

rew_oof, rew_test, m5_selected = (
    molecular_oof_and_test(
        rewiring,
        top_k=10
    )
)

saved_m5 = (
    selected_saved.loc[
        selected_saved["model"]
        == "M5"
    ]
    .sort_values("rank")[
        "feature"
    ]
    .astype(str)
    .str.replace(
        "REWIRE::",
        "",
        regex=False
    )
    .tolist()
)

checkpoint(
    m5_selected == saved_m5,
    "M5 final selected features exactly reproduced"
)

risk_m5 = meta_predict(
    [
        clin_oof,
        rew_oof
    ],
    [
        clin_test,
        rew_test
    ]
)

c_m5 = cindex(
    y_test,
    risk_m5
)

print(
    "M5 reconstructed C-index:",
    f"{c_m5:.12f}"
)


# ============================================================
# 10. M6
# ============================================================

print("\n" + "=" * 80)
print("M6")
print("=" * 80)

# Same expression/rewiring OOF generation,
# because K=10 for each block.

m6_expr_oof, m6_expr_test, m6_expr_selected = (
    molecular_oof_and_test(
        expression,
        top_k=10
    )
)

m6_rew_oof, m6_rew_test, m6_rew_selected = (
    molecular_oof_and_test(
        rewiring,
        top_k=10
    )
)

saved_m6 = (
    selected_saved.loc[
        selected_saved["model"]
        == "M6"
    ]
    .sort_values("rank")[
        "feature"
    ]
    .astype(str)
    .tolist()
)

saved_m6_expr = [
    x.replace(
        "EXPR::",
        ""
    )
    for x in saved_m6
    if x.startswith(
        "EXPR::"
    )
]

saved_m6_rew = [
    x.replace(
        "REWIRE::",
        ""
    )
    for x in saved_m6
    if x.startswith(
        "REWIRE::"
    )
]


checkpoint(
    m6_expr_selected
    == saved_m6_expr,
    "M6 expression final features exactly reproduced"
)

checkpoint(
    m6_rew_selected
    == saved_m6_rew,
    "M6 rewiring final features exactly reproduced"
)


risk_m6 = meta_predict(
    [
        clin_oof,
        m6_expr_oof,
        m6_rew_oof
    ],
    [
        clin_test,
        m6_expr_test,
        m6_rew_test
    ]
)

c_m6 = cindex(
    y_test,
    risk_m6
)

print(
    "M6 reconstructed C-index:",
    f"{c_m6:.12f}"
)


# ============================================================
# 11. FINAL CHECK
# ============================================================

print("\n" + "=" * 80)
print("CORRECTED V4 STACKING RECONSTRUCTION")
print("=" * 80)

print(
    f"M4: {c_m4:.12f} "
    "(saved = 0.500000)"
)

print(
    f"M5: {c_m5:.12f} "
    "(saved = 0.500000)"
)

print(
    f"M6: {c_m6:.12f} "
    "(saved = 0.500000)"
)


checkpoint(
    abs(c_m4 - 0.5) < 1e-10,
    "M4 reproduced"
)

checkpoint(
    abs(c_m5 - 0.5) < 1e-10,
    "M5 reproduced"
)

checkpoint(
    abs(c_m6 - 0.5) < 1e-10,
    "M6 reproduced"
)

print(
    "\nSUCCESS: exact corrected V4 stacking "
    "pipeline reconstructed."
)

print("\nSTOP HERE.")

### FINAL CGGA EXTERNAL TRANSPORT

In [ ]:
# ============================================================
# FINAL CGGA EXTERNAL TRANSPORT
# M1 + M4 + M5 + M6
#
# M1:
#   age + sex + MGMT
#   KPS unavailable in CGGA -> dropped
#
# M4:
#   adapted clinical + frozen 10 expression
#
# M5:
#   adapted clinical + frozen 9/10 available rewiring
#
# M6:
#   adapted clinical + frozen 10 expression
#   + frozen 9/10 available rewiring
#
# IMPORTANT:
# - no replacement feature
# - no CGGA feature ranking
# - no CGGA tuning
# - no CGGA model selection
# ============================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis
)

from sksurv.linear_model import (
    CoxnetSurvivalAnalysis
)

from sksurv.metrics import (
    concordance_index_censored
)

warnings.filterwarnings("ignore")


# ============================================================
# 0. CONSTANTS / PATHS
# ============================================================

RANDOM_SEED = 42
META_FOLDS = 3

MISSING_REWIRE = (
    "ENSG00000145107"
)

DOWNLOADS = (
    BASE
)

STAGE_D = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

MODEL_DIR = (
    STAGE_D
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)

CGGA_DIR = (
    DOWNLOADS
    / "CGGA_GBM_EXTERNAL_REWIRING"
)

SPLIT_FILE = (
    STAGE_D
    / "STAGE_D_LOCKED_SPLIT.csv"
)

SURVIVAL_FILE = (
    STAGE_D
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

CLIN_FILE = (
    STAGE_D
    / "STAGE_D_M1_CLINICAL_RAW.csv"
)

EXPR_FILE = (
    STAGE_D
    / "STAGE_D_M2_EXPRESSION_RAW.csv"
)

REWIRE_FILE = (
    STAGE_D
    / "STAGE_D_M3_REWIRING_RAW.csv"
)

FEATURE_FILE = (
    MODEL_DIR
    / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv"
)

CGGA_EXPR_FILE = (
    CGGA_DIR
    / "CGGA_104_HARMONIZED_TO_GTEX.csv"
)

CGGA_REWIRE_FILE = (
    CGGA_DIR
    / "CGGA_104_EXTERNAL_ENESH_3944.csv"
)


clinical_candidates = [
    p
    for p in DOWNLOADS.rglob(
        "CGGA.mRNAseq_693_clinical.20200506.txt"
    )
    if p.is_file()
]

if not clinical_candidates:
    raise FileNotFoundError(
        "CGGA clinical file not found."
    )

CGGA_CLIN_FILE = (
    clinical_candidates[0]
)


# ============================================================
# 1. HELPERS
# ============================================================

def checkpoint(cond, msg):

    if cond:
        print(
            f"[CHECKPOINT PASS] {msg}"
        )

    else:
        print(
            f"[CHECKPOINT FAIL] {msg}"
        )

        raise AssertionError(msg)


def make_y(df):

    return np.array(
        [
            (
                bool(e),
                float(t)
            )
            for e, t in zip(
                df["OS_event"],
                df["OS_months"]
            )
        ],
        dtype=[
            ("event", "?"),
            ("time", "<f8")
        ]
    )


def cindex(
    event,
    time,
    risk
):

    return float(
        concordance_index_censored(
            np.asarray(
                event,
                dtype=bool
            ),
            np.asarray(
                time,
                dtype=float
            ),
            np.asarray(
                risk,
                dtype=float
            )
        )[0]
    )


def make_onehot():

    try:

        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

    except TypeError:

        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )


def build_clinical_preprocessor(
    df
):

    numeric_cols = [
        c
        for c in [
            "Diagnosis Age",
            "Karnofsky Performance Score"
        ]
        if c in df.columns
    ]

    categorical_cols = [
        c
        for c in [
            "Sex",
            "MGMT Status"
        ]
        if c in df.columns
    ]

    transformers = []

    if numeric_cols:

        transformers.append(
            (
                "numeric",
                SimpleImputer(
                    strategy="median"
                ),
                numeric_cols
            )
        )

    if categorical_cols:

        cat_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                make_onehot()
            )
        ])

        transformers.append(
            (
                "categorical",
                cat_pipe,
                categorical_cols
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


def fit_gbsa_predict(
    X_train,
    y_train,
    X_target
):

    model = (
        GradientBoostingSurvivalAnalysis(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            subsample=0.8,
            random_state=42
        )
    )

    model.fit(
        X_train,
        y_train
    )

    return model.predict(
        X_target
    )


def fit_rsf_ensemble_predict(
    X_train,
    y_train,
    X_target
):

    predictions = []

    for seed in range(
        42,
        47
    ):

        model = RandomSurvivalForest(
            n_estimators=300,
            min_samples_leaf=5,
            min_samples_split=10,
            max_features="sqrt",
            random_state=seed,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        predictions.append(
            model.predict(
                X_target
            )
        )

    return np.mean(
        np.vstack(
            predictions
        ),
        axis=0
    )


# ============================================================
# 2. EXACT TRAIN-FOLD MOLECULAR RANKING
# ============================================================

def rank_molecular_features(
    X_train,
    y_train
):

    feature_names = list(
        X_train.columns
    )

    values = (
        X_train.to_numpy(
            dtype=np.float64
        )
    )

    scores = np.full(
        len(feature_names),
        0.5,
        dtype=float
    )

    for j in range(
        values.shape[1]
    ):

        x = values[:, j]

        if (
            np.nanstd(x)
            <= 1e-12
        ):
            continue

        try:

            c = (
                concordance_index_censored(
                    y_train["event"],
                    y_train["time"],
                    x
                )[0]
            )

            scores[j] = (
                0.5
                if np.isnan(c)
                else max(
                    c,
                    1.0 - c
                )
            )

        except Exception:
            pass

    ranking = pd.DataFrame({
        "feature":
            feature_names,

        "univariate_c":
            scores
    })

    return (
        ranking
        .sort_values(
            [
                "univariate_c",
                "feature"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )


def select_molecular(
    molecular,
    train_ids,
    y_train,
    top_k
):

    ranking = (
        rank_molecular_features(
            molecular.loc[
                train_ids
            ],
            y_train
        )
    )

    return (
        ranking[
            "feature"
        ]
        .head(
            top_k
        )
        .tolist()
    )


# ============================================================
# 3. LOAD TCGA
# ============================================================

print(
    "=" * 90
)

print(
    "FINAL EXTERNAL RUN — LOAD TCGA"
)

print(
    "=" * 90
)


split_df = pd.read_csv(
    SPLIT_FILE
)

survival = (
    pd.read_csv(
        SURVIVAL_FILE
    )
    .set_index(
        "patient_id"
    )
)

clinical = pd.read_csv(
    CLIN_FILE,
    index_col=0
)

expression = pd.read_csv(
    EXPR_FILE,
    index_col=0
)

rewiring = pd.read_csv(
    REWIRE_FILE,
    index_col=0
)

selected = pd.read_csv(
    FEATURE_FILE
)


split_df[
    "patient_id"
] = (
    split_df[
        "patient_id"
    ]
    .astype(str)
)


for df in [
    survival,
    clinical,
    expression,
    rewiring
]:

    df.index = (
        df.index
        .astype(str)
    )


development_ids = (
    split_df.loc[
        split_df["split"]
        == "development",
        "patient_id"
    ]
    .tolist()
)

test_ids = (
    split_df.loc[
        split_df["split"]
        == "locked_test",
        "patient_id"
    ]
    .tolist()
)


checkpoint(
    len(
        development_ids
    ) == 115,
    "115 development patients"
)

checkpoint(
    len(
        test_ids
    ) == 29,
    "29 internal-test patients"
)


y_dev = make_y(
    survival.loc[
        development_ids
    ]
)


# ============================================================
# 4. FROZEN FINAL FEATURE LISTS
# ============================================================

def frozen_features(
    model
):

    return (
        selected.loc[
            selected["model"]
            == model
        ]
        .sort_values(
            "rank"
        )[
            "feature"
        ]
        .astype(str)
        .tolist()
    )


m4_raw = frozen_features(
    "M4"
)

m5_raw = frozen_features(
    "M5"
)

m6_raw = frozen_features(
    "M6"
)


m4_expr = [
    x.replace(
        "EXPR::",
        ""
    )
    for x in m4_raw
]


m5_rewire = [
    x.replace(
        "REWIRE::",
        ""
    )
    for x in m5_raw
]


m6_expr = [
    x.replace(
        "EXPR::",
        ""
    )
    for x in m6_raw
    if x.startswith(
        "EXPR::"
    )
]


m6_rewire = [
    x.replace(
        "REWIRE::",
        ""
    )
    for x in m6_raw
    if x.startswith(
        "REWIRE::"
    )
]


checkpoint(
    len(
        m4_expr
    ) == 10,
    "M4 frozen expression = 10"
)

checkpoint(
    len(
        m5_rewire
    ) == 10,
    "M5 frozen rewiring = 10"
)

checkpoint(
    len(
        m6_expr
    ) == 10,
    "M6 frozen expression = 10"
)

checkpoint(
    len(
        m6_rewire
    ) == 10,
    "M6 frozen rewiring = 10"
)


# Drop ONLY unavailable frozen feature.
# No rank-11 replacement.

m5_rewire_adapt = [
    g
    for g in m5_rewire
    if g != MISSING_REWIRE
]

m6_rewire_adapt = [
    g
    for g in m6_rewire
    if g != MISSING_REWIRE
]


checkpoint(
    len(
        m5_rewire_adapt
    ) == 9,
    "M5 availability-adapted rewiring = 9/10"
)

checkpoint(
    len(
        m6_rewire_adapt
    ) == 9,
    "M6 availability-adapted rewiring = 9/10"
)


# ============================================================
# 5. AVAILABILITY-ADAPTED CLINICAL BLOCK
#
# KPS removed BEFORE fitting.
# ============================================================

ADAPTED_CLINICAL_COLS = [
    "Diagnosis Age",
    "Sex",
    "MGMT Status"
]


checkpoint(
    set(
        ADAPTED_CLINICAL_COLS
    ).issubset(
        clinical.columns
    ),
    "adapted TCGA clinical variables available"
)


# ============================================================
# 6. ADAPTED CLINICAL OOF RISKS
# ============================================================

def adapted_clinical_oof():

    n = len(
        development_ids
    )

    oof = np.full(
        n,
        np.nan
    )

    events = (
        y_dev[
            "event"
        ]
        .astype(int)
    )

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    for tr_idx, ho_idx in (
        cv.split(
            np.zeros(n),
            events
        )
    ):

        tr_ids = [
            development_ids[i]
            for i in tr_idx
        ]

        ho_ids = [
            development_ids[i]
            for i in ho_idx
        ]

        y_tr = make_y(
            survival.loc[
                tr_ids
            ]
        )

        processor = (
            build_clinical_preprocessor(
                clinical.loc[
                    tr_ids,
                    ADAPTED_CLINICAL_COLS
                ]
            )
        )

        X_tr = (
            processor
            .fit_transform(
                clinical.loc[
                    tr_ids,
                    ADAPTED_CLINICAL_COLS
                ]
            )
        )

        X_ho = (
            processor
            .transform(
                clinical.loc[
                    ho_ids,
                    ADAPTED_CLINICAL_COLS
                ]
            )
        )

        oof[
            ho_idx
        ] = (
            fit_gbsa_predict(
                X_tr,
                y_tr,
                X_ho
            )
        )

    checkpoint(
        np.isfinite(
            oof
        ).all(),
        "adapted clinical OOF risks finite"
    )

    return oof


clin_oof = (
    adapted_clinical_oof()
)


# ============================================================
# 7. EXACT MOLECULAR OOF RISKS
#
# Selection still occurs ONLY inside TCGA meta-training folds.
# CGGA does not participate.
# ============================================================

def molecular_oof(
    molecular,
    top_k=10
):

    n = len(
        development_ids
    )

    oof = np.full(
        n,
        np.nan
    )

    events = (
        y_dev[
            "event"
        ]
        .astype(int)
    )

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    for tr_idx, ho_idx in (
        cv.split(
            np.zeros(n),
            events
        )
    ):

        tr_ids = [
            development_ids[i]
            for i in tr_idx
        ]

        ho_ids = [
            development_ids[i]
            for i in ho_idx
        ]

        y_tr = make_y(
            survival.loc[
                tr_ids
            ]
        )

        fold_features = (
            select_molecular(
                molecular,
                tr_ids,
                y_tr,
                top_k
            )
        )

        X_tr = (
            molecular.loc[
                tr_ids,
                fold_features
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        X_ho = (
            molecular.loc[
                ho_ids,
                fold_features
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        oof[
            ho_idx
        ] = (
            fit_gbsa_predict(
                X_tr,
                y_tr,
                X_ho
            )
        )

    checkpoint(
        np.isfinite(
            oof
        ).all(),
        "molecular OOF risks finite"
    )

    return oof


expr_oof = molecular_oof(
    expression,
    top_k=10
)

rewire_oof = molecular_oof(
    rewiring,
    top_k=10
)


# ============================================================
# 8. FIT META MODELS
# ============================================================

def fit_meta(
    block_oof
):

    X_meta = np.column_stack(
        block_oof
    )

    model = CoxnetSurvivalAnalysis(
        l1_ratio=0.01,
        alpha_min_ratio=0.5
    )

    model.fit(
        X_meta,
        y_dev
    )

    return model


meta_M4 = fit_meta(
    [
        clin_oof,
        expr_oof
    ]
)

meta_M5 = fit_meta(
    [
        clin_oof,
        rewire_oof
    ]
)

meta_M6 = fit_meta(
    [
        clin_oof,
        expr_oof,
        rewire_oof
    ]
)


# ============================================================
# 9. LOAD / FILTER CGGA
# ============================================================

cgga_clin_raw = pd.read_csv(
    CGGA_CLIN_FILE,
    sep="\t",
    low_memory=False
)


EVENT_COL = (
    "Censor (alive=0; dead=1)"
)


cgga_clin_raw[
    "Age"
] = pd.to_numeric(
    cgga_clin_raw[
        "Age"
    ],
    errors="coerce"
)


cgga_clin_raw[
    "OS"
] = pd.to_numeric(
    cgga_clin_raw[
        "OS"
    ],
    errors="coerce"
)


cgga_clin_raw[
    EVENT_COL
] = pd.to_numeric(
    cgga_clin_raw[
        EVENT_COL
    ],
    errors="coerce"
)


eligible = (
    cgga_clin_raw.loc[
        (
            cgga_clin_raw[
                "PRS_type"
            ]
            == "Primary"
        )
        &
        (
            cgga_clin_raw[
                "Histology"
            ]
            == "GBM"
        )
        &
        (
            cgga_clin_raw[
                "Grade"
            ]
            == "WHO IV"
        )
        &
        (
            cgga_clin_raw[
                "IDH_mutation_status"
            ]
            == "Wildtype"
        )
        &
        (
            cgga_clin_raw[
                "Age"
            ]
            >= 18
        )
        &
        (
            cgga_clin_raw[
                "OS"
            ]
            > 0
        )
        &
        (
            cgga_clin_raw[
                EVENT_COL
            ]
            .isin(
                [0, 1]
            )
        )
    ]
    .copy()
)


eligible[
    "CGGA_ID"
] = (
    eligible[
        "CGGA_ID"
    ]
    .astype(str)
)


checkpoint(
    len(
        eligible
    ) == 104,
    "CGGA cohort = 104"
)

checkpoint(
    int(
        eligible[
            EVENT_COL
        ].sum()
    ) == 92,
    "CGGA events = 92"
)


# ============================================================
# 10. LOAD CGGA MOLECULAR MATRICES
# ============================================================

cgga_expr = pd.read_csv(
    CGGA_EXPR_FILE,
    index_col=0
)

cgga_rewire = pd.read_csv(
    CGGA_REWIRE_FILE,
    index_col=0
)


cgga_expr.index = (
    cgga_expr.index
    .astype(str)
)

cgga_expr.columns = (
    cgga_expr.columns
    .astype(str)
)

cgga_rewire.index = (
    cgga_rewire.index
    .astype(str)
)

cgga_rewire.columns = (
    cgga_rewire.columns
    .astype(str)
)


checkpoint(
    set(
        eligible[
            "CGGA_ID"
        ]
    )
    ==
    set(
        cgga_rewire.index
    ),
    "clinical IDs match CGGA rewiring"
)

checkpoint(
    set(
        eligible[
            "CGGA_ID"
        ]
    )
    ==
    set(
        cgga_expr.columns
    ),
    "clinical IDs match CGGA expression"
)


patient_order = (
    cgga_rewire
    .index
    .tolist()
)


eligible = (
    eligible
    .set_index(
        "CGGA_ID"
    )
    .loc[
        patient_order
    ]
)


# ============================================================
# 11. MAP CGGA CLINICAL CATEGORIES TO TCGA LABELS
# ============================================================

def normalize_label(x):

    if pd.isna(x):
        return None

    return "".join(
        ch
        for ch in str(x).lower()
        if ch.isalnum()
    )


def map_categories(
    external_series,
    tcga_series
):

    train_values = (
        tcga_series
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    lookup = {
        normalize_label(v): v
        for v in train_values
    }

    mapped = []

    unmapped = []

    for value in external_series:

        if pd.isna(value):
            mapped.append(
                np.nan
            )
            continue

        key = normalize_label(
            value
        )

        result = lookup.get(
            key,
            np.nan
        )

        mapped.append(
            result
        )

        if pd.isna(result):
            unmapped.append(
                str(value)
            )

    return (
        pd.Series(
            mapped,
            index=external_series.index
        ),
        sorted(
            set(
                unmapped
            )
        )
    )


cgga_clin = pd.DataFrame(
    index=patient_order
)


cgga_clin[
    "Diagnosis Age"
] = (
    eligible[
        "Age"
    ]
    .astype(float)
)


cgga_clin[
    "Sex"
], sex_unmapped = (
    map_categories(
        eligible[
            "Gender"
        ],
        clinical.loc[
            development_ids,
            "Sex"
        ]
    )
)


cgga_clin[
    "MGMT Status"
], mgmt_unmapped = (
    map_categories(
        eligible[
            "MGMTp_methylation_status"
        ],
        clinical.loc[
            development_ids,
            "MGMT Status"
        ]
    )
)


print(
    "\nTCGA Sex categories:"
)

print(
    clinical.loc[
        development_ids,
        "Sex"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nCGGA mapped Sex:"
)

print(
    cgga_clin[
        "Sex"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nUnmapped non-null Sex labels:",
    sex_unmapped
)


print(
    "\nTCGA MGMT categories:"
)

print(
    clinical.loc[
        development_ids,
        "MGMT Status"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nCGGA mapped MGMT:"
)

print(
    cgga_clin[
        "MGMT Status"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nUnmapped non-null MGMT labels:",
    mgmt_unmapped
)


checkpoint(
    cgga_clin[
        "Sex"
    ]
    .notna()
    .any(),
    "CGGA Sex mapped"
)

checkpoint(
    cgga_clin[
        "MGMT Status"
    ]
    .notna()
    .any(),
    "CGGA MGMT mapped"
)


# ============================================================
# 12. FINAL ADAPTED CLINICAL MATRICES
# ============================================================

processor = (
    build_clinical_preprocessor(
        clinical.loc[
            development_ids,
            ADAPTED_CLINICAL_COLS
        ]
    )
)


X_clin_dev = (
    processor
    .fit_transform(
        clinical.loc[
            development_ids,
            ADAPTED_CLINICAL_COLS
        ]
    )
)


X_clin_test = (
    processor
    .transform(
        clinical.loc[
            test_ids,
            ADAPTED_CLINICAL_COLS
        ]
    )
)


X_clin_cgga = (
    processor
    .transform(
        cgga_clin[
            ADAPTED_CLINICAL_COLS
        ]
    )
)


checkpoint(
    np.isfinite(
        X_clin_cgga
    ).all(),
    "CGGA clinical matrix finite"
)


# ============================================================
# 13. M1 — ADAPTED CLINICAL MODEL
# ============================================================

risk_M1_test = (
    fit_rsf_ensemble_predict(
        X_clin_dev,
        y_dev,
        X_clin_test
    )
)

risk_M1_cgga = (
    fit_rsf_ensemble_predict(
        X_clin_dev,
        y_dev,
        X_clin_cgga
    )
)


# ============================================================
# 14. FINAL GBSA BASE RISKS — INTERNAL TEST
# ============================================================

clin_GBSA_test = (
    fit_gbsa_predict(
        X_clin_dev,
        y_dev,
        X_clin_test
    )
)


# M4 expression

X_m4_dev = (
    expression.loc[
        development_ids,
        m4_expr
    ]
    .to_numpy(
        dtype=np.float64
    )
)

X_m4_test = (
    expression.loc[
        test_ids,
        m4_expr
    ]
    .to_numpy(
        dtype=np.float64
    )
)

m4_expr_test_risk = (
    fit_gbsa_predict(
        X_m4_dev,
        y_dev,
        X_m4_test
    )
)


# M5 adapted rewiring

X_m5_dev = (
    rewiring.loc[
        development_ids,
        m5_rewire_adapt
    ]
    .to_numpy(
        dtype=np.float64
    )
)

X_m5_test = (
    rewiring.loc[
        test_ids,
        m5_rewire_adapt
    ]
    .to_numpy(
        dtype=np.float64
    )
)

m5_rewire_test_risk = (
    fit_gbsa_predict(
        X_m5_dev,
        y_dev,
        X_m5_test
    )
)


# M6 expression

X_m6_expr_dev = (
    expression.loc[
        development_ids,
        m6_expr
    ]
    .to_numpy(
        dtype=np.float64
    )
)

X_m6_expr_test = (
    expression.loc[
        test_ids,
        m6_expr
    ]
    .to_numpy(
        dtype=np.float64
    )
)

m6_expr_test_risk = (
    fit_gbsa_predict(
        X_m6_expr_dev,
        y_dev,
        X_m6_expr_test
    )
)


# M6 adapted rewiring

X_m6_rew_dev = (
    rewiring.loc[
        development_ids,
        m6_rewire_adapt
    ]
    .to_numpy(
        dtype=np.float64
    )
)

X_m6_rew_test = (
    rewiring.loc[
        test_ids,
        m6_rewire_adapt
    ]
    .to_numpy(
        dtype=np.float64
    )
)

m6_rew_test_risk = (
    fit_gbsa_predict(
        X_m6_rew_dev,
        y_dev,
        X_m6_rew_test
    )
)


# ============================================================
# 15. ADAPTED INTERNAL STACKED PREDICTIONS
# ============================================================

risk_M4_test = (
    meta_M4.predict(
        np.column_stack([
            clin_GBSA_test,
            m4_expr_test_risk
        ])
    )
)


risk_M5_test = (
    meta_M5.predict(
        np.column_stack([
            clin_GBSA_test,
            m5_rewire_test_risk
        ])
    )
)


risk_M6_test = (
    meta_M6.predict(
        np.column_stack([
            clin_GBSA_test,
            m6_expr_test_risk,
            m6_rew_test_risk
        ])
    )
)


test_event = (
    survival.loc[
        test_ids,
        "OS_event"
    ]
    .to_numpy(
        dtype=bool
    )
)

test_time = (
    survival.loc[
        test_ids,
        "OS_months"
    ]
    .to_numpy(
        dtype=float
    )
)


internal_results = {
    "M1":
        risk_M1_test,

    "M4":
        risk_M4_test,

    "M5":
        risk_M5_test,

    "M6":
        risk_M6_test
}


saved_original = {
    "M1": 0.509554,
    "M4": 0.500000,
    "M5": 0.500000,
    "M6": 0.500000
}


print(
    "\n"
    + "=" * 90
)

print(
    "AVAILABILITY-ADAPTED INTERNAL IMPACT"
)

print(
    "=" * 90
)


internal_rows = []


for model, risk in (
    internal_results.items()
):

    adapted_c = cindex(
        test_event,
        test_time,
        risk
    )

    original_c = (
        saved_original[
            model
        ]
    )

    print(
        f"{model}: "
        f"original={original_c:.6f} | "
        f"adapted={adapted_c:.6f} | "
        f"delta={adapted_c-original_c:+.6f}"
    )

    internal_rows.append({
        "model":
            model,

        "original_internal_cindex":
            original_c,

        "adapted_internal_cindex":
            adapted_c,

        "delta":
            adapted_c
            - original_c
    })


# ============================================================
# 16. FINAL CGGA MOLECULAR BASE RISKS
# ============================================================

checkpoint(
    set(
        m4_expr
    ).issubset(
        cgga_expr.index
    ),
    "M4 expression 10/10 available"
)

checkpoint(
    set(
        m6_expr
    ).issubset(
        cgga_expr.index
    ),
    "M6 expression 10/10 available"
)

checkpoint(
    set(
        m5_rewire_adapt
    ).issubset(
        cgga_rewire.columns
    ),
    "M5 rewiring 9/9 available"
)

checkpoint(
    set(
        m6_rewire_adapt
    ).issubset(
        cgga_rewire.columns
    ),
    "M6 rewiring 9/9 available"
)


clin_GBSA_cgga = (
    fit_gbsa_predict(
        X_clin_dev,
        y_dev,
        X_clin_cgga
    )
)


X_m4_cgga = (
    cgga_expr.loc[
        m4_expr,
        patient_order
    ]
    .T
    .to_numpy(
        dtype=np.float64
    )
)


m4_expr_cgga_risk = (
    fit_gbsa_predict(
        X_m4_dev,
        y_dev,
        X_m4_cgga
    )
)


X_m5_cgga = (
    cgga_rewire.loc[
        patient_order,
        m5_rewire_adapt
    ]
    .to_numpy(
        dtype=np.float64
    )
)


m5_rewire_cgga_risk = (
    fit_gbsa_predict(
        X_m5_dev,
        y_dev,
        X_m5_cgga
    )
)


X_m6_expr_cgga = (
    cgga_expr.loc[
        m6_expr,
        patient_order
    ]
    .T
    .to_numpy(
        dtype=np.float64
    )
)


m6_expr_cgga_risk = (
    fit_gbsa_predict(
        X_m6_expr_dev,
        y_dev,
        X_m6_expr_cgga
    )
)


X_m6_rew_cgga = (
    cgga_rewire.loc[
        patient_order,
        m6_rewire_adapt
    ]
    .to_numpy(
        dtype=np.float64
    )
)


m6_rew_cgga_risk = (
    fit_gbsa_predict(
        X_m6_rew_dev,
        y_dev,
        X_m6_rew_cgga
    )
)


# ============================================================
# 17. FINAL STACKED CGGA RISKS
# ============================================================

risk_M4_cgga = (
    meta_M4.predict(
        np.column_stack([
            clin_GBSA_cgga,
            m4_expr_cgga_risk
        ])
    )
)


risk_M5_cgga = (
    meta_M5.predict(
        np.column_stack([
            clin_GBSA_cgga,
            m5_rewire_cgga_risk
        ])
    )
)


risk_M6_cgga = (
    meta_M6.predict(
        np.column_stack([
            clin_GBSA_cgga,
            m6_expr_cgga_risk,
            m6_rew_cgga_risk
        ])
    )
)


external_event = (
    eligible[
        EVENT_COL
    ]
    .to_numpy(
        dtype=bool
    )
)

external_time = (
    eligible[
        "OS"
    ]
    .to_numpy(
        dtype=float
    )
)


external_risks = {
    "M1":
        risk_M1_cgga,

    "M4":
        risk_M4_cgga,

    "M5":
        risk_M5_cgga,

    "M6":
        risk_M6_cgga
}


# ============================================================
# 18. BOOTSTRAP
# ============================================================

def bootstrap_ci(
    event,
    time,
    risk,
    n_boot=2000,
    seed=42
):

    rng = (
        np.random.default_rng(
            seed
        )
    )

    n = len(
        risk
    )

    values = []

    for _ in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            n,
            size=n
        )

        try:

            value = cindex(
                event[idx],
                time[idx],
                risk[idx]
            )

            if np.isfinite(
                value
            ):
                values.append(
                    value
                )

        except Exception:
            pass

    values = np.asarray(
        values,
        dtype=float
    )

    return (
        float(
            np.percentile(
                values,
                2.5
            )
        ),
        float(
            np.percentile(
                values,
                97.5
            )
        ),
        len(
            values
        )
    )


# ============================================================
# 19. FINAL EXTERNAL RESULTS
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "CGGA AVAILABILITY-ADAPTED EXTERNAL RESULTS"
)

print(
    "=" * 90
)


external_rows = []


for model in [
    "M1",
    "M4",
    "M5",
    "M6"
]:

    risk = (
        external_risks[
            model
        ]
    )

    checkpoint(
        np.isfinite(
            risk
        ).all(),
        f"{model} external risks finite"
    )

    c = cindex(
        external_event,
        external_time,
        risk
    )

    low, high, valid = (
        bootstrap_ci(
            external_event,
            external_time,
            risk,
            n_boot=2000,
            seed=42
        )
    )

    print(
        f"{model}: "
        f"C={c:.6f} | "
        f"95% CI "
        f"[{low:.6f}, {high:.6f}] | "
        f"valid bootstrap={valid}"
    )

    external_rows.append({
        "model":
            model,

        "status":
            "availability-adapted",

        "n":
            104,

        "events":
            92,

        "harrell_cindex":
            c,

        "ci_low":
            low,

        "ci_high":
            high,

        "bootstrap_valid":
            valid
    })


# ============================================================
# 20. SAVE EVERYTHING
# ============================================================

internal_df = pd.DataFrame(
    internal_rows
)

external_df = pd.DataFrame(
    external_rows
)


internal_df.to_csv(
    CGGA_DIR
    / "CGGA_FINAL_ADAPTED_INTERNAL_IMPACT_M1_M4_M5_M6.csv",
    index=False
)


external_df.to_csv(
    CGGA_DIR
    / "CGGA_FINAL_EXTERNAL_RESULTS_M1_M4_M5_M6.csv",
    index=False
)


prediction_df = pd.DataFrame({
    "CGGA_ID":
        patient_order,

    "OS_days":
        external_time,

    "event":
        external_event.astype(int),

    "M1_adapted_risk":
        risk_M1_cgga,

    "M4_adapted_risk":
        risk_M4_cgga,

    "M5_adapted_risk":
        risk_M5_cgga,

    "M6_adapted_risk":
        risk_M6_cgga
})


prediction_df.to_csv(
    CGGA_DIR
    / "CGGA_104_FINAL_M1_M4_M5_M6_PREDICTIONS.csv",
    index=False
)


# ============================================================
# 21. FREEZE
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "FINAL FREEZE NOTE"
)

print(
    "=" * 90
)

print(
    "M1: age + sex + MGMT; KPS unavailable."
)

print(
    "M4: adapted clinical + frozen 10/10 expression."
)

print(
    "M5: adapted clinical + frozen 9/10 rewiring."
)

print(
    "M6: adapted clinical + frozen 10/10 expression "
    "+ frozen 9/10 rewiring."
)

print(
    "Unavailable rewiring feature: "
    "ENSG00000145107 / TM4SF19"
)

print(
    "Replacement clinical variable: NONE"
)

print(
    "Replacement molecular feature: NONE"
)

print(
    "CGGA feature ranking: NO"
)

print(
    "CGGA hyperparameter tuning: NO"
)

print(
    "CGGA model selection: NO"
)

print(
    "CGGA outcomes used before scoring: NO"
)

print(
    "\nFINAL EXTERNAL MODELING COMPLETE."
)

### STEP 2C -- CGGA EXTERNAL EVALUATION OF THE DEVIATION BASELINES (R1, R2, R2B)

In [ ]:
"""
STEP 2C -- CGGA EXTERNAL EVALUATION OF THE DEVIATION BASELINES (R1, R2, R2B)
==========================================================================

Same rules as your frozen M2 / M3 CGGA evaluation (Untitled5, cells 4 and 6):
  * the final TCGA specification (learner, hyperparameters, Top-K) and the
    selected features are read from each Step 2B result folder -- FROZEN
  * the model is refit on the 115 TCGA development patients only
  * NO CGGA feature selection, NO tuning, NO replacement features
  * features that cannot be built in CGGA are dropped (availability-adapted),
    exactly as TM4SF19 was for M3
  * CGGA deviation scores use the SAME GTEx-334 reference mean and SD as TCGA
  * Harrell C-index + 2,000-replicate patient bootstrap (seed 42)

Extra (for the paper):
  * paired bootstrap difference in C-index versus your saved M2 and M3 CGGA
    predictions (same bootstrap samples for both models)
  * distribution shift check: TCGA vs CGGA for the selected features

How to run: paste into a new notebook cell (Stage D environment) and run once.
It evaluates all three representations in one go (~2-5 min).
Check everything printed from "SUMMARY" onwards.
"""

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

# =============================================================================
# 0. PATHS
# =============================================================================
DOWNLOADS = BASE
STAGE_C_REW = DOWNLOADS / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
STAGE_D = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL"
CGGA_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"
LADDER_DIR = STAGE_D / "REVISION_STEP2_DEVIATION_LADDER"

GTEX_FILE = STAGE_C_REW / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
EDGE_FILE = STAGE_C_REW / "STAGE_C_FINAL_FROZEN_EDGE_UNIVERSE.csv"
SPLIT_FILE = STAGE_D / "STAGE_D_LOCKED_SPLIT.csv"
SURVIVAL_FILE = STAGE_D / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
CGGA_EXPR_FILE = CGGA_DIR / "CGGA_104_HARMONIZED_TO_GTEX.csv"
CGGA_M2_PRED = CGGA_DIR / "CGGA_104_M2_EXACT_FROZEN_PREDICTIONS.csv"
CGGA_M3_PRED = CGGA_DIR / "CGGA_104_M3_AVAILABILITY_ADAPTED_PREDICTIONS.csv"

REPRESENTATIONS = {
    "R1": ("STAGE_D_R1_ABS_DEVIATION_RAW.csv", "REVISION_STEP2B_R1"),
    "R2": ("STAGE_D_R2_NEIGHBOUR_ABS_DEVIATION_RAW.csv", "REVISION_STEP2B_R2"),
    "R2B": ("STAGE_D_R2B_GENE_X_NEIGHBOUR_DEVIATION_RAW.csv", "REVISION_STEP2B_R2B"),
}

OUT_DIR = STAGE_D / "REVISION_STEP2C_CGGA_LADDER"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_FINAL_SEEDS = 5
N_BOOT = 2000
N_REF = 334
EVENT_COL = "Censor (alive=0; dead=1)"


def checkpoint(cond, msg):
    print(f"[CHECKPOINT {'PASS' if cond else 'FAIL'}] {msg}")
    if not cond:
        raise AssertionError(msg)


def section(t):
    print("\n" + "=" * 80 + f"\n{t}\n" + "=" * 80)


def strip_version(idx):
    return pd.Index(idx).astype(str).str.split(".").str[0].str.strip()


def make_y(event, time):
    return np.array([(bool(e), float(t)) for e, t in zip(event, time)],
                    dtype=[("event", "?"), ("time", "<f8")])


def cindex(y, risk):
    return float(concordance_index_censored(y["event"], y["time"],
                                            np.asarray(risk, float))[0])


# --- final-model fitting: identical to your Stage D fit_predict /
#     fit_predict_seed_ensemble (RSF averaged over seeds 42-46) -----------------
def fit_predict_final(params, X_train, y_train, X_target):
    if params["learner"] == "rsf":
        preds = []
        for k in range(N_FINAL_SEEDS):
            m = RandomSurvivalForest(
                n_estimators=params["n_estimators"],
                min_samples_leaf=params["min_samples_leaf"],
                max_features=params["max_features"],
                min_samples_split=10,
                random_state=RANDOM_SEED + k, n_jobs=-1)
            m.fit(X_train, y_train)
            preds.append(m.predict(X_target))
        return np.mean(preds, axis=0)
    if params["learner"] == "coxnet":
        m = CoxnetSurvivalAnalysis(l1_ratio=params["l1_ratio"],
                                   alpha_min_ratio=params["alpha_min_ratio"])
        m.fit(X_train, y_train)
        return m.predict(X_target)
    if params["learner"] == "gbsa":
        m = GradientBoostingSurvivalAnalysis(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"], subsample=params["subsample"],
            random_state=RANDOM_SEED)
        m.fit(X_train, y_train)
        return m.predict(X_target)
    raise ValueError(params["learner"])


# =============================================================================
# 1. TCGA
# =============================================================================
section("1. LOAD TCGA")
split = pd.read_csv(SPLIT_FILE)
split["patient_id"] = split["patient_id"].astype(str)
dev_ids = split.loc[split["split"] == "development", "patient_id"].tolist()
test_ids = split.loc[split["split"] == "locked_test", "patient_id"].tolist()
checkpoint(len(dev_ids) == 115 and len(test_ids) == 29, "115 development / 29 internal test")

surv = pd.read_csv(SURVIVAL_FILE)
surv["patient_id"] = surv["patient_id"].astype(str)
surv = surv.set_index("patient_id")
y_dev = make_y(surv.loc[dev_ids, "OS_event"], surv.loc[dev_ids, "OS_months"])
y_test = make_y(surv.loc[test_ids, "OS_event"], surv.loc[test_ids, "OS_months"])

# =============================================================================
# 2. GTEx reference statistics (identical to Step 2A: mean, population SD)
# =============================================================================
section("2. GTEx REFERENCE + CGGA DEVIATION SCORES")
gtex = pd.read_csv(GTEX_FILE, index_col=0, low_memory=False)
gtex.index = strip_version(gtex.index)
checkpoint(gtex.shape[1] == N_REF, "GTEx has 334 samples")

cgga = pd.read_csv(CGGA_EXPR_FILE, index_col=0)
cgga.index = strip_version(cgga.index)
cgga.columns = cgga.columns.astype(str)
checkpoint(cgga.shape[1] == 104, "CGGA harmonized matrix has 104 patients (genes x patients)")
checkpoint(cgga.index.isin(gtex.index).all(), "all CGGA genes present in GTEx")

G = gtex.loc[cgga.index].to_numpy(np.float64)
mu = G.mean(axis=1)
sd = np.sqrt(((G - mu[:, None]) ** 2).sum(axis=1) / N_REF)
checkpoint(np.all(sd > 0), "non-zero GTEx variance for all CGGA genes")
Zc = pd.DataFrame((cgga.to_numpy(np.float64) - mu[:, None]) / sd[:, None],
                  index=cgga.index, columns=cgga.columns)          # genes x 104
absZc = Zc.abs()

# Neighbours: the frozen TCGA candidate universe, restricted to genes
# measurable in CGGA (= the CGGA-compatible universe used for external eNESH)
edges = pd.read_csv(EDGE_FILE, low_memory=False)
for c in ("gene1", "gene2"):
    edges[c] = strip_version(edges[c])
checkpoint(len(edges) == 90959, "frozen TCGA candidate universe = 90,959 edges")
avail = set(cgga.index)
edges_c = edges[edges["gene1"].isin(avail) & edges["gene2"].isin(avail)]
print(f"CGGA-compatible edges: {len(edges_c):,} (your external network used 89,551)")
nbrs = {}
for a, b in zip(edges_c["gene1"], edges_c["gene2"]):
    nbrs.setdefault(a, []).append(b)
    nbrs.setdefault(b, []).append(a)


def cgga_feature(rep, gene):
    """CGGA value of one selected feature; None if it cannot be built."""
    if gene not in avail:
        return None
    if rep == "R1":
        return absZc.loc[gene].to_numpy()
    nb = nbrs.get(gene, [])
    if len(nb) == 0:
        return None
    nmean = absZc.loc[nb].mean(axis=0).to_numpy()
    if rep == "R2":
        return nmean
    return absZc.loc[gene].to_numpy() * nmean          # R2B


# =============================================================================
# 3. CGGA clinical cohort (same eligibility filter as your M2 cell)
# =============================================================================
section("3. CGGA CLINICAL COHORT")
cands = [p for p in DOWNLOADS.rglob("CGGA.mRNAseq_693_clinical.20200506.txt") if p.is_file()]
checkpoint(len(cands) >= 1, "CGGA clinical file found")
clin = pd.read_csv(cands[0], sep="\t", low_memory=False)
for c in ("Age", "OS", EVENT_COL):
    clin[c] = pd.to_numeric(clin[c], errors="coerce")
elig = clin.loc[(clin["PRS_type"] == "Primary") & (clin["Histology"] == "GBM")
                & (clin["Grade"] == "WHO IV") & (clin["IDH_mutation_status"] == "Wildtype")
                & (clin["Age"] >= 18) & (clin["OS"] > 0) & clin[EVENT_COL].isin([0, 1])].copy()
elig["CGGA_ID"] = elig["CGGA_ID"].astype(str)
checkpoint(len(elig) == 104, "CGGA external cohort = 104")
checkpoint(int(elig[EVENT_COL].sum()) == 92, "CGGA events = 92")
checkpoint(set(elig["CGGA_ID"]) == set(cgga.columns), "clinical IDs match expression IDs")
order = elig["CGGA_ID"].tolist()
elig = elig.set_index("CGGA_ID").loc[order]
ext_event = elig[EVENT_COL].to_numpy(bool)
ext_time = elig["OS"].to_numpy(float)
y_ext = make_y(ext_event, ext_time)
col_pos = pd.Series(np.arange(cgga.shape[1]), index=cgga.columns)[order].to_numpy()

# saved M2 / M3 external predictions for paired comparison
ref_risks = {}
for name, f, col in [("M2", CGGA_M2_PRED, "M2_frozen_risk"),
                     ("M3", CGGA_M3_PRED, "M3_availability_adapted_risk")]:
    if f.exists():
        d = pd.read_csv(f)
        d["CGGA_ID"] = d["CGGA_ID"].astype(str)
        d = d.set_index("CGGA_ID")
        if set(order).issubset(d.index) and col in d.columns:
            ref_risks[name] = d.loc[order, col].to_numpy(float)
            print(f"{name} saved CGGA predictions loaded: C = {cindex(y_ext, ref_risks[name]):.4f}")
    else:
        print(f"(note) {f.name} not found -> no paired comparison with {name}")

# common bootstrap indices for every model -> paired differences
rng = np.random.default_rng(RANDOM_SEED)
boot_idx = [rng.integers(0, len(order), size=len(order)) for _ in range(N_BOOT)]


def boot_c(risk):
    out = np.full(N_BOOT, np.nan)
    for b, idx in enumerate(boot_idx):
        try:
            out[b] = concordance_index_censored(ext_event[idx], ext_time[idx], risk[idx])[0]
        except Exception:
            pass
    return out


ref_boot = {k: boot_c(v) for k, v in ref_risks.items()}

# =============================================================================
# 4. EVALUATE EACH REPRESENTATION
# =============================================================================
rows = []
for rep, (mat_file, res_folder) in REPRESENTATIONS.items():
    section(f"4. {rep}")
    res_dir = STAGE_D / res_folder
    for f in (LADDER_DIR / mat_file, res_dir / "STAGE_D_V4_OOF_FINAL_SPECS.json",
              res_dir / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv",
              res_dir / "STAGE_D_V4_OOF_INTERNAL_TEST_RESULTS.csv"):
        checkpoint(f.exists(), f"exists: {f.parent.name}/{f.name}")

    X_tcga = pd.read_csv(LADDER_DIR / mat_file, index_col=0)
    X_tcga.index = X_tcga.index.astype(str)
    X_tcga.columns = strip_version(X_tcga.columns)
    spec = json.load(open(res_dir / "STAGE_D_V4_OOF_FINAL_SPECS.json"))["M3"]
    params = spec["params"]
    sel = pd.read_csv(res_dir / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv")
    feats = (sel.loc[sel["model"] == "M3"].sort_values("rank")["feature"].astype(str)
             .str.replace("REWIRE::", "", regex=False)
             .str.replace("EXPR::", "", regex=False).tolist())
    feats = list(strip_version(feats))
    checkpoint(len(feats) == spec["top_k"], f"{len(feats)} frozen features = Top-K {spec['top_k']}")
    print(f"Frozen spec: {params}")
    saved_internal = float(pd.read_csv(res_dir / "STAGE_D_V4_OOF_INTERNAL_TEST_RESULTS.csv")
                           .query("model == 'M3'")["internal_test_cindex"].iloc[0])

    # --- reproduce the TCGA internal result -----------------------------------
    Xd = X_tcga.loc[dev_ids, feats].to_numpy(float)
    Xt = X_tcga.loc[test_ids, feats].to_numpy(float)
    c_int = cindex(y_test, fit_predict_final(params, Xd, y_dev, Xt))
    print(f"Reconstructed internal C = {c_int:.6f} | saved = {saved_internal:.6f}")
    checkpoint(abs(c_int - saved_internal) < 1e-4, "frozen TCGA internal result reproduced")

    # --- build CGGA features; drop unavailable ones ---------------------------
    ext_cols, keep = [], []
    for g in feats:
        v = cgga_feature(rep, g)
        if v is not None:
            ext_cols.append(v[col_pos])
            keep.append(g)
    dropped = [g for g in feats if g not in keep]
    print(f"Available in CGGA: {len(keep)}/{len(feats)}; dropped: {dropped if dropped else 'none'}")
    X_ext = np.column_stack(ext_cols)
    checkpoint(np.isfinite(X_ext).all(), "all CGGA feature values finite")

    if dropped:
        Xd_a = X_tcga.loc[dev_ids, keep].to_numpy(float)
        Xt_a = X_tcga.loc[test_ids, keep].to_numpy(float)
        c_int_adapted = cindex(y_test, fit_predict_final(params, Xd_a, y_dev, Xt_a))
        print(f"Availability-adapted internal C = {c_int_adapted:.6f}")
    else:
        Xd_a, c_int_adapted = Xd, c_int

    # --- distribution shift (selected features, TCGA dev vs CGGA) -------------
    tc = pd.DataFrame(Xd_a, columns=keep)
    cg = pd.DataFrame(X_ext, columns=keep)
    pooled = np.sqrt((tc.var() + cg.var()) / 2).replace(0, np.nan)
    smd = ((cg.mean() - tc.mean()) / pooled).abs()
    print(f"|standardized mean difference| TCGA vs CGGA: median {smd.median():.2f}, "
          f"max {smd.max():.2f}")

    # --- external prediction ---------------------------------------------------
    risk = fit_predict_final(params, Xd_a, y_dev, X_ext)
    c_ext = cindex(y_ext, risk)
    bc = boot_c(risk)
    ok = np.isfinite(bc)
    lo, hi = np.percentile(bc[ok], [2.5, 97.5])
    print(f"CGGA C = {c_ext:.4f} (95% CI {lo:.4f}-{hi:.4f}; {ok.sum()} valid replicates)")

    row = {"representation": rep, "learner": params["learner"], "top_k": spec["top_k"],
           "features_used_in_CGGA": len(keep), "internal_C": c_int,
           "internal_C_adapted": c_int_adapted, "CGGA_C": c_ext,
           "CGGA_CI_low": lo, "CGGA_CI_high": hi, "median_abs_SMD": smd.median()}
    for ref_name, rb in ref_boot.items():
        diff = bc - rb
        okd = np.isfinite(diff)
        dlo, dhi = np.percentile(diff[okd], [2.5, 97.5])
        delta = c_ext - cindex(y_ext, ref_risks[ref_name])
        row[f"delta_vs_{ref_name}"] = delta
        row[f"delta_vs_{ref_name}_CI_low"] = dlo
        row[f"delta_vs_{ref_name}_CI_high"] = dhi
        print(f"  C({rep}) - C({ref_name}) = {delta:+.4f} (paired 95% CI {dlo:+.4f} to {dhi:+.4f})")
    rows.append(row)
    pd.DataFrame({"CGGA_ID": order, "OS_days": ext_time, "event": ext_event.astype(int),
                  f"{rep}_risk": risk}).to_csv(
        OUT_DIR / f"CGGA_104_{rep}_FROZEN_PREDICTIONS.csv", index=False)

# =============================================================================
# 5. SUMMARY
# =============================================================================
section("SUMMARY")
summary = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
summary.to_csv(OUT_DIR / "STEP2C_CGGA_LADDER_SUMMARY.csv", index=False)
print(f"\nSaved to: {OUT_DIR}")
print("Check everything printed from SUMMARY onwards (and any CHECKPOINT FAIL).")